In [ ]:
# ============================================================
# SETUP — 10.01 Sparse-Label Detection-Loss Source Audit
# Purpose:
#   Recover the exact current detection-loss implementation
#   used by the frozen training adapter and identify:
#
#       - positive target definition
#       - negative/unlabeled target definition
#       - neg_weight application
#       - reduction convention
#       - detection-loss call site
#
#   This is preparation for a sparse-label / PU-style training
#   experiment. No loss behavior is changed here.
#
# Changes files: NO
# Runs training: NO
# Uses GT: NO
# Keep after running: YES — Stage-10 source audit
# ============================================================

from pathlib import Path
import ast


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)


# ============================================================
# Read current frozen training adapter
# ============================================================

source = TRAIN_SCRIPT.read_text(
    encoding="utf-8"
)

tree = ast.parse(
    source
)


# ============================================================
# Search all functions / methods containing detection-loss
# related terms.
# ============================================================

KEYWORDS = [
    "det_loss",
    "detection_loss",
    "neg_weight",
    "binary_cross_entropy",
    "bce",
    "target",
    "heatmap",
    "detection",
]


matches = []

for node in ast.walk(tree):

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):

        node_source = ast.get_source_segment(
            source,
            node,
        )

        if node_source is None:
            continue

        lowered = node_source.lower()

        if any(
            keyword in lowered
            for keyword in KEYWORDS
        ):

            matches.append(
                (
                    node.name,
                    node.lineno,
                    node_source,
                )
            )


print(
    "=== 10.01 DETECTION-LOSS SOURCE AUDIT ==="
)

print(
    "Training script:",
    TRAIN_SCRIPT
)

print(
    "Matching functions:",
    len(matches)
)


for name, lineno, node_source in matches:

    print()
    print("=" * 72)
    print(
        f"FUNCTION: {name} | line {lineno}"
    )
    print("=" * 72)

    print(
        node_source
    )


# ============================================================
# Direct line-context search for critical terms
# ============================================================

lines = source.splitlines()

SEARCH_TERMS = [
    "neg_weight",
    "det_loss",
    "detection_loss",
    "binary_cross_entropy",
    "BCE",
]


print()
print("=" * 72)
print("DIRECT TERM CONTEXT")
print("=" * 72)


for term in SEARCH_TERMS:

    found = []

    for i, line in enumerate(
        lines,
        start=1,
    ):

        if term.lower() in line.lower():

            start = max(
                1,
                i - 8,
            )

            end = min(
                len(lines),
                i + 12,
            )

            found.append(
                (
                    i,
                    start,
                    end,
                )
            )

    print()
    print(
        f"TERM: {term}"
    )

    if not found:

        print(
            "  no matches"
        )

        continue

    for hit, start, end in found:

        print()
        print(
            f"--- hit line {hit} "
            f"(context {start}:{end}) ---"
        )

        for lineno in range(
            start,
            end + 1,
        ):

            marker = (
                ">>"
                if lineno == hit
                else "  "
            )

            print(
                f"{marker} "
                f"{lineno:04d}: "
                f"{lines[lineno - 1]}"
            )


print()
print("=" * 72)
print("10.01 RESULT")
print("=" * 72)

print(
    "Detection-loss implementation audit: COMPLETE"
)

In [ ]:
# ============================================================
# SETUP — 10.02 Sparse-Label Detection-Loss Prototype
# Purpose:
#   Prototype a minimal sparse-label-aware detection loss:
#
#       - GT voxels remain positive
#       - ordinary non-GT voxels remain negative
#       - extremely high-confidence non-GT voxels are treated
#         as UNLABELED and excluded from the negative term
#
#   Critically, remaining negatives are renormalized so that
#   total negative-loss mass remains approximately neg_weight.
#
#   This Cell tests loss/gradient semantics ONLY.
#
# Changes files: NO
# Runs training: NO
# Uses GT: NO — synthetic tensors only
# Keep after running: YES — Stage-10 loss-design evidence
# ============================================================

import torch
import torch.nn.functional as F
import pandas as pd


# ============================================================
# Exact current loss, isolated for semantic comparison
# ============================================================

def baseline_det_loss_1002(
    logits,
    target,
    neg_weight=1e-2,
):
    """
    Exact weighting semantics of current detection loss,
    after target construction.
    """

    B = logits.shape[0]

    flat_target = target.reshape(
        B,
        -1,
    )

    n_pos = (
        flat_target
        .sum(dim=1)
        .clamp(min=1)
    )

    n_total = (
        flat_target.shape[1]
    )

    n_neg = (
        n_total
        - n_pos
    ).clamp(min=1)

    shape = (
        (B,)
        + (1,) * (
            logits.ndim - 1
        )
    )

    w_pos = (
        1.0 / n_pos
    ).reshape(shape)

    w_neg = (
        neg_weight
        / n_neg
    ).reshape(shape)

    weight = torch.where(
        target == 1.0,
        w_pos,
        w_neg,
    )

    return (
        F.binary_cross_entropy_with_logits(
            logits,
            target,
            weight=weight,
            reduction="sum",
        )
        / B
    )


# ============================================================
# Candidate sparse-label-aware loss
# ============================================================

def confidence_excluded_det_loss_1002(
    logits,
    target,
    neg_weight=1e-2,
    ignore_prob=0.9975,
):
    """
    Positive:
        target == 1

    Nominal negative:
        target == 0

    High-confidence nominal negative:
        sigmoid(logit) >= ignore_prob

    High-confidence nominal negatives are NOT promoted to
    positives. They are simply treated as unlabeled and omitted
    from the negative BCE term.

    The remaining negatives are renormalized so their total
    nominal loss weight remains neg_weight.
    """

    B = logits.shape[0]

    # Important:
    # supervision mask must not receive gradient.
    with torch.no_grad():

        probs = torch.sigmoid(
            logits
        )

        positive = (
            target == 1.0
        )

        nominal_negative = (
            ~positive
        )

        ignored_unlabeled = (
            nominal_negative
            &
            (
                probs >= ignore_prob
            )
        )

        supervised_negative = (
            nominal_negative
            &
            ~ignored_unlabeled
        )


    # --------------------------------------------------------
    # Per-sample counts
    # --------------------------------------------------------

    positive_flat = (
        positive.reshape(
            B,
            -1,
        )
    )

    negative_flat = (
        supervised_negative.reshape(
            B,
            -1,
        )
    )

    n_pos = (
        positive_flat
        .sum(dim=1)
        .clamp(min=1)
    )

    n_neg = (
        negative_flat
        .sum(dim=1)
        .clamp(min=1)
    )


    shape = (
        (B,)
        + (1,) * (
            logits.ndim - 1
        )
    )

    w_pos = (
        1.0 / n_pos
    ).reshape(shape)

    w_neg = (
        neg_weight
        / n_neg
    ).reshape(shape)


    # --------------------------------------------------------
    # Weight map
    # --------------------------------------------------------

    weight = torch.zeros_like(
        logits
    )

    weight = torch.where(
        positive,
        w_pos.expand_as(
            logits
        ),
        weight,
    )

    weight = torch.where(
        supervised_negative,
        w_neg.expand_as(
            logits
        ),
        weight,
    )


    loss = (
        F.binary_cross_entropy_with_logits(
            logits,
            target,
            weight=weight,
            reduction="sum",
        )
        / B
    )


    stats = {
        "positive_voxels":
            int(
                positive.sum().item()
            ),

        "supervised_negative_voxels":
            int(
                supervised_negative
                .sum()
                .item()
            ),

        "ignored_unlabeled_voxels":
            int(
                ignored_unlabeled
                .sum()
                .item()
            ),
    }

    return loss, stats


# ============================================================
# Synthetic semantic test
#
# One positive + several nominal negatives:
#
#   index 0: GT positive
#   index 1: extremely confident nominal negative
#            -> should be ignored by candidate loss
#   index 2: moderately confident nominal negative
#            -> must STILL receive negative gradient
#   index 3: easy background
#   index 4: easy background
# ============================================================

probs = torch.tensor(
    [
        [
            0.90,
            0.999,
            0.80,
            0.10,
            0.01,
        ]
    ],
    dtype=torch.float32,
)

logits_template = torch.logit(
    probs
)

target = torch.tensor(
    [
        [
            1.0,
            0.0,
            0.0,
            0.0,
            0.0,
        ]
    ],
    dtype=torch.float32,
)


# ============================================================
# Baseline gradients
# ============================================================

baseline_logits = (
    logits_template
    .clone()
    .detach()
    .requires_grad_(True)
)

baseline_loss = (
    baseline_det_loss_1002(
        baseline_logits,
        target,
        neg_weight=1e-2,
    )
)

baseline_loss.backward()

baseline_grad = (
    baseline_logits.grad
    .detach()
    .clone()
)


# ============================================================
# Sparse-label-aware gradients
# ============================================================

pu_logits = (
    logits_template
    .clone()
    .detach()
    .requires_grad_(True)
)

pu_loss, pu_stats = (
    confidence_excluded_det_loss_1002(
        pu_logits,
        target,
        neg_weight=1e-2,
        ignore_prob=0.9975,
    )
)

pu_loss.backward()

pu_grad = (
    pu_logits.grad
    .detach()
    .clone()
)


# ============================================================
# Comparison table
# ============================================================

labels = [
    "GT_POSITIVE",
    "HIGH_CONF_UNLABELED",
    "MODERATE_NEGATIVE",
    "BACKGROUND_0.10",
    "BACKGROUND_0.01",
]


result_1002 = pd.DataFrame({
    "role":
        labels,

    "probability":
        probs[0].numpy(),

    "target":
        target[0].numpy(),

    "baseline_gradient":
        baseline_grad[
            0
        ].numpy(),

    "candidate_gradient":
        pu_grad[
            0
        ].numpy(),
})


# ============================================================
# Formal invariants
# ============================================================

# Positive supervision must remain active.
assert (
    pu_grad[0, 0] < 0
)

# Extremely confident nominal negative must be ignored.
assert torch.isclose(
    pu_grad[0, 1],
    torch.tensor(
        0.0
    ),
    atol=1e-10,
)

# Moderate nominal negative must still be pushed downward.
assert (
    pu_grad[0, 2] > 0
)

# Background negatives must still be supervised.
assert (
    pu_grad[0, 3] > 0
)

assert (
    pu_grad[0, 4] > 0
)

# Baseline must penalize the high-confidence non-GT voxel.
assert (
    baseline_grad[0, 1] > 0
)


# ============================================================
# No-ignore reproduction gate
#
# ignore_prob > 1 means no nominal negative can be ignored.
# Candidate must numerically reproduce the baseline.
# ============================================================

repro_logits = (
    logits_template
    .clone()
    .detach()
)

repro_loss, repro_stats = (
    confidence_excluded_det_loss_1002(
        repro_logits,
        target,
        neg_weight=1e-2,
        ignore_prob=1.1,
    )
)

assert torch.allclose(
    repro_loss,
    baseline_det_loss_1002(
        repro_logits,
        target,
        neg_weight=1e-2,
    ),
    atol=1e-8,
    rtol=1e-6,
)


# ============================================================
# Result
# ============================================================

print(
    "=== 10.02 SPARSE-LABEL LOSS PROTOTYPE ==="
)

display(
    result_1002
)

print()
print(
    "Baseline loss:",
    float(
        baseline_loss
    ),
)

print(
    "Candidate loss:",
    float(
        pu_loss
    ),
)

print()
print(
    "Candidate stats:",
    pu_stats,
)

print()
print(
    "No-ignore reproduction:",
    True,
)

print()
print(
    "Expected semantic result:"
)

print(
    "  GT positive gradient < 0"
)

print(
    "  high-confidence unlabeled gradient = 0"
)

print(
    "  ordinary/background negative gradients > 0"
)

print()
print(
    "10.02 sparse-label loss prototype: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.04 Baseline vs Sparse-Label Integration Smoke Test
# Purpose:
#   Verify that BOTH detection-loss modes run end-to-end through
#   the real training pipeline from the same Warm2 checkpoint:
#
#       A. BASELINE
#          det_ignore_prob = None
#
#       B. SPARSE_9975
#          det_ignore_prob = 0.9975
#
#   This is ONLY an integration smoke test:
#       - one Fold0 TRAIN video
#       - first 12 frames only
#       - 12 optimizer iterations
#       - 1 epoch
#
#   Success criteria:
#       - both subprocesses exit with code 0
#       - both use the Biohub project .venv Python
#       - both load the same Warm2 checkpoint
#       - both produce finite epoch losses
#       - baseline log reports ignore_prob=None
#       - sparse log reports ignore_prob=0.9975
#       - both fresh checkpoints are produced
#
# IMPORTANT:
#   Dataset resolution is restricted explicitly to:
#
#       competition/.../train/
#
#   It NEVER searches test/, even when an identically named
#   image volume exists there.
#
# Changes files: YES — temporary smoke checkpoints/logs only
# Runs training: YES — 12 iterations per variant
# Uses GT: YES — normal training supervision on one TRAIN video
# Keep after running: YES — keep RESULT; smoke artifacts can be
#                     deleted after Stage-10 integration verifies
# ============================================================

from pathlib import Path
import json
import os
import re
import subprocess
import time
import hashlib
import sys

import numpy as np
import pandas as pd


# ============================================================
# Project paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

DATA_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development"
)

# ------------------------------------------------------------
# IMPORTANT:
# Training data must come ONLY from train/.
# ------------------------------------------------------------

TRAIN_ROOT = (
    DATA_ROOT
    / "train"
)

TEST_ROOT = (
    DATA_ROOT
    / "test"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

LOG_DIR = (
    PROJECT
    / "reports/stage10_smoke"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Frozen hashes
# ============================================================

EXPECTED_WARM2_SHA = (
    "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"
)

EXPECTED_PATCHED_TRAIN_SHA = (
    "ce4fe8b3a16771039e1c687c02e71cad2747ee48d94417f0a26c929192f1be7b"
)


# ============================================================
# Helpers
# ============================================================

def sha256_file_1004(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# Preconditions
# ============================================================

assert PROJECT.exists(), (
    f"Project missing: {PROJECT}"
)

assert TRAIN_SCRIPT.exists(), (
    f"Training script missing: {TRAIN_SCRIPT}"
)

assert DATA_ROOT.exists(), (
    f"Competition root missing: {DATA_ROOT}"
)

assert TRAIN_ROOT.exists(), (
    f"Training directory missing: {TRAIN_ROOT}"
)

assert TEST_ROOT.exists(), (
    f"Test directory missing: {TEST_ROOT}"
)

assert SPLITS.exists(), (
    f"Fold manifest missing: {SPLITS}"
)

assert WARM2.exists(), (
    f"Warm2 checkpoint missing: {WARM2}"
)


# ============================================================
# Verify frozen Warm2 checkpoint
# ============================================================

warm2_sha_1004 = (
    sha256_file_1004(
        WARM2
    )
)

assert (
    warm2_sha_1004
    == EXPECTED_WARM2_SHA
), (
    "Warm2 checkpoint SHA mismatch.\n"
    f"Expected: {EXPECTED_WARM2_SHA}\n"
    f"Actual:   {warm2_sha_1004}"
)


# ============================================================
# Verify patched Stage-10 training script
# ============================================================

train_script_sha_1004 = (
    sha256_file_1004(
        TRAIN_SCRIPT
    )
)

assert (
    train_script_sha_1004
    == EXPECTED_PATCHED_TRAIN_SHA
), (
    "Training script no longer matches the verified 10.03 patch.\n"
    f"Expected: {EXPECTED_PATCHED_TRAIN_SHA}\n"
    f"Actual:   {train_script_sha_1004}"
)


script_text_1004 = (
    TRAIN_SCRIPT.read_text(
        encoding="utf-8"
    )
)

assert (
    "--det-ignore-prob"
    in script_text_1004
)

assert (
    "det_ignore_prob"
    in script_text_1004
)

assert (
    "--init-weights"
    in script_text_1004
)


# ============================================================
# Read Fold0 manifest
# ============================================================

splits_obj = json.loads(
    SPLITS.read_text(
        encoding="utf-8"
    )
)


# Support:
#
#   [ {...fold0...}, ... ]
#
# or
#
#   {"folds": [...]}
#
# or
#
#   {"splits": [...]}
# ============================================================

if isinstance(
    splits_obj,
    dict,
):

    if "folds" in splits_obj:

        folds = (
            splits_obj[
                "folds"
            ]
        )

    elif "splits" in splits_obj:

        folds = (
            splits_obj[
                "splits"
            ]
        )

    else:

        raise ValueError(
            "dataset_splits.json is a dict, but no "
            "'folds' or 'splits' field was found."
        )

elif isinstance(
    splits_obj,
    list,
):

    folds = (
        splits_obj
    )

else:

    raise TypeError(
        "Unsupported dataset_splits.json structure: "
        f"{type(splits_obj)}"
    )


assert len(
    folds
) > 0


fold0 = (
    folds[0]
)

assert (
    "train"
    in fold0
)

assert len(
    fold0[
        "train"
    ]
) > 0


# ============================================================
# Resolve one Fold0 TRAIN video
# ============================================================

train_entry = (
    fold0[
        "train"
    ][0]
)


def resolve_dataset_1004(
    entry,
) -> Path:
    """
    Resolve a Fold0 dataset ONLY inside competition/train/.

    Fold entries may be:
        44b6_0113de3b

    or:
        44b6_0113de3b.zarr

    Never search competition/test/.
    """

    entry = str(
        entry
    )


    if entry.endswith(
        ".zarr"
    ):

        filename = (
            entry
        )

    else:

        filename = (
            f"{entry}.zarr"
        )


    zarr_path = (
        TRAIN_ROOT
        / filename
    )


    if not zarr_path.exists():

        raise FileNotFoundError(
            "Could not resolve Fold0 TRAIN dataset:\n"
            f"{zarr_path}"
        )


    # --------------------------------------------------------
    # Paired annotation must also exist in train/.
    # --------------------------------------------------------

    if zarr_path.name.endswith(
        ".zarr"
    ):

        stem = (
            zarr_path.name[
                :-5
            ]
        )

    else:

        stem = (
            zarr_path.stem
        )


    geff_path = (
        TRAIN_ROOT
        / f"{stem}.geff"
    )


    if not geff_path.exists():

        raise FileNotFoundError(
            "Paired training GEFF missing:\n"
            f"{geff_path}"
        )


    # --------------------------------------------------------
    # Safety:
    # explicitly verify this is under train/.
    # --------------------------------------------------------

    resolved_zarr = (
        zarr_path.resolve()
    )

    resolved_train = (
        TRAIN_ROOT.resolve()
    )


    assert (
        resolved_train
        in resolved_zarr.parents
    ), (
        "Resolved smoke dataset is not under TRAIN_ROOT."
    )


    return (
        zarr_path
    )


DEBUG_VIDEO = (
    resolve_dataset_1004(
        train_entry
    )
)


DEBUG_STEM = (
    DEBUG_VIDEO.name[
        :-5
    ]
)


DEBUG_GEFF = (
    TRAIN_ROOT
    / f"{DEBUG_STEM}.geff"
)


print(
    "Smoke video:",
    DEBUG_VIDEO.name,
)

print(
    "Smoke Zarr:",
    DEBUG_VIDEO,
)

print(
    "Smoke GEFF:",
    DEBUG_GEFF,
)

print(
    "Confirmed TRAIN-only:",
    (
        TRAIN_ROOT.resolve()
        in DEBUG_VIDEO.resolve().parents
    ),
)


# ============================================================
# Extra safety:
# acknowledge but NEVER use identically named test volume
# ============================================================

same_name_test = (
    TEST_ROOT
    / DEBUG_VIDEO.name
)


print(
    "Same-name test volume exists:",
    same_name_test.exists(),
)

print(
    "Same-name test volume used:",
    False,
)


# ============================================================
# Resolve Biohub project Python interpreter
#
# IMPORTANT:
#   DO NOT use CONDA_PREFIX.
#
# Current VS Code Notebook kernel has already been verified as:
#
#   /Users/heqiuyan/Desktop/biohub-cell-tracking/.venv/bin/python
#
# CONDA_PREFIX may still incorrectly point to:
#
#   /opt/anaconda3/envs/bankiller-py311
# ============================================================

PROJECT_PYTHON = (
    PROJECT
    / ".venv/bin/python"
)


assert PROJECT_PYTHON.exists(), (
    f"Biohub project Python missing: {PROJECT_PYTHON}"
)


NOTEBOOK_PYTHON = (
    Path(
        sys.executable
    )
)


assert (
    PROJECT_PYTHON.resolve()
    == NOTEBOOK_PYTHON.resolve()
), (
    "Notebook kernel and Biohub .venv Python do not match.\n"
    f"Notebook: {NOTEBOOK_PYTHON}\n"
    f"Project:  {PROJECT_PYTHON}"
)


PYTHON = (
    PROJECT_PYTHON
)


print()
print(
    "Python:",
    PYTHON,
)

print(
    "Notebook sys.executable:",
    NOTEBOOK_PYTHON,
)

print(
    "Matches notebook kernel:",
    True,
)

print(
    "CONDA_PREFIX ignored:",
    os.environ.get(
        "CONDA_PREFIX"
    ),
)


# ============================================================
# Official baseline scripts import path
# ============================================================

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)


assert OFFICIAL_SCRIPTS.exists(), (
    f"Official scripts missing: {OFFICIAL_SCRIPTS}"
)


env = (
    os.environ.copy()
)


existing_pythonpath = (
    env.get(
        "PYTHONPATH",
        ""
    )
)


env["PYTHONPATH"] = (
    str(
        OFFICIAL_SCRIPTS
    )
    + (
        os.pathsep
        + existing_pythonpath
        if existing_pythonpath
        else ""
    )
)


# ============================================================
# Strong subprocess interpreter sanity check
# ============================================================

interpreter_check = subprocess.run(
    [
        str(
            PYTHON
        ),
        "-c",
        (
            "import sys, torch, polars, zarr, numpy; "
            "print(sys.executable); "
            "print(torch.__version__); "
            "print(polars.__version__)"
        ),
    ],
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


assert (
    interpreter_check.returncode
    == 0
), (
    "Biohub subprocess interpreter import check failed:\n"
    + interpreter_check.stdout
)


print()
print(
    "Biohub subprocess imports:",
    "PASS",
)


# ============================================================
# Training-script CLI sanity check
# ============================================================

help_check = subprocess.run(
    [
        str(
            PYTHON
        ),
        str(
            TRAIN_SCRIPT
        ),
        "--help",
    ],
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


assert (
    help_check.returncode
    == 0
), (
    "Training script --help failed:\n"
    + help_check.stdout
)


assert (
    "--det-ignore-prob"
    in help_check.stdout
)

assert (
    "--init-weights"
    in help_check.stdout
)


print(
    "Training script CLI:",
    "PASS",
)


# ============================================================
# Smoke configuration
#
# Same configuration for BOTH variants except ignore mode.
# ============================================================

COMMON_ARGS = [

    str(
        PYTHON
    ),

    str(
        TRAIN_SCRIPT
    ),

    "--debug-video",
    str(
        DEBUG_VIDEO
    ),

    "--epochs",
    "1",

    "--max-iters",
    "12",

    "--max-frames",
    "12",

    "--batch-size",
    "2",

    "--num-workers",
    "0",

    "--lr",
    "1e-5",

    "--seed",
    "1640",

    "--downsample",
    "1,4,4",

    "--det-loss-weight",
    "1.0",

    "--det-neg-weight",
    "0.01",

    "--init-weights",
    str(
        WARM2
    ),

    "--pool-kernel-um",
    "5.0",

    "--single-gpu",
]


# ============================================================
# Variants
# ============================================================

VARIANTS_1004 = {

    "BASELINE": {

        "method":
            "stage10_smoke_baseline",

        "extra_args":
            [],
    },


    "SPARSE_9975": {

        "method":
            "stage10_smoke_sparse9975",

        "extra_args":
            [
                "--det-ignore-prob",
                "0.9975",
            ],
    },
}


# ============================================================
# Log parser
# ============================================================

# Expected epoch line:
#
# Epoch 0/1 | edge=... | det=... |
# test_loss=... | acc=... | recall=...
#
epoch_pattern = re.compile(
    r"Epoch\s+\d+/\d+\s+\|\s+"
    r"edge=([0-9.eE+-]+)\s+\|\s+"
    r"det=([0-9.eE+-]+)\s+\|\s+"
    r"test_loss=([0-9.eE+-]+)\s+\|\s+"
    r"acc=([0-9.eE+-]+)\s+\|\s+"
    r"recall=([0-9.eE+-]+)"
)


# Expected save line:
#
# Best score (...) ..., saved to /path/...pth
#
save_pattern = re.compile(
    r"saved to\s+(.+?edge_predictor_best\.pth)"
)


# ============================================================
# Execute both smoke variants
# ============================================================

rows_1004 = []


print()
print(
    "=" * 72
)

print(
    "10.04 INTEGRATION SMOKE START"
)

print(
    "=" * 72
)


for variant, cfg in (
    VARIANTS_1004.items()
):

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )


    # --------------------------------------------------------
    # Remove previous log.
    # --------------------------------------------------------

    if log_path.exists():

        log_path.unlink()


    cmd = (
        COMMON_ARGS
        + [
            "--method",
            cfg[
                "method"
            ],
        ]
        + cfg[
            "extra_args"
        ]
    )


    print()
    print(
        f"Starting {variant}..."
    )


    start_wall = (
        time.time()
    )


    # --------------------------------------------------------
    # Long stdout/stderr goes to log.
    # --------------------------------------------------------

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_f:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            text=True,
        )


    end_wall = (
        time.time()
    )


    elapsed = (
        end_wall
        - start_wall
    )


    log_text = (
        log_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )


    # ========================================================
    # Configuration wiring checks
    # ========================================================

    if variant == "BASELINE":

        config_ok = (
            "ignore_prob=None"
            in log_text
        )

    elif variant == "SPARSE_9975":

        config_ok = (
            "ignore_prob=0.9975"
            in log_text
        )

    else:

        raise ValueError(
            variant
        )


    # ========================================================
    # Warm2 initialization check
    # ========================================================

    warm_start_ok = (
        "Full-model init weights loaded:"
        in log_text
    )


    # ========================================================
    # Parse epoch result
    # ========================================================

    epoch_matches = list(
        epoch_pattern.finditer(
            log_text
        )
    )


    epoch_found = (
        len(
            epoch_matches
        )
        > 0
    )


    if epoch_found:

        m = (
            epoch_matches[
                -1
            ]
        )


        edge_loss = float(
            m.group(1)
        )

        det_loss = float(
            m.group(2)
        )

        test_loss = float(
            m.group(3)
        )

        test_acc = float(
            m.group(4)
        )

        test_recall = float(
            m.group(5)
        )

    else:

        edge_loss = np.nan
        det_loss = np.nan
        test_loss = np.nan
        test_acc = np.nan
        test_recall = np.nan


    # ========================================================
    # Resolve produced checkpoint
    # ========================================================

    save_matches = list(
        save_pattern.finditer(
            log_text
        )
    )


    checkpoint_path = (
        None
    )


    if save_matches:

        raw_path = (
            save_matches[
                -1
            ]
            .group(1)
            .strip()
        )


        checkpoint_path = (
            Path(
                raw_path
            )
        )


        if not checkpoint_path.is_absolute():

            checkpoint_path = (
                PROJECT
                / checkpoint_path
            )


        checkpoint_path = (
            checkpoint_path.resolve()
        )


    checkpoint_exists = bool(
        checkpoint_path
        is not None
        and checkpoint_path.exists()
    )


    # --------------------------------------------------------
    # Verify checkpoint belongs to THIS smoke invocation.
    # --------------------------------------------------------

    checkpoint_fresh = (
        False
    )

    checkpoint_sha = (
        None
    )


    if checkpoint_exists:

        checkpoint_mtime = (
            checkpoint_path
            .stat()
            .st_mtime
        )


        checkpoint_fresh = (
            checkpoint_mtime
            >= start_wall - 2.0
        )


        checkpoint_sha = (
            sha256_file_1004(
                checkpoint_path
            )
        )


    # ========================================================
    # Store result
    # ========================================================

    rows_1004.append({

        "variant":
            variant,

        "return_code":
            proc.returncode,

        "config_ok":
            config_ok,

        "warm_start_ok":
            warm_start_ok,

        "epoch_summary_found":
            epoch_found,

        "checkpoint_exists":
            checkpoint_exists,

        "checkpoint_fresh":
            checkpoint_fresh,

        "edge_loss":
            edge_loss,

        "det_loss":
            det_loss,

        "test_loss":
            test_loss,

        "test_acc":
            test_acc,

        "test_recall":
            test_recall,

        "elapsed_sec":
            elapsed,

        "log":
            str(
                log_path
            ),

        "checkpoint":
            (
                str(
                    checkpoint_path
                )
                if checkpoint_path
                is not None
                else None
            ),

        "checkpoint_sha256":
            checkpoint_sha,
    })


    print(
        f"{variant}: "
        f"return={proc.returncode}, "
        f"elapsed={elapsed:.1f}s, "
        f"config_ok={config_ok}, "
        f"warm_start={warm_start_ok}, "
        f"checkpoint={checkpoint_exists}"
    )


# ============================================================
# Results dataframe
# ============================================================

smoke_results_1004 = (
    pd.DataFrame(
        rows_1004
    )
)


# ============================================================
# If subprocess failed:
# print ONLY useful log tail before assertion.
# ============================================================

failed_variants = (
    smoke_results_1004[
        smoke_results_1004[
            "return_code"
        ] != 0
    ]
)


if len(
    failed_variants
):

    print()
    print(
        "=" * 72
    )

    print(
        "FAILED RUN LOG TAIL"
    )

    print(
        "=" * 72
    )


    for row in (
        failed_variants
        .itertuples(
            index=False
        )
    ):

        log_path = (
            Path(
                row.log
            )
        )


        text = (
            log_path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        )


        tail = "\n".join(
            text.splitlines()[
                -50:
            ]
        )


        print()
        print(
            f"--- {row.variant} ---"
        )

        print(
            tail
        )


# ============================================================
# Formal integration verification
# ============================================================

assert (
    smoke_results_1004[
        "return_code"
    ] == 0
).all(), (
    "At least one smoke run failed. "
    "See FAILED RUN LOG TAIL above."
)


assert (
    smoke_results_1004[
        "config_ok"
    ]
).all(), (
    "Detection-loss mode was not wired correctly."
)


assert (
    smoke_results_1004[
        "warm_start_ok"
    ]
).all(), (
    "Warm2 full-model initialization was not confirmed."
)


assert (
    smoke_results_1004[
        "epoch_summary_found"
    ]
).all(), (
    "Could not parse final epoch summary."
)


assert (
    smoke_results_1004[
        "checkpoint_exists"
    ]
).all(), (
    "At least one smoke checkpoint was not produced."
)


assert (
    smoke_results_1004[
        "checkpoint_fresh"
    ]
).all(), (
    "At least one checkpoint appears stale."
)


# ============================================================
# Finite-number checks
# ============================================================

for col in [
    "edge_loss",
    "det_loss",
    "test_loss",
    "test_acc",
    "test_recall",
]:

    vals = (
        smoke_results_1004[
            col
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert np.isfinite(
        vals
    ).all(), (
        f"Non-finite values found in {col}: {vals}"
    )


# ============================================================
# Sanity ranges
# ============================================================

assert (
    smoke_results_1004[
        "edge_loss"
    ] >= 0
).all()


assert (
    smoke_results_1004[
        "det_loss"
    ] >= 0
).all()


assert (
    smoke_results_1004[
        "test_loss"
    ] >= 0
).all()


assert (
    smoke_results_1004[
        "test_acc"
    ].between(
        0.0,
        1.0,
    )
).all()


assert (
    smoke_results_1004[
        "test_recall"
    ].between(
        0.0,
        1.0,
    )
).all()


# ============================================================
# Checkpoint comparison
#
# They SHOULD generally differ because training gradients differ.
# This is diagnostic only, not a hard pass/fail requirement.
# ============================================================

baseline_sha = (
    smoke_results_1004.loc[
        smoke_results_1004[
            "variant"
        ] == "BASELINE",
        "checkpoint_sha256",
    ]
    .iloc[0]
)


sparse_sha = (
    smoke_results_1004.loc[
        smoke_results_1004[
            "variant"
        ] == "SPARSE_9975",
        "checkpoint_sha256",
    ]
    .iloc[0]
)


checkpoint_hashes_differ = (
    baseline_sha
    != sparse_sha
)


# ============================================================
# Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.04 RESULT"
)

print(
    "=" * 72
)


display(
    smoke_results_1004[
        [
            "variant",
            "return_code",
            "config_ok",
            "warm_start_ok",
            "checkpoint_exists",
            "checkpoint_fresh",

            "edge_loss",
            "det_loss",

            "test_loss",
            "test_acc",
            "test_recall",

            "elapsed_sec",
        ]
    ]
)


print()
print(
    "TRAIN DATA SAFETY"
)

print(
    "  Zarr:",
    DEBUG_VIDEO
)

print(
    "  GEFF:",
    DEBUG_GEFF
)

print(
    "  Under train/:",
    (
        TRAIN_ROOT.resolve()
        in DEBUG_VIDEO.resolve().parents
    )
)

print(
    "  Same-name test exists:",
    same_name_test.exists()
)

print(
    "  Test volume used:",
    False
)


print()
print(
    "PYTHON ENVIRONMENT"
)

print(
    "  Notebook Python:",
    NOTEBOOK_PYTHON
)

print(
    "  Subprocess Python:",
    PYTHON
)

print(
    "  Same interpreter:",
    (
        NOTEBOOK_PYTHON.resolve()
        == PYTHON.resolve()
    )
)

print(
    "  Biohub subprocess imports:",
    "PASS"
)


print()
print(
    "MODEL INITIALIZATION"
)

print(
    "  Warm2 SHA256:",
    warm2_sha_1004
)

print(
    "  Training-script SHA256:",
    train_script_sha_1004
)


print()
print(
    "SMOKE CHECKPOINTS"
)


for row in (
    smoke_results_1004
    .itertuples(
        index=False
    )
):

    print(
        f"  {row.variant}:"
    )

    print(
        f"    {row.checkpoint}"
    )

    print(
        f"    SHA256={row.checkpoint_sha256}"
    )


print()
print(
    "Checkpoint hashes differ:",
    checkpoint_hashes_differ
)


print()
print(
    "Baseline integration:",
    "PASS"
)

print(
    "Sparse integration:",
    "PASS"
)


print()
print(
    "IMPORTANT:"
)

print(
    "  This smoke test is NOT a model-quality comparison."
)

print(
    "  No formal Fold0 model decision was made."
)


print()
print(
    "10.04 baseline-vs-sparse integration smoke: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.05 Warm2 Sparse-Ignore Threshold Calibration
# Purpose:
#   Measure what the proposed probability-based sparse-label
#   ignore thresholds actually do on REAL Warm2 outputs.
#
#   Audit thresholds:
#       sigmoid(logit) >= 0.9900
#       sigmoid(logit) >= 0.9950
#       sigmoid(logit) >= 0.9975
#       sigmoid(logit) >= 0.9990
#
#   For each threshold measure:
#       - ignored nominal-negative voxels
#       - ignored voxels per annotated positive
#       - ignored 5-um local maxima
#       - ignored local maxima within 2 / 4 um of GT
#       - ignored local maxima > 4 um from GT
#
#   Sampling:
#       - all 40 Fold0 validation datasets
#       - 2 deterministic training-style windows per dataset
#
#   IMPORTANT:
#       GT is sparse.
#       A high-confidence non-GT voxel is NOT called a false
#       positive here.
#
# Changes files: NO
# Runs training: NO
# Runs inference: YES — Warm2 forward passes only
# Uses GT: YES — target construction / distance diagnostics only
# Keep after running: YES — Stage-10 threshold calibration evidence
# ============================================================

from pathlib import Path
import sys
import json
import math
import importlib.util

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

DATA_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development"
)

TRAIN_ROOT = (
    DATA_ROOT
    / "train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

WARM2_CONFIG = (
    WARM2.parent
    / "config.json"
)


# ============================================================
# Preconditions
# ============================================================

assert TRAIN_SCRIPT.exists()
assert OFFICIAL_SCRIPTS.exists()
assert TRAIN_ROOT.exists()
assert SPLITS.exists()
assert WARM2.exists()
assert WARM2_CONFIG.exists()

assert (
    Path(sys.executable).resolve()
    ==
    (
        PROJECT
        / ".venv/bin/python"
    ).resolve()
), (
    "Notebook is not using the Biohub project .venv."
)


# ============================================================
# Import project training module
# ============================================================

if str(
    OFFICIAL_SCRIPTS
) not in sys.path:

    sys.path.insert(
        0,
        str(
            OFFICIAL_SCRIPTS
        ),
    )


MODULE_NAME_1005 = (
    "biohub_train_stage10_1005"
)


spec = (
    importlib.util.spec_from_file_location(
        MODULE_NAME_1005,
        TRAIN_SCRIPT,
    )
)

assert spec is not None
assert spec.loader is not None


train_mod_1005 = (
    importlib.util.module_from_spec(
        spec
    )
)

sys.modules[
    MODULE_NAME_1005
] = train_mod_1005

spec.loader.exec_module(
    train_mod_1005
)


# ============================================================
# Warm2 architecture
# ============================================================

cfg = json.loads(
    WARM2_CONFIG.read_text(
        encoding="utf-8"
    )
)


UNET_OUT_CHANNELS = int(
    cfg[
        "unet_out_channels"
    ]
)

UNET_LAYERS = [
    int(x)
    for x in cfg[
        "unet_layers"
    ]
]

DOWNSAMPLE = tuple(
    int(x)
    for x in cfg[
        "downsample"
    ]
)

WINDOW_SIZE = int(
    cfg.get(
        "window_size",
        2,
    )
)

POOL_KERNEL_UM = float(
    cfg.get(
        "pool_kernel_um",
        5.0,
    )
)


assert (
    DOWNSAMPLE
    == (1, 4, 4)
)

assert (
    WINDOW_SIZE
    == 2
)


# ============================================================
# Device
# ============================================================

if (
    hasattr(
        torch.backends,
        "mps",
    )
    and
    torch.backends.mps.is_available()
):

    DEVICE = torch.device(
        "mps"
    )

else:

    DEVICE = torch.device(
        "cpu"
    )


print(
    "Device:",
    DEVICE,
)


# ============================================================
# Reconstruct frozen Warm2 model
# ============================================================

unet = (
    train_mod_1005.TemporalUNet3D(
        in_channels=1,
        out_channels=UNET_OUT_CHANNELS,
        layers=UNET_LAYERS,
    )
)


model_1005 = (
    train_mod_1005.UNetNodeTransformer(
        unet=unet,
        unet_out_channels=UNET_OUT_CHANNELS,
        pos_feat_dim=(
            4
            * train_mod_1005._POS_EMBED_DIM
        ),
    )
)


state = torch.load(
    WARM2,
    map_location="cpu",
    weights_only=True,
)


model_1005.load_state_dict(
    state,
    strict=True,
)


model_1005 = (
    model_1005
    .to(
        DEVICE
    )
    .eval()
)


print(
    "Warm2 strict load:",
    "PASS",
)


# ============================================================
# Fold0 validation manifest
# ============================================================

splits_obj = json.loads(
    SPLITS.read_text(
        encoding="utf-8"
    )
)


if isinstance(
    splits_obj,
    dict,
):

    if "folds" in splits_obj:

        folds = (
            splits_obj[
                "folds"
            ]
        )

    elif "splits" in splits_obj:

        folds = (
            splits_obj[
                "splits"
            ]
        )

    else:

        raise ValueError(
            "No folds/splits field in manifest."
        )

else:

    folds = splits_obj


fold0 = folds[0]


# Training adapter calls the validation side "test".
VAL_NAMES = [
    str(x)
    for x in fold0[
        "test"
    ]
]


assert len(
    VAL_NAMES
) == 40


# ============================================================
# Threshold grid
# ============================================================

IGNORE_PROBS_1005 = [
    0.9900,
    0.9950,
    0.9975,
    0.9990,
]


# ============================================================
# Semantics table
# ============================================================

threshold_semantics_1005 = (
    pd.DataFrame({
        "ignore_prob":
            IGNORE_PROBS_1005,

        "equivalent_raw_logit":
            [
                math.log(
                    p
                    / (
                        1.0 - p
                    )
                )
                for p in IGNORE_PROBS_1005
            ],
    })
)


print()
print(
    "PROBABILITY -> RAW-LOGIT THRESHOLDS"
)

display(
    threshold_semantics_1005
)


# ============================================================
# Helpers
# ============================================================

def dataset_path_1005(
    name: str,
) -> Path:

    name = str(
        name
    )

    if name.endswith(
        ".zarr"
    ):

        p = (
            TRAIN_ROOT
            / name
        )

    else:

        p = (
            TRAIN_ROOT
            / f"{name}.zarr"
        )

    assert p.exists(), p

    return p


def prefix_1005(
    name: str,
) -> str:

    return str(
        name
    ).split(
        "_"
    )[0]


def pool_kernel_1005(
    voxel_size,
    pool_um,
):

    kernel = []

    for s in voxel_size:

        k = max(
            1,
            round(
                pool_um
                / float(s)
            ),
        )

        if (
            k % 2
            == 0
        ):

            k += 1

        kernel.append(
            int(k)
        )

    return tuple(
        kernel
    )


# ============================================================
# Audit storage
# ============================================================

frame_rows_1005 = []

positive_prob_rows_1005 = []


# ============================================================
# Run
# ============================================================

WINDOWS_PER_DATASET = 2


print()
print(
    "=== 10.05 WARM2 SPARSE-IGNORE CALIBRATION START ==="
)


with torch.no_grad():

    for dataset_i, name in enumerate(
        VAL_NAMES,
        start=1,
    ):

        path = (
            dataset_path_1005(
                name
            )
        )


        # ----------------------------------------------------
        # Exact training-style GT windows.
        # ----------------------------------------------------

        video_meta, windows = (
            train_mod_1005.load_dataset_windows(
                path,
                window_size=WINDOW_SIZE,
                max_frames=None,
                downsample=DOWNSAMPLE,
            )
        )


        if len(
            windows
        ) == 0:

            continue


        # ----------------------------------------------------
        # Deterministic coverage:
        # first and last available window.
        #
        # If only one window exists, audit it once.
        # ----------------------------------------------------

        if len(
            windows
        ) == 1:

            selected_indices = [
                0
            ]

        else:

            selected_indices = [
                0,
                len(windows) - 1,
            ]


        selected_windows = [
            windows[
                idx
            ]
            for idx in selected_indices
        ]


        max_nodes = max(
            max(
                w.node_counts
            )
            for w in selected_windows
        )


        ds = (
            train_mod_1005.FrameWindowDataset(
                [
                    (
                        video_meta,
                        selected_windows,
                    )
                ],
                max_nodes=max_nodes,
                augmentations=None,
            )
        )


        for sample_i in range(
            len(ds)
        ):

            item = ds[
                sample_i
            ]


            imgs = (
                item[
                    "imgs"
                ]
                .unsqueeze(0)
                .to(
                    DEVICE,
                    dtype=torch.float32,
                )
            )


            # ------------------------------------------------
            # Real Warm2 forward
            # ------------------------------------------------

            _, det_logits_list = (
                model_1005.encode(
                    imgs
                )
            )


            voxel_size = tuple(
                float(x)
                for x in item[
                    "voxel_size"
                ].tolist()
            )


            kernel = (
                pool_kernel_1005(
                    voxel_size,
                    POOL_KERNEL_UM,
                )
            )


            pad = tuple(
                k // 2
                for k in kernel
            )


            for frame_i in range(
                WINDOW_SIZE
            ):

                # --------------------------------------------
                # Raw detector output
                # --------------------------------------------

                logits = (
                    det_logits_list[
                        frame_i
                    ][
                        0,
                        0,
                    ]
                    .detach()
                    .float()
                    .cpu()
                )


                probs = torch.sigmoid(
                    logits
                )


                spatial = (
                    logits.shape
                )


                # --------------------------------------------
                # Build positive target EXACTLY like current
                # detection loss.
                # --------------------------------------------

                coords = (
                    item[
                        "coords"
                    ][
                        frame_i
                    ]
                )

                mask = (
                    item[
                        "masks"
                    ][
                        frame_i
                    ]
                )


                gt_coords = (
                    coords[
                        mask
                    ]
                    .float()
                )


                assert len(
                    gt_coords
                ) > 0


                zi = (
                    gt_coords[
                        :,
                        0,
                    ]
                    .long()
                    .clamp(
                        0,
                        spatial[
                            0
                        ] - 1,
                    )
                )

                yi = (
                    gt_coords[
                        :,
                        1,
                    ]
                    .long()
                    .clamp(
                        0,
                        spatial[
                            1
                        ] - 1,
                    )
                )

                xi = (
                    gt_coords[
                        :,
                        2,
                    ]
                    .long()
                    .clamp(
                        0,
                        spatial[
                            2
                        ] - 1,
                    )
                )


                gt_idx = torch.unique(
                    torch.stack(
                        [
                            zi,
                            yi,
                            xi,
                        ],
                        dim=1,
                    ),
                    dim=0,
                )


                positive = torch.zeros(
                    spatial,
                    dtype=torch.bool,
                )


                positive[
                    gt_idx[
                        :,
                        0
                    ],
                    gt_idx[
                        :,
                        1
                    ],
                    gt_idx[
                        :,
                        2
                    ],
                ] = True


                nominal_negative = (
                    ~positive
                )


                n_positive = int(
                    positive.sum().item()
                )


                # --------------------------------------------
                # Positive confidence diagnostics
                # --------------------------------------------

                pos_probs = (
                    probs[
                        positive
                    ]
                )


                for p in (
                    pos_probs.tolist()
                ):

                    positive_prob_rows_1005.append({
                        "dataset":
                            name,

                        "prefix":
                            prefix_1005(
                                name
                            ),

                        "prob":
                            float(
                                p
                            ),
                    })


                # --------------------------------------------
                # 5-um local maxima
                #
                # Using raw logits exactly as training adapter
                # peak detector does.
                # --------------------------------------------

                pooled = (
                    F.max_pool3d(
                        logits[
                            None,
                            None,
                        ],
                        kernel_size=kernel,
                        stride=1,
                        padding=pad,
                    )[
                        0,
                        0,
                    ]
                )


                local_peak = (
                    logits
                    == pooled
                )


                # --------------------------------------------
                # Threshold audit
                # --------------------------------------------

                for threshold in (
                    IGNORE_PROBS_1005
                ):

                    ignored = (
                        nominal_negative
                        &
                        (
                            probs
                            >= threshold
                        )
                    )


                    ignored_voxels = int(
                        ignored
                        .sum()
                        .item()
                    )


                    ignored_peaks_mask = (
                        ignored
                        &
                        local_peak
                    )


                    peak_coords = (
                        torch.nonzero(
                            ignored_peaks_mask,
                            as_tuple=False,
                        )
                        .float()
                    )


                    ignored_peaks = int(
                        len(
                            peak_coords
                        )
                    )


                    # ----------------------------------------
                    # Distances from ignored local maxima
                    # to nearest annotated GT voxel.
                    #
                    # Sparse GT => distance > 4 um is NOT a
                    # negative label.
                    # ----------------------------------------

                    peak_within_2 = 0
                    peak_within_4 = 0
                    peak_farther_4 = 0
                    median_peak_gt_dist = np.nan


                    if ignored_peaks > 0:

                        scale_t = torch.tensor(
                            voxel_size,
                            dtype=torch.float32,
                        )


                        peak_um = (
                            peak_coords
                            * scale_t
                        )


                        gt_um = (
                            gt_idx
                            .float()
                            * scale_t
                        )


                        d = torch.cdist(
                            peak_um,
                            gt_um,
                        )


                        nearest = (
                            d.min(
                                dim=1
                            )
                            .values
                        )


                        peak_within_2 = int(
                            (
                                nearest
                                <= 2.0
                            )
                            .sum()
                            .item()
                        )


                        peak_within_4 = int(
                            (
                                nearest
                                <= 4.0
                            )
                            .sum()
                            .item()
                        )


                        peak_farther_4 = int(
                            (
                                nearest
                                > 4.0
                            )
                            .sum()
                            .item()
                        )


                        median_peak_gt_dist = float(
                            nearest
                            .median()
                            .item()
                        )


                    frame_rows_1005.append({
                        "dataset":
                            name,

                        "prefix":
                            prefix_1005(
                                name
                            ),

                        "window_index":
                            int(
                                selected_indices[
                                    sample_i
                                ]
                            ),

                        "frame_index":
                            int(
                                frame_i
                            ),

                        "threshold":
                            float(
                                threshold
                            ),

                        "n_positive":
                            n_positive,

                        "ignored_voxels":
                            ignored_voxels,

                        "ignored_voxels_per_positive":
                            (
                                ignored_voxels
                                / max(
                                    n_positive,
                                    1,
                                )
                            ),

                        "ignored_local_peaks":
                            ignored_peaks,

                        "ignored_peaks_within_2um":
                            peak_within_2,

                        "ignored_peaks_within_4um":
                            peak_within_4,

                        "ignored_peaks_farther_4um":
                            peak_farther_4,

                        "median_ignored_peak_gt_dist_um":
                            median_peak_gt_dist,
                    })


        if (
            dataset_i == 1
            or dataset_i % 10 == 0
            or dataset_i == len(
                VAL_NAMES
            )
        ):

            print(
                f"Processed "
                f"{dataset_i:02d}/"
                f"{len(VAL_NAMES)}"
            )


# ============================================================
# DataFrames
# ============================================================

calibration_frames_1005 = (
    pd.DataFrame(
        frame_rows_1005
    )
)


positive_probs_1005 = (
    pd.DataFrame(
        positive_prob_rows_1005
    )
)


assert len(
    calibration_frames_1005
) > 0

assert len(
    positive_probs_1005
) > 0


# ============================================================
# Aggregate threshold summary
# ============================================================

summary_rows_1005 = []


for threshold in (
    IGNORE_PROBS_1005
):

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        sub = (
            calibration_frames_1005[
                calibration_frames_1005[
                    "threshold"
                ]
                == threshold
            ]
        )


        if prefix != "ALL":

            sub = sub[
                sub[
                    "prefix"
                ]
                == prefix
            ]


        total_pos = int(
            sub[
                "n_positive"
            ].sum()
        )


        total_ignored = int(
            sub[
                "ignored_voxels"
            ].sum()
        )


        total_peaks = int(
            sub[
                "ignored_local_peaks"
            ].sum()
        )


        near2 = int(
            sub[
                "ignored_peaks_within_2um"
            ].sum()
        )


        near4 = int(
            sub[
                "ignored_peaks_within_4um"
            ].sum()
        )


        far4 = int(
            sub[
                "ignored_peaks_farther_4um"
            ].sum()
        )


        summary_rows_1005.append({
            "threshold":
                threshold,

            "prefix":
                prefix,

            "frames":
                len(sub),

            "annotated_positive_voxels":
                total_pos,

            "ignored_voxels":
                total_ignored,

            "ignored_voxels_per_positive":
                (
                    total_ignored
                    / max(
                        total_pos,
                        1,
                    )
                ),

            "ignored_local_peaks":
                total_peaks,

            "ignored_local_peaks_per_frame":
                (
                    total_peaks
                    / max(
                        len(sub),
                        1,
                    )
                ),

            "peak_within_2um":
                near2,

            "peak_within_4um":
                near4,

            "peak_farther_4um":
                far4,

            "farther_4um_fraction":
                (
                    far4
                    / total_peaks
                    if total_peaks
                    else np.nan
                ),
        })


ignore_summary_1005 = (
    pd.DataFrame(
        summary_rows_1005
    )
)


# ============================================================
# Positive probability distribution
# ============================================================

positive_summary_rows_1005 = []


for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    q = (
        positive_probs_1005
        if prefix == "ALL"
        else positive_probs_1005[
            positive_probs_1005[
                "prefix"
            ]
            == prefix
        ]
    )


    s = (
        q[
            "prob"
        ]
    )


    positive_summary_rows_1005.append({
        "prefix":
            prefix,

        "n_positive":
            len(s),

        "p10":
            float(
                s.quantile(
                    0.10
                )
            ),

        "p25":
            float(
                s.quantile(
                    0.25
                )
            ),

        "median":
            float(
                s.median()
            ),

        "p75":
            float(
                s.quantile(
                    0.75
                )
            ),

        "p90":
            float(
                s.quantile(
                    0.90
                )
            ),

        "p95":
            float(
                s.quantile(
                    0.95
                )
            ),

        "p99":
            float(
                s.quantile(
                    0.99
                )
            ),
    })


positive_prob_summary_1005 = (
    pd.DataFrame(
        positive_summary_rows_1005
    )
)


# ============================================================
# Dataset-level activation robustness
# ============================================================

dataset_activation_1005 = (
    calibration_frames_1005
    .groupby(
        [
            "threshold",
            "prefix",
            "dataset",
        ],
        as_index=False,
    )
    .agg(
        ignored_local_peaks=(
            "ignored_local_peaks",
            "sum",
        ),

        ignored_peaks_farther_4um=(
            "ignored_peaks_farther_4um",
            "sum",
        ),
    )
)


activation_rows_1005 = []


for threshold in (
    IGNORE_PROBS_1005
):

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        q = (
            dataset_activation_1005[
                dataset_activation_1005[
                    "threshold"
                ]
                == threshold
            ]
        )


        if prefix != "ALL":

            q = q[
                q[
                    "prefix"
                ]
                == prefix
            ]


        activation_rows_1005.append({
            "threshold":
                threshold,

            "prefix":
                prefix,

            "datasets":
                len(q),

            "datasets_with_any_ignored_peak":
                int(
                    (
                        q[
                            "ignored_local_peaks"
                        ]
                        > 0
                    )
                    .sum()
                ),

            "dataset_activation_rate":
                float(
                    (
                        q[
                            "ignored_local_peaks"
                        ]
                        > 0
                    )
                    .mean()
                ),

            "datasets_with_farther_4um_peak":
                int(
                    (
                        q[
                            "ignored_peaks_farther_4um"
                        ]
                        > 0
                    )
                    .sum()
                ),
        })


activation_summary_1005 = (
    pd.DataFrame(
        activation_rows_1005
    )
)


# ============================================================
# Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.05 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "IGNORE THRESHOLD SUMMARY"
)

display(
    ignore_summary_1005
)


print()
print(
    "ANNOTATED-POSITIVE PROBABILITY DISTRIBUTION"
)

display(
    positive_prob_summary_1005
)


print()
print(
    "DATASET ACTIVATION ROBUSTNESS"
)

display(
    activation_summary_1005
)


print()
print(
    "RAW-LOGIT EQUIVALENTS"
)

display(
    threshold_semantics_1005
)


print()
print(
    "Datasets audited:",
    calibration_frames_1005[
        "dataset"
    ].nunique(),
)

print(
    "Frames audited:",
    calibration_frames_1005[
        [
            "dataset",
            "window_index",
            "frame_index",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

print()
print(
    "No model parameter was updated."
)

print(
    "No graph/file was modified."
)

print(
    "GT was used only for calibration diagnostics."
)

print()
print(
    "10.05 Warm2 sparse-ignore calibration: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.06 Sparse-Ignore Negative Loss / Gradient Mass Audit
# Purpose:
#   Quantify how much of the CURRENT baseline nominal-negative
#   detection supervision would be removed by sparse-ignore
#   probability thresholds.
#
#   For a nominal negative voxel:
#
#       BCE gradient wrt logit = sigmoid(logit)
#
#   Therefore audit BOTH:
#
#       1. negative BCE loss mass removed
#       2. negative gradient mass removed
#
#   Thresholds:
#       0.9975
#       0.9990
#       0.9995
#       0.9999
#
#   Same deterministic Fold0 validation sampling as 10.05:
#       - 40 datasets
#       - first + last training-style window
#       - 160 frames total
#
# Changes files: NO
# Runs training: NO
# Runs inference: YES — Warm2 forward passes only
# Uses GT: YES — only to exclude annotated-positive voxels
# Keep after running: YES — Stage-10 loss calibration evidence
# ============================================================

import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


# ============================================================
# Preconditions from 10.05
# ============================================================

assert "model_1005" in globals()
assert "train_mod_1005" in globals()
assert "VAL_NAMES" in globals()
assert "TRAIN_ROOT" in globals()
assert "DOWNSAMPLE" in globals()
assert "WINDOW_SIZE" in globals()
assert "DEVICE" in globals()

assert len(VAL_NAMES) == 40
assert WINDOW_SIZE == 2


# ============================================================
# Threshold grid
# ============================================================

THRESHOLDS_1006 = [
    0.9975,
    0.9990,
    0.9995,
    0.9999,
]


threshold_table_1006 = pd.DataFrame({
    "threshold":
        THRESHOLDS_1006,

    "raw_logit":
        [
            math.log(
                p / (1.0 - p)
            )
            for p in THRESHOLDS_1006
        ],
})


# ============================================================
# Helpers
# ============================================================

def dataset_path_1006(name):

    name = str(name)

    if name.endswith(".zarr"):
        p = TRAIN_ROOT / name
    else:
        p = TRAIN_ROOT / f"{name}.zarr"

    assert p.exists(), p

    return p


def prefix_1006(name):

    return str(name).split("_")[0]


# ============================================================
# Audit
# ============================================================

rows_1006 = []


print(
    "=== 10.06 NEGATIVE LOSS / GRADIENT MASS AUDIT START ==="
)


model_1005.eval()


with torch.no_grad():

    for dataset_i, name in enumerate(
        VAL_NAMES,
        start=1,
    ):

        path = dataset_path_1006(
            name
        )


        # ----------------------------------------------------
        # Exact same training-style windows as 10.05
        # ----------------------------------------------------

        video_meta, windows = (
            train_mod_1005.load_dataset_windows(
                path,
                window_size=WINDOW_SIZE,
                max_frames=None,
                downsample=DOWNSAMPLE,
            )
        )


        if len(windows) == 0:
            continue


        if len(windows) == 1:

            selected_indices = [
                0
            ]

        else:

            selected_indices = [
                0,
                len(windows) - 1,
            ]


        selected_windows = [
            windows[i]
            for i in selected_indices
        ]


        max_nodes = max(
            max(w.node_counts)
            for w in selected_windows
        )


        ds = (
            train_mod_1005.FrameWindowDataset(
                [
                    (
                        video_meta,
                        selected_windows,
                    )
                ],
                max_nodes=max_nodes,
                augmentations=None,
            )
        )


        for sample_i in range(
            len(ds)
        ):

            item = ds[
                sample_i
            ]


            imgs = (
                item["imgs"]
                .unsqueeze(0)
                .to(
                    DEVICE,
                    dtype=torch.float32,
                )
            )


            _, det_logits_list = (
                model_1005.encode(
                    imgs
                )
            )


            for frame_i in range(
                WINDOW_SIZE
            ):

                logits = (
                    det_logits_list[
                        frame_i
                    ][0, 0]
                    .detach()
                    .float()
                    .cpu()
                )


                probs = torch.sigmoid(
                    logits
                )


                # ====================================================
                # Construct positive mask exactly like training loss
                # ====================================================

                coords = (
                    item[
                        "coords"
                    ][
                        frame_i
                    ]
                )

                mask = (
                    item[
                        "masks"
                    ][
                        frame_i
                    ]
                )


                gt_coords = (
                    coords[
                        mask
                    ]
                    .float()
                )


                spatial = logits.shape


                zi = (
                    gt_coords[:, 0]
                    .long()
                    .clamp(
                        0,
                        spatial[0] - 1,
                    )
                )

                yi = (
                    gt_coords[:, 1]
                    .long()
                    .clamp(
                        0,
                        spatial[1] - 1,
                    )
                )

                xi = (
                    gt_coords[:, 2]
                    .long()
                    .clamp(
                        0,
                        spatial[2] - 1,
                    )
                )


                gt_idx = torch.unique(
                    torch.stack(
                        [
                            zi,
                            yi,
                            xi,
                        ],
                        dim=1,
                    ),
                    dim=0,
                )


                positive = torch.zeros(
                    spatial,
                    dtype=torch.bool,
                )


                positive[
                    gt_idx[:, 0],
                    gt_idx[:, 1],
                    gt_idx[:, 2],
                ] = True


                negative = (
                    ~positive
                )


                # ====================================================
                # Baseline negative BCE
                #
                # BCEWithLogits(logit, target=0)
                # ====================================================

                negative_logits = (
                    logits[
                        negative
                    ]
                )


                negative_probs = (
                    probs[
                        negative
                    ]
                )


                negative_bce = (
                    F.softplus(
                        negative_logits
                    )
                )


                # Gradient d BCE(z,0)/dz = sigmoid(z)
                negative_grad = (
                    negative_probs
                )


                total_neg_voxels = int(
                    negative.sum().item()
                )


                total_bce_mass = float(
                    negative_bce
                    .sum()
                    .item()
                )


                total_grad_mass = float(
                    negative_grad
                    .sum()
                    .item()
                )


                # ====================================================
                # Threshold-specific removal
                # ====================================================

                for threshold in (
                    THRESHOLDS_1006
                ):

                    ignored = (
                        negative_probs
                        >= threshold
                    )


                    ignored_voxels = int(
                        ignored
                        .sum()
                        .item()
                    )


                    ignored_bce_mass = float(
                        negative_bce[
                            ignored
                        ]
                        .sum()
                        .item()
                    )


                    ignored_grad_mass = float(
                        negative_grad[
                            ignored
                        ]
                        .sum()
                        .item()
                    )


                    rows_1006.append({

                        "dataset":
                            name,

                        "prefix":
                            prefix_1006(
                                name
                            ),

                        "window_index":
                            int(
                                selected_indices[
                                    sample_i
                                ]
                            ),

                        "frame_index":
                            int(
                                frame_i
                            ),

                        "threshold":
                            float(
                                threshold
                            ),

                        "negative_voxels":
                            total_neg_voxels,

                        "ignored_voxels":
                            ignored_voxels,

                        "ignored_voxel_fraction":
                            (
                                ignored_voxels
                                / max(
                                    total_neg_voxels,
                                    1,
                                )
                            ),

                        "total_negative_bce_mass":
                            total_bce_mass,

                        "ignored_negative_bce_mass":
                            ignored_bce_mass,

                        "removed_bce_fraction":
                            (
                                ignored_bce_mass
                                / total_bce_mass
                                if total_bce_mass > 0
                                else np.nan
                            ),

                        "total_negative_gradient_mass":
                            total_grad_mass,

                        "ignored_negative_gradient_mass":
                            ignored_grad_mass,

                        "removed_gradient_fraction":
                            (
                                ignored_grad_mass
                                / total_grad_mass
                                if total_grad_mass > 0
                                else np.nan
                            ),
                    })


        if (
            dataset_i == 1
            or dataset_i % 10 == 0
            or dataset_i == len(
                VAL_NAMES
            )
        ):

            print(
                f"Processed "
                f"{dataset_i:02d}/"
                f"{len(VAL_NAMES)}"
            )


mass_audit_1006 = pd.DataFrame(
    rows_1006
)


assert len(
    mass_audit_1006
) > 0


# ============================================================
# Aggregate by threshold / prefix
#
# Use mass-weighted aggregate fractions rather than simply
# averaging per-frame fractions.
# ============================================================

summary_rows_1006 = []


for threshold in (
    THRESHOLDS_1006
):

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        sub = (
            mass_audit_1006[
                mass_audit_1006[
                    "threshold"
                ] == threshold
            ]
        )


        if prefix != "ALL":

            sub = sub[
                sub[
                    "prefix"
                ] == prefix
            ]


        total_voxels = int(
            sub[
                "negative_voxels"
            ].sum()
        )


        ignored_voxels = int(
            sub[
                "ignored_voxels"
            ].sum()
        )


        total_bce = float(
            sub[
                "total_negative_bce_mass"
            ].sum()
        )


        ignored_bce = float(
            sub[
                "ignored_negative_bce_mass"
            ].sum()
        )


        total_grad = float(
            sub[
                "total_negative_gradient_mass"
            ].sum()
        )


        ignored_grad = float(
            sub[
                "ignored_negative_gradient_mass"
            ].sum()
        )


        summary_rows_1006.append({

            "threshold":
                threshold,

            "prefix":
                prefix,

            "frames":
                len(sub),

            "ignored_voxel_fraction":
                (
                    ignored_voxels
                    / max(
                        total_voxels,
                        1,
                    )
                ),

            "removed_negative_BCE_fraction":
                (
                    ignored_bce
                    / total_bce
                    if total_bce > 0
                    else np.nan
                ),

            "removed_negative_gradient_fraction":
                (
                    ignored_grad
                    / total_grad
                    if total_grad > 0
                    else np.nan
                ),

            "median_frame_removed_BCE_fraction":
                float(
                    sub[
                        "removed_bce_fraction"
                    ].median()
                ),

            "median_frame_removed_gradient_fraction":
                float(
                    sub[
                        "removed_gradient_fraction"
                    ].median()
                ),

            "p90_frame_removed_gradient_fraction":
                float(
                    sub[
                        "removed_gradient_fraction"
                    ].quantile(
                        0.90
                    )
                ),
        })


mass_summary_1006 = pd.DataFrame(
    summary_rows_1006
)


# ============================================================
# Dataset-level robustness
# ============================================================

dataset_mass_1006 = (
    mass_audit_1006
    .groupby(
        [
            "threshold",
            "prefix",
            "dataset",
        ],
        as_index=False,
    )
    .agg(
        total_negative_gradient_mass=(
            "total_negative_gradient_mass",
            "sum",
        ),

        ignored_negative_gradient_mass=(
            "ignored_negative_gradient_mass",
            "sum",
        ),

        total_negative_bce_mass=(
            "total_negative_bce_mass",
            "sum",
        ),

        ignored_negative_bce_mass=(
            "ignored_negative_bce_mass",
            "sum",
        ),
    )
)


dataset_mass_1006[
    "removed_gradient_fraction"
] = (
    dataset_mass_1006[
        "ignored_negative_gradient_mass"
    ]
    /
    dataset_mass_1006[
        "total_negative_gradient_mass"
    ]
)


dataset_mass_1006[
    "removed_BCE_fraction"
] = (
    dataset_mass_1006[
        "ignored_negative_bce_mass"
    ]
    /
    dataset_mass_1006[
        "total_negative_bce_mass"
    ]
)


robust_rows_1006 = []


for threshold in (
    THRESHOLDS_1006
):

    for prefix in [
        "ALL",
        "44b6",
        "6bba",
    ]:

        sub = (
            dataset_mass_1006[
                dataset_mass_1006[
                    "threshold"
                ] == threshold
            ]
        )


        if prefix != "ALL":

            sub = sub[
                sub[
                    "prefix"
                ] == prefix
            ]


        robust_rows_1006.append({

            "threshold":
                threshold,

            "prefix":
                prefix,

            "datasets":
                len(sub),

            "median_dataset_removed_gradient":
                float(
                    sub[
                        "removed_gradient_fraction"
                    ].median()
                ),

            "p25_dataset_removed_gradient":
                float(
                    sub[
                        "removed_gradient_fraction"
                    ].quantile(
                        0.25
                    )
                ),

            "p75_dataset_removed_gradient":
                float(
                    sub[
                        "removed_gradient_fraction"
                    ].quantile(
                        0.75
                    )
                ),

            "median_dataset_removed_BCE":
                float(
                    sub[
                        "removed_BCE_fraction"
                    ].median()
                ),
        })


dataset_robustness_1006 = (
    pd.DataFrame(
        robust_rows_1006
    )
)


# ============================================================
# Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.06 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "NEGATIVE SUPERVISION MASS REMOVAL"
)

display(
    mass_summary_1006
)


print()
print(
    "DATASET-LEVEL ROBUSTNESS"
)

display(
    dataset_robustness_1006
)


print()
print(
    "THRESHOLD SEMANTICS"
)

display(
    threshold_table_1006
)


print()
print(
    "Datasets audited:",
    mass_audit_1006[
        "dataset"
    ].nunique(),
)

print(
    "Unique frames audited:",
    mass_audit_1006[
        [
            "dataset",
            "window_index",
            "frame_index",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

print()
print(
    "No model parameter was updated."
)

print(
    "No graph/file was modified."
)

print()
print(
    "10.06 negative supervision mass audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.07 Paired Fold0 Sparse-Label Pilot
# Purpose:
#   Run a medium-budget paired training experiment from the
#   EXACT SAME frozen Warm2 checkpoint:
#
#       A. BASELINE_256
#          original detection loss
#
#       B. SPARSE_9975_256
#          det_ignore_prob = 0.9975
#
#   Both runs use:
#       - full Fold0 training set
#       - full Fold0 validation set
#       - 256 optimizer iterations
#       - batch size 8
#       - fresh AdamW
#       - lr = 1e-5
#       - seed = 1640
#       - same architecture / downsample / pool kernel
#       - same Warm2 initialization
#
#   This is still a PILOT, not the final Stage-10 training run.
#
#   IMPORTANT:
#       Training-script validation metrics are diagnostic only.
#       Competition-metric comparison comes AFTER inference.
#
# Changes files: YES
#   - two pilot model directories
#   - two log files
#
# Runs training: YES — 256 iterations × 2 variants
# Uses GT: YES — normal Fold0 training/validation supervision
# Keep after running: YES — keep RESULT; logs/checkpoints retained
#                     until Stage-10 decision is complete
# ============================================================

from pathlib import Path
import os
import re
import sys
import time
import json
import hashlib
import subprocess

import numpy as np
import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

MODEL_ROOT = (
    PROJECT
    / "models/official_baseline"
)

LOG_DIR = (
    PROJECT
    / "reports/stage10_pilot"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Frozen identities
# ============================================================

EXPECTED_WARM2_SHA = (
    "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"
)

EXPECTED_TRAIN_SCRIPT_SHA = (
    "ce4fe8b3a16771039e1c687c02e71cad2747ee48d94417f0a26c929192f1be7b"
)


# ============================================================
# Helpers
# ============================================================

def sha256_file_1007(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# Preconditions
# ============================================================

assert PYTHON.exists()
assert TRAIN_SCRIPT.exists()
assert OFFICIAL_SCRIPTS.exists()
assert TRAIN_ROOT.exists()
assert SPLITS.exists()
assert WARM2.exists()


assert (
    Path(sys.executable).resolve()
    == PYTHON.resolve()
), (
    "Notebook kernel is not the Biohub project .venv."
)


warm2_sha_1007 = (
    sha256_file_1007(
        WARM2
    )
)

assert (
    warm2_sha_1007
    == EXPECTED_WARM2_SHA
), (
    "Warm2 SHA mismatch."
)


train_sha_1007 = (
    sha256_file_1007(
        TRAIN_SCRIPT
    )
)

assert (
    train_sha_1007
    == EXPECTED_TRAIN_SCRIPT_SHA
), (
    "Training-script SHA mismatch."
)


# ============================================================
# Verify Fold0 manifest
# ============================================================

split_obj = json.loads(
    SPLITS.read_text(
        encoding="utf-8"
    )
)


if isinstance(
    split_obj,
    dict,
):

    if "folds" in split_obj:
        folds = split_obj["folds"]

    elif "splits" in split_obj:
        folds = split_obj["splits"]

    else:
        raise ValueError(
            "No folds/splits field."
        )

else:
    folds = split_obj


fold0 = folds[0]

assert len(
    fold0["train"]
) == 159

assert len(
    fold0["test"]
) == 40


print(
    "Fold0:",
    len(fold0["train"]),
    "train /",
    len(fold0["test"]),
    "validation",
)


# ============================================================
# Subprocess environment
# ============================================================

env = os.environ.copy()


existing_pythonpath = (
    env.get(
        "PYTHONPATH",
        ""
    )
)


env["PYTHONPATH"] = (
    str(OFFICIAL_SCRIPTS)
    + (
        os.pathsep
        + existing_pythonpath
        if existing_pythonpath
        else ""
    )
)


# ============================================================
# Fixed paired-pilot configuration
# ============================================================

MAX_ITERS = 256
LR = 1e-5
BATCH_SIZE = 8
SEED = 1640


COMMON_ARGS = [

    str(PYTHON),
    str(TRAIN_SCRIPT),

    "--data-dir",
    str(TRAIN_ROOT),

    "--splits",
    str(SPLITS),

    "--split",
    "0",

    "--epochs",
    "1",

    "--max-iters",
    str(MAX_ITERS),

    "--lr",
    str(LR),

    "--batch-size",
    str(BATCH_SIZE),

    "--num-workers",
    "4",

    "--seed",
    str(SEED),

    "--downsample",
    "1,4,4",

    "--window-size",
    "2",

    "--det-loss-weight",
    "1.0",

    "--det-neg-weight",
    "0.01",

    "--pool-kernel-um",
    "5.0",

    "--init-weights",
    str(WARM2),

    "--single-gpu",
]


VARIANTS_1007 = {

    "BASELINE_256": {
        "method":
            "stage10_pilot_baseline_256",

        "extra_args":
            [],
    },

    "SPARSE_9975_256": {
        "method":
            "stage10_pilot_sparse9975_256",

        "extra_args":
            [
                "--det-ignore-prob",
                "0.9975",
            ],
    },
}


# ============================================================
# Safety:
# refuse to silently overwrite an existing pilot.
#
# If this Cell ever has to be rerun after a partial failure,
# inspect the existing artifacts first instead of deleting them.
# ============================================================

for variant, cfg in (
    VARIANTS_1007.items()
):

    output_dir = (
        MODEL_ROOT
        / cfg["method"]
        / "split_0"
    )

    assert not output_dir.exists(), (
        f"Pilot output already exists:\n"
        f"{output_dir}\n\n"
        "Do not overwrite it silently."
    )


# ============================================================
# Parsers
# ============================================================

epoch_pattern = re.compile(
    r"Epoch\s+\d+/\d+\s+\|\s+"
    r"edge=([0-9.eE+-]+)\s+\|\s+"
    r"det=([0-9.eE+-]+)\s+\|\s+"
    r"test_loss=([0-9.eE+-]+)\s+\|\s+"
    r"acc=([0-9.eE+-]+)\s+\|\s+"
    r"recall=([0-9.eE+-]+)"
)


save_pattern = re.compile(
    r"saved to\s+(.+?edge_predictor_best\.pth)"
)


# ============================================================
# Run paired pilot
# ============================================================

rows_1007 = []


print()
print("=" * 72)
print("10.07 PAIRED FOLD0 PILOT START")
print("=" * 72)

print(
    "Warm2 SHA256:",
    warm2_sha_1007,
)

print(
    "Training-script SHA256:",
    train_sha_1007,
)

print(
    "Iterations per variant:",
    MAX_ITERS,
)


for variant, cfg in (
    VARIANTS_1007.items()
):

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )


    assert not log_path.exists(), (
        f"Pilot log already exists:\n"
        f"{log_path}"
    )


    cmd = (
        COMMON_ARGS
        + [
            "--method",
            cfg["method"],
        ]
        + cfg["extra_args"]
    )


    print()
    print(
        f"Starting {variant}..."
    )

    print(
        "  log:",
        log_path,
    )


    start_wall = time.time()


    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_f:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            text=True,
        )


    elapsed = (
        time.time()
        - start_wall
    )


    log_text = (
        log_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )


    # --------------------------------------------------------
    # Configuration checks
    # --------------------------------------------------------

    if variant == "BASELINE_256":

        mode_ok = (
            "ignore_prob=None"
            in log_text
        )

    else:

        mode_ok = (
            "ignore_prob=0.9975"
            in log_text
        )


    warm_start_ok = (
        "Full-model init weights loaded:"
        in log_text
    )


    fold_ok = (
        "Fold 0: 159 train, 40 test"
        in log_text
    )


    # --------------------------------------------------------
    # Parse epoch summary
    # --------------------------------------------------------

    matches = list(
        epoch_pattern.finditer(
            log_text
        )
    )


    epoch_found = (
        len(matches) > 0
    )


    if epoch_found:

        m = matches[-1]

        edge_loss = float(
            m.group(1)
        )

        det_loss = float(
            m.group(2)
        )

        test_loss = float(
            m.group(3)
        )

        test_acc = float(
            m.group(4)
        )

        test_recall = float(
            m.group(5)
        )

    else:

        edge_loss = np.nan
        det_loss = np.nan
        test_loss = np.nan
        test_acc = np.nan
        test_recall = np.nan


    # --------------------------------------------------------
    # Locate checkpoint
    # --------------------------------------------------------

    save_matches = list(
        save_pattern.finditer(
            log_text
        )
    )


    checkpoint = None


    if save_matches:

        checkpoint = Path(
            save_matches[-1]
            .group(1)
            .strip()
        )


        if not checkpoint.is_absolute():

            checkpoint = (
                PROJECT
                / checkpoint
            )


        checkpoint = (
            checkpoint.resolve()
        )


    checkpoint_exists = bool(
        checkpoint is not None
        and checkpoint.exists()
    )


    checkpoint_fresh = False
    checkpoint_sha = None


    if checkpoint_exists:

        checkpoint_fresh = (
            checkpoint.stat().st_mtime
            >= start_wall - 2.0
        )

        checkpoint_sha = (
            sha256_file_1007(
                checkpoint
            )
        )


    rows_1007.append({

        "variant":
            variant,

        "return_code":
            proc.returncode,

        "mode_ok":
            mode_ok,

        "warm_start_ok":
            warm_start_ok,

        "fold_ok":
            fold_ok,

        "epoch_found":
            epoch_found,

        "checkpoint_exists":
            checkpoint_exists,

        "checkpoint_fresh":
            checkpoint_fresh,

        "edge_loss":
            edge_loss,

        "det_loss":
            det_loss,

        "test_loss":
            test_loss,

        "test_acc":
            test_acc,

        "test_recall":
            test_recall,

        "diagnostic_acc_x_recall":
            (
                test_acc
                * test_recall
            ),

        "elapsed_sec":
            elapsed,

        "checkpoint":
            (
                str(checkpoint)
                if checkpoint is not None
                else None
            ),

        "checkpoint_sha256":
            checkpoint_sha,

        "log":
            str(log_path),
    })


    print(
        f"{variant}: "
        f"return={proc.returncode}, "
        f"elapsed={elapsed / 60:.1f} min"
    )


    # --------------------------------------------------------
    # Stop immediately if first variant failed.
    # --------------------------------------------------------

    if proc.returncode != 0:

        print()
        print(
            "FAILED LOG TAIL:"
        )

        print(
            "\n".join(
                log_text.splitlines()[
                    -60:
                ]
            )
        )

        raise RuntimeError(
            f"{variant} pilot failed."
        )


# ============================================================
# Verification
# ============================================================

pilot_results_1007 = pd.DataFrame(
    rows_1007
)


for col in [
    "return_code",
]:

    assert (
        pilot_results_1007[
            col
        ] == 0
    ).all()


for col in [
    "mode_ok",
    "warm_start_ok",
    "fold_ok",
    "epoch_found",
    "checkpoint_exists",
    "checkpoint_fresh",
]:

    assert (
        pilot_results_1007[
            col
        ]
    ).all(), (
        f"10.07 verification failed: {col}"
    )


for col in [
    "edge_loss",
    "det_loss",
    "test_loss",
    "test_acc",
    "test_recall",
]:

    assert np.isfinite(
        pilot_results_1007[
            col
        ].to_numpy(
            dtype=float
        )
    ).all()


# ============================================================
# Paired diagnostic deltas
#
# NOT competition metric.
# ============================================================

baseline_row = (
    pilot_results_1007[
        pilot_results_1007[
            "variant"
        ] == "BASELINE_256"
    ]
    .iloc[0]
)


sparse_row = (
    pilot_results_1007[
        pilot_results_1007[
            "variant"
        ] == "SPARSE_9975_256"
    ]
    .iloc[0]
)


diagnostic_delta_1007 = {
    "delta_edge_loss_sparse_minus_baseline":
        (
            sparse_row[
                "edge_loss"
            ]
            - baseline_row[
                "edge_loss"
            ]
        ),

    "delta_det_loss_sparse_minus_baseline":
        (
            sparse_row[
                "det_loss"
            ]
            - baseline_row[
                "det_loss"
            ]
        ),

    "delta_test_acc_sparse_minus_baseline":
        (
            sparse_row[
                "test_acc"
            ]
            - baseline_row[
                "test_acc"
            ]
        ),

    "delta_test_recall_sparse_minus_baseline":
        (
            sparse_row[
                "test_recall"
            ]
            - baseline_row[
                "test_recall"
            ]
        ),

    "delta_acc_x_recall_sparse_minus_baseline":
        (
            sparse_row[
                "diagnostic_acc_x_recall"
            ]
            - baseline_row[
                "diagnostic_acc_x_recall"
            ]
        ),
}


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("10.07 RESULT")
print("=" * 72)


display(
    pilot_results_1007[
        [
            "variant",
            "return_code",
            "edge_loss",
            "det_loss",
            "test_loss",
            "test_acc",
            "test_recall",
            "diagnostic_acc_x_recall",
            "elapsed_sec",
        ]
    ]
)


print()
print(
    "DIAGNOSTIC DELTAS"
)

for k, v in (
    diagnostic_delta_1007.items()
):

    print(
        f"  {k}: {v:+.8f}"
    )


print()
print(
    "CHECKPOINTS"
)

for row in (
    pilot_results_1007
    .itertuples(
        index=False
    )
):

    print()
    print(
        row.variant
    )

    print(
        " ",
        row.checkpoint
    )

    print(
        "  SHA256=",
        row.checkpoint_sha256,
        sep="",
    )


print()
print(
    "IMPORTANT:"
)

print(
    "  These validation metrics are NOT the competition metric."
)

print(
    "  Do not select the sparse model from acc/recall alone."
)

print(
    "  Next decision requires identical Fold0 inference + "
    "baseline-v9 graph evaluation."
)

print()
print(
    "10.07 paired Fold0 pilot: COMPLETE"
)

### 10.07 — Paired Fold0 Sparse-Label Pilot

Both models were warm-started from the identical frozen Warm2 checkpoint and trained for 256 optimizer iterations on the same Fold0 data with the same seed, LR, batch size, optimizer, and training configuration.

| Variant | Edge loss | Det loss | Val acc | Val recall | Acc × Recall |
|---|---:|---:|---:|---:|---:|
| Baseline-256 | 0.0005 | 0.0106 | 0.9997 | 0.9170 | 0.916725 |
| Sparse-9975-256 | 0.0005 | 0.0087 | 0.9997 | 0.9193 | 0.919024 |

Sparse-label deltas:
- Detection loss: -0.0019
- Validation accuracy: +0.0000
- Validation recall: +0.0023
- Accuracy × recall: +0.002299

Checkpoint identities:
- Baseline-256: `8bddad68aaabf240d36eab9cda158f12d4969fa1cd6b21277570043ef83ab689`
- Sparse-9975-256: `dd39a621dbf371da2707712ea6ddb4f5b94190b0ad208560f88a795bc00f61a9`

Interpretation:
The sparse-label loss produced a positive diagnostic recall signal without reducing validation association accuracy. These diagnostics are not the competition metric. No model-selection decision is made until both checkpoints are evaluated through identical Fold0 inference and the frozen baseline-v9 graph pipeline.

In [ ]:
# ============================================================
# SETUP — 10.08 Paired Pilot Inference Source / CLI Audit
# Purpose:
#   Audit the existing prediction adapter before launching the
#   expensive paired Fold0 inference for:
#
#       BASELINE_256
#       SPARSE_9975_256
#
#   Confirm:
#       - prediction-script identity
#       - checkpoint identities
#       - prediction CLI
#       - checkpoint/config arguments
#       - dataset/input arguments
#       - detection-threshold argument
#       - output-path semantics
#       - pool-kernel argument
#       - device behavior
#
#   Import environment explicitly includes:
#       project/scripts
#       external/official/scripts
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — Stage-10 inference provenance
# ============================================================

from pathlib import Path
import subprocess
import sys
import hashlib
import ast
import os


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

SCRIPTS_DIR = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

PREDICT_SCRIPT = (
    SCRIPTS_DIR
    / "predict_unet_transformer_mps.py"
)

BASELINE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

SPARSE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_sparse9975_256/"
      "split_0/edge_predictor_best.pth"
)


# ============================================================
# Helpers
# ============================================================

def sha256_file_1008(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# Preconditions
# ============================================================

assert PROJECT.exists()
assert PYTHON.exists()
assert SCRIPTS_DIR.exists()
assert OFFICIAL_SCRIPTS.exists()
assert PREDICT_SCRIPT.exists()
assert BASELINE_256.exists()
assert SPARSE_256.exists()


assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON.resolve()
), (
    "Notebook kernel is not the Biohub project .venv.\n"
    f"Notebook: {sys.executable}\n"
    f"Expected: {PYTHON}"
)


# ============================================================
# Checkpoint identities
# ============================================================

baseline_sha = (
    sha256_file_1008(
        BASELINE_256
    )
)

sparse_sha = (
    sha256_file_1008(
        SPARSE_256
    )
)


assert (
    baseline_sha
    ==
    "8bddad68aaabf240d36eab9cda158f12d4969fa1cd6b21277570043ef83ab689"
), (
    "BASELINE_256 checkpoint SHA mismatch."
)


assert (
    sparse_sha
    ==
    "dd39a621dbf371da2707712ea6ddb4f5b94190b0ad208560f88a795bc00f61a9"
), (
    "SPARSE_9975_256 checkpoint SHA mismatch."
)


# ============================================================
# Static syntax check
# ============================================================

source = (
    PREDICT_SCRIPT.read_text(
        encoding="utf-8"
    )
)

ast.parse(
    source
)


# ============================================================
# Script identity
# ============================================================

predict_sha = (
    sha256_file_1008(
        PREDICT_SCRIPT
    )
)


print(
    "=== 10.08 PAIRED INFERENCE AUDIT ==="
)

print()

print(
    "Prediction script:",
    PREDICT_SCRIPT,
)

print(
    "Prediction-script SHA256:",
    predict_sha,
)

print()

print(
    "BASELINE_256 SHA256:",
    baseline_sha,
)

print(
    "SPARSE_9975_256 SHA256:",
    sparse_sha,
)


# ============================================================
# Explicit subprocess import environment
#
# IMPORTANT:
# train_unet_transformer_mps.py imports modules such as
# augmentations using bare imports.
#
# Keep BOTH project scripts and locked official scripts on
# PYTHONPATH for prediction-adapter import resolution.
# ============================================================

env = (
    os.environ.copy()
)


pythonpath_parts = [
    str(
        SCRIPTS_DIR
    ),
    str(
        OFFICIAL_SCRIPTS
    ),
]


existing_pythonpath = (
    env.get(
        "PYTHONPATH",
        ""
    )
)


if existing_pythonpath:

    pythonpath_parts.append(
        existing_pythonpath
    )


env[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts
)


print()
print(
    "Subprocess PYTHONPATH:"
)

for part in pythonpath_parts:
    print(
        " ",
        part,
    )


# ============================================================
# Direct import sanity test
#
# This verifies the exact dependency chain that failed before:
#
#   prediction adapter
#       -> training adapter
#       -> augmentations
# ============================================================

import_test = subprocess.run(
    [
        str(
            PYTHON
        ),
        "-c",
        (
            "import sys; "
            "import augmentations; "
            "import train_unet_transformer_mps; "
            "print('PYTHON=' + sys.executable); "
            "print('AUGMENTATIONS=' + augmentations.__file__); "
            "print('TRAIN_MODULE=' + "
            "train_unet_transformer_mps.__file__)"
        ),
    ],
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


assert (
    import_test.returncode
    == 0
), (
    "Prediction dependency import test failed:\n\n"
    + import_test.stdout
)


print()
print(
    "Dependency import test:",
    "PASS",
)

print(
    import_test.stdout.strip()
)


# ============================================================
# Exact prediction CLI help
# ============================================================

proc = subprocess.run(
    [
        str(
            PYTHON
        ),
        str(
            PREDICT_SCRIPT
        ),
        "--help",
    ],
    cwd=PROJECT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)


assert (
    proc.returncode
    == 0
), (
    "Prediction CLI --help failed:\n\n"
    + proc.stdout
)


print()
print(
    "=" * 72
)

print(
    "PREDICTION CLI"
)

print(
    "=" * 72
)

print(
    proc.stdout
)


# ============================================================
# Relevant source fragments
# ============================================================

SEARCH_TERMS = [
    "ArgumentParser",
    "add_argument",

    "threshold",
    "det_threshold",

    "pool_kernel",
    "pool-kernel",

    "checkpoint",
    "weights",

    "config.json",

    "output",
    "prediction",

    "device",
    "mps",

    "data_dir",
    "data-dir",

    "dataset",
]


lines = (
    source.splitlines()
)


print()
print(
    "=" * 72
)

print(
    "RELEVANT SOURCE CONTEXT"
)

print(
    "=" * 72
)


# ============================================================
# Merge overlapping matching line ranges so output stays
# readable instead of printing the same block repeatedly.
# ============================================================

raw_ranges = []


for i, line in enumerate(
    lines
):

    lower = line.lower()

    if not any(
        term.lower()
        in lower
        for term in SEARCH_TERMS
    ):
        continue


    start = max(
        0,
        i - 3,
    )

    end = min(
        len(
            lines
        ),
        i + 6,
    )


    raw_ranges.append(
        (
            start,
            end,
        )
    )


merged_ranges = []


for start, end in sorted(
    raw_ranges
):

    if (
        not merged_ranges
        or start
        > merged_ranges[-1][1]
    ):

        merged_ranges.append(
            [
                start,
                end,
            ]
        )

    else:

        merged_ranges[-1][1] = max(
            merged_ranges[-1][1],
            end,
        )


for start, end in (
    merged_ranges
):

    print()

    print(
        f"--- lines "
        f"{start + 1}:{end} ---"
    )


    for j in range(
        start,
        end,
    ):

        print(
            f"{j + 1:04d}: "
            f"{lines[j]}"
        )


# ============================================================
# Compact automatic CLI feature checks
# ============================================================

cli_text = (
    proc.stdout
)


candidate_flags = [
    "--weights",
    "--checkpoint",
    "--model",

    "--data-dir",
    "--dataset",

    "--output",
    "--output-dir",

    "--threshold",
    "--det-threshold",

    "--pool-kernel-um",

    "--device",
]


flag_rows = []


for flag in (
    candidate_flags
):

    flag_rows.append(
        (
            flag,
            flag in cli_text,
        )
    )


print()
print(
    "=" * 72
)

print(
    "CLI FLAG PRESENCE"
)

print(
    "=" * 72
)


for flag, present in (
    flag_rows
):

    print(
        f"{flag:<20} "
        f"{present}"
    )


# ============================================================
# Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.08 RESULT"
)

print(
    "=" * 72
)


print(
    "Prediction script syntax:",
    "PASS",
)

print(
    "Prediction dependency imports:",
    "PASS",
)

print(
    "Prediction CLI:",
    "PASS",
)

print(
    "Pilot checkpoint identities:",
    "PASS",
)

print()

print(
    "No inference was run."
)

print(
    "No file was modified."
)

print()

print(
    "10.08 paired-inference audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.09 Paired Fold0 Full Inference
# Purpose:
#   Run IDENTICAL full Fold0 inference for the two Stage-10
#   256-iteration pilot checkpoints:
#
#       A. BASELINE_256
#       B. SPARSE_9975_256
#
#   Frozen inference policy:
#       det threshold = 0.995
#       detection TTA = ON (prediction-script default)
#       pool kernel = 3.0 um (PredictConfig default)
#       edge activation = softmax
#       edge threshold = 0.5
#       ILP = OFF
#
#   The only model-quality difference between the two runs is
#   the checkpoint.
#
#   Predictions are saved separately for later application of
#   the exact frozen baseline-v9 graph pipeline.
#
# Changes files: YES
#   - prediction GEFFs for two methods
#   - two inference logs
#
# Runs training: NO
# Runs inference: YES — full 40-volume Fold0 × 2
# Uses GT: NO during prediction
# Keep after running: YES — keep RESULT; prediction artifacts
#                     are required for 10.10 evaluation
# ============================================================

from pathlib import Path
import os
import sys
import re
import time
import json
import hashlib
import subprocess

import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

PREDICT_SCRIPT = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

SCRIPTS_DIR = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

BASELINE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

SPARSE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_sparse9975_256/"
      "split_0/edge_predictor_best.pth"
)

LOG_DIR = (
    PROJECT
    / "reports/stage10_inference"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Frozen identities
# ============================================================

EXPECTED_PREDICT_SHA = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)

EXPECTED_BASELINE_SHA = (
    "8bddad68aaabf240d36eab9cda158f12d4969fa1cd6b21277570043ef83ab689"
)

EXPECTED_SPARSE_SHA = (
    "dd39a621dbf371da2707712ea6ddb4f5b94190b0ad208560f88a795bc00f61a9"
)


# ============================================================
# Helpers
# ============================================================

def sha256_file_1009(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# Preconditions
# ============================================================

assert PYTHON.exists()
assert PREDICT_SCRIPT.exists()
assert SCRIPTS_DIR.exists()
assert OFFICIAL_SCRIPTS.exists()
assert TRAIN_ROOT.exists()
assert SPLITS.exists()
assert BASELINE_256.exists()
assert SPARSE_256.exists()


assert (
    Path(sys.executable).resolve()
    == PYTHON.resolve()
), (
    "Notebook kernel is not the Biohub project .venv."
)


assert (
    sha256_file_1009(
        PREDICT_SCRIPT
    )
    == EXPECTED_PREDICT_SHA
), (
    "Prediction-script SHA mismatch."
)


assert (
    sha256_file_1009(
        BASELINE_256
    )
    == EXPECTED_BASELINE_SHA
), (
    "Baseline checkpoint SHA mismatch."
)


assert (
    sha256_file_1009(
        SPARSE_256
    )
    == EXPECTED_SPARSE_SHA
), (
    "Sparse checkpoint SHA mismatch."
)


# ============================================================
# Verify checkpoint configs exist and match
# ============================================================

BASELINE_CONFIG = (
    BASELINE_256.parent
    / "config.json"
)

SPARSE_CONFIG = (
    SPARSE_256.parent
    / "config.json"
)


assert BASELINE_CONFIG.exists()
assert SPARSE_CONFIG.exists()


baseline_cfg_1009 = json.loads(
    BASELINE_CONFIG.read_text(
        encoding="utf-8"
    )
)

sparse_cfg_1009 = json.loads(
    SPARSE_CONFIG.read_text(
        encoding="utf-8"
    )
)


assert (
    baseline_cfg_1009
    == sparse_cfg_1009
), (
    "Pilot checkpoint architecture configs differ."
)


print(
    "Checkpoint config equality:",
    "PASS"
)

print(
    "Config:",
    baseline_cfg_1009,
)


# ============================================================
# Verify Fold0 has 40 validation videos
# ============================================================

folds = json.loads(
    SPLITS.read_text(
        encoding="utf-8"
    )
)


assert isinstance(
    folds,
    list,
), (
    "Prediction adapter directly expects a list-style split file."
)


assert len(
    folds[0]["test"]
) == 40


print(
    "Fold0 validation videos:",
    len(
        folds[0]["test"]
    ),
)


# ============================================================
# Subprocess environment
# ============================================================

env = os.environ.copy()


pythonpath_parts = [
    str(
        SCRIPTS_DIR
    ),
    str(
        OFFICIAL_SCRIPTS
    ),
]


existing_pythonpath = (
    env.get(
        "PYTHONPATH",
        ""
    )
)


if existing_pythonpath:

    pythonpath_parts.append(
        existing_pythonpath
    )


env[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts
)


# ============================================================
# Frozen inference configuration
# ============================================================

DET_THRESHOLD = 0.995
UNET_BATCH_SIZE = 4


COMMON_ARGS = [

    str(
        PYTHON
    ),

    str(
        PREDICT_SCRIPT
    ),

    "--data-dir",
    str(
        TRAIN_ROOT
    ),

    "--splits",
    str(
        SPLITS
    ),

    "--split",
    "0",

    "--det-threshold",
    str(
        DET_THRESHOLD
    ),

    "--unet-batch-size",
    str(
        UNET_BATCH_SIZE
    ),
]


# ============================================================
# Variants
# ============================================================

VARIANTS_1009 = {

    "BASELINE_256": {

        "method":
            "stage10_infer_baseline_256_det995",

        "weights":
            BASELINE_256,
    },


    "SPARSE_9975_256": {

        "method":
            "stage10_infer_sparse9975_256_det995",

        "weights":
            SPARSE_256,
    },
}


# ============================================================
# Determine prediction root from script
#
# We do NOT assume USERNAME here.
# Existing output is detected later from the script's
# "Saved 40 predictions to ..." line.
# ============================================================

save_pattern = re.compile(
    r"Saved\s+(\d+)\s+predictions\s+to\s+(.+)"
)


start_pattern = re.compile(
    r"Fold\s+0:\s+40 datasets\s+\|\s+"
    r"weights=(.+?)\s+\|\s+device=(\w+)\s+\|\s+"
    r"window_size=(\d+)\s+\|\s+pool_kernel_um=([0-9.]+)"
)


# ============================================================
# Safety:
# refuse to reuse Stage-10 inference logs.
#
# Prediction script itself deletes prior .geff files inside an
# output method directory, so we prevent accidental reruns by
# refusing to start if this Stage-10 log already exists.
# ============================================================

for variant in (
    VARIANTS_1009
):

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )

    assert not log_path.exists(), (
        f"Inference log already exists:\n"
        f"{log_path}\n\n"
        "Do not silently rerun an expensive full inference."
    )


# ============================================================
# Run paired inference
# ============================================================

rows_1009 = []


print()
print("=" * 72)
print("10.09 PAIRED FOLD0 FULL INFERENCE START")
print("=" * 72)

print(
    "Prediction-script SHA256:",
    EXPECTED_PREDICT_SHA,
)

print(
    "Detection threshold:",
    DET_THRESHOLD,
)

print(
    "UNet batch size:",
    UNET_BATCH_SIZE,
)

print(
    "ILP:",
    False,
)

print(
    "Expected pool kernel:",
    "3.0 um",
)


for variant, cfg in (
    VARIANTS_1009.items()
):

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )


    cmd = (
        COMMON_ARGS
        + [
            "--method",
            cfg["method"],

            "--weights",
            str(
                cfg["weights"]
            ),
        ]
    )


    print()
    print(
        f"Starting {variant}..."
    )

    print(
        "  checkpoint:",
        cfg["weights"],
    )

    print(
        "  log:",
        log_path,
    )


    start_wall = time.time()


    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_f:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            text=True,
        )


    elapsed = (
        time.time()
        - start_wall
    )


    log_text = (
        log_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )


    # --------------------------------------------------------
    # Failure handling
    # --------------------------------------------------------

    if proc.returncode != 0:

        print()
        print(
            f"{variant} FAILED"
        )

        print()
        print(
            "\n".join(
                log_text.splitlines()[
                    -80:
                ]
            )
        )

        raise RuntimeError(
            f"{variant} inference failed."
        )


    # --------------------------------------------------------
    # Parse start provenance
    # --------------------------------------------------------

    start_matches = list(
        start_pattern.finditer(
            log_text
        )
    )


    start_ok = (
        len(
            start_matches
        )
        > 0
    )


    if start_ok:

        sm = (
            start_matches[
                -1
            ]
        )

        logged_weights = (
            sm.group(1)
            .strip()
        )

        device = (
            sm.group(2)
        )

        window_size = int(
            sm.group(3)
        )

        pool_kernel_um = float(
            sm.group(4)
        )

    else:

        logged_weights = None
        device = None
        window_size = None
        pool_kernel_um = None


    # --------------------------------------------------------
    # Parse save result
    # --------------------------------------------------------

    save_matches = list(
        save_pattern.finditer(
            log_text
        )
    )


    save_ok = (
        len(
            save_matches
        )
        > 0
    )


    if save_ok:

        save_match = (
            save_matches[
                -1
            ]
        )

        n_saved_reported = int(
            save_match.group(1)
        )

        output_dir = Path(
            save_match.group(2)
            .strip()
        )


        if not output_dir.is_absolute():

            output_dir = (
                PROJECT
                / output_dir
            )


        output_dir = (
            output_dir.resolve()
        )

    else:

        n_saved_reported = 0
        output_dir = None


    # --------------------------------------------------------
    # Inspect actual GEFF outputs
    # --------------------------------------------------------

    geffs = []


    if (
        output_dir is not None
        and output_dir.exists()
    ):

        geffs = sorted(
            output_dir.glob(
                "*.geff"
            )
        )


    actual_geffs = len(
        geffs
    )


    # --------------------------------------------------------
    # Formal provenance checks
    # --------------------------------------------------------

    weights_ok = bool(
        logged_weights is not None
        and Path(
            logged_weights
        ).resolve()
        == cfg[
            "weights"
        ].resolve()
    )


    pool_ok = (
        pool_kernel_um
        == 3.0
    )


    complete = (
        proc.returncode == 0
        and start_ok
        and save_ok
        and weights_ok
        and pool_ok
        and n_saved_reported == 40
        and actual_geffs == 40
    )


    rows_1009.append({

        "variant":
            variant,

        "return_code":
            proc.returncode,

        "weights_ok":
            weights_ok,

        "device":
            device,

        "window_size":
            window_size,

        "pool_kernel_um":
            pool_kernel_um,

        "det_threshold":
            DET_THRESHOLD,

        "n_saved_reported":
            n_saved_reported,

        "actual_geffs":
            actual_geffs,

        "complete":
            complete,

        "elapsed_sec":
            elapsed,

        "output_dir":
            (
                str(
                    output_dir
                )
                if output_dir is not None
                else None
            ),

        "log":
            str(
                log_path
            ),
    })


    print(
        f"{variant}: "
        f"return={proc.returncode}, "
        f"saved={actual_geffs}/40, "
        f"elapsed={elapsed / 60:.1f} min"
    )


# ============================================================
# Formal verification
# ============================================================

inference_results_1009 = (
    pd.DataFrame(
        rows_1009
    )
)


assert (
    inference_results_1009[
        "complete"
    ]
).all(), (
    "At least one paired inference failed provenance/completeness checks."
)


assert (
    inference_results_1009[
        "device"
    ] == "mps"
).all(), (
    "Expected both runs to use MPS."
)


assert (
    inference_results_1009[
        "window_size"
    ]
    == inference_results_1009[
        "window_size"
    ].iloc[0]
).all()


assert (
    inference_results_1009[
        "pool_kernel_um"
    ] == 3.0
).all()


assert (
    inference_results_1009[
        "det_threshold"
    ] == 0.995
).all()


# ============================================================
# Ensure the two output directories differ
# ============================================================

assert (
    inference_results_1009[
        "output_dir"
    ].nunique()
    == 2
), (
    "Baseline and sparse predictions must be isolated."
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("10.09 RESULT")
print("=" * 72)


display(
    inference_results_1009[
        [
            "variant",
            "return_code",
            "weights_ok",
            "device",
            "window_size",
            "pool_kernel_um",
            "det_threshold",
            "actual_geffs",
            "elapsed_sec",
            "output_dir",
        ]
    ]
)


print()
print(
    "PAIRED INFERENCE CONDITIONS"
)

print(
    "  Same Fold0 validation set: TRUE"
)

print(
    "  Same prediction adapter: TRUE"
)

print(
    "  Same det threshold: 0.995"
)

print(
    "  Same pool kernel: 3.0 um"
)

print(
    "  Same TTA policy: TRUE"
)

print(
    "  Same edge policy: TRUE"
)

print(
    "  ILP used: FALSE"
)

print(
    "  Only checkpoint differs: TRUE"
)


print()
print(
    "Baseline predictions:",
    inference_results_1009.loc[
        inference_results_1009[
            "variant"
        ] == "BASELINE_256",
        "output_dir",
    ].iloc[0],
)

print(
    "Sparse predictions:",
    inference_results_1009.loc[
        inference_results_1009[
            "variant"
        ] == "SPARSE_9975_256",
        "output_dir",
    ].iloc[0],
)


print()
print(
    "No competition-metric model decision made yet."
)

print(
    "Next: apply the exact frozen baseline-v9 graph pipeline "
    "to BOTH prediction sets."
)

print()
print(
    "10.09 paired Fold0 full inference: COMPLETE"
)

In [ ]:
# ============================================================
# SETUP — 10.10 Frozen Baseline-v9 Implementation Audit
# Purpose:
#   Locate the EXACT existing notebook implementation of the
#   frozen baseline-v9 graph pipeline before applying it to the
#   new Stage-10 paired predictions.
#
#   Frozen target pipeline:
#
#       G2 division
#           ->
#       K9 confidence-aware pruning
#           ->
#       mutual single-edge recovery, residual <= 4 um
#           ->
#       strict existing-node two-edge recovery, residual <= 4 um
#           ->
#       singleton + long 2-node cleanup
#
#   This Cell searches existing Stage 07/08/09 notebooks for:
#       - function definitions
#       - constants / thresholds
#       - pipeline-order calls
#       - v8/v9 evaluation cells
#
#   It does NOT recreate or modify any implementation.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES — provenance for exact v9 reuse
# ============================================================

from pathlib import Path
import json
import ast
import re

import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

NOTEBOOKS = [
    PROJECT / "notebooks/07_detection_threshold_sweep.ipynb",
    PROJECT / "notebooks/08_training_improvement.ipynb",
    PROJECT / "notebooks/09_gap_recovery.ipynb",
]


for nb in NOTEBOOKS:
    assert nb.exists(), nb


# ============================================================
# Search vocabulary
# ============================================================

SEARCH_TERMS = [
    "baseline-v9",
    "baseline_v9",
    "v9",

    "G2",
    "division",
    "p2",

    "K9",
    "confidence",
    "pruning",

    "single-edge",
    "single_edge",
    "mutual",
    "residual",
    "ambiguity",

    "two-edge",
    "two_edge",
    "isolated",
    "pair_count",
    "conflict_free",

    "cleanup",
    "singleton",
    "2-node",
    "two_node",
    "3.5",

    "0.80",
    "0.9975",
    "4.0",
    "8.0",
    "60",
    "120",
]


# ============================================================
# Helpers
# ============================================================

def source_text(cell):
    src = cell.get(
        "source",
        []
    )

    if isinstance(src, list):
        return "".join(src)

    return str(src)


def function_defs(text):
    try:
        tree = ast.parse(text)
    except Exception:
        return []

    names = []

    for node in ast.walk(tree):
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            names.append(
                node.name
            )

    return names


def assignment_names(text):
    try:
        tree = ast.parse(text)
    except Exception:
        return []

    names = []

    for node in ast.walk(tree):

        if isinstance(
            node,
            ast.Assign,
        ):
            for target in node.targets:
                if isinstance(
                    target,
                    ast.Name,
                ):
                    names.append(
                        target.id
                    )

        elif isinstance(
            node,
            ast.AnnAssign,
        ):
            if isinstance(
                node.target,
                ast.Name,
            ):
                names.append(
                    node.target.id
                )

    return names


# ============================================================
# Scan notebooks
# ============================================================

rows_1010 = []


for nb_path in NOTEBOOKS:

    notebook = json.loads(
        nb_path.read_text(
            encoding="utf-8"
        )
    )


    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        text = source_text(
            cell
        )

        lower = text.lower()


        matched = [
            term
            for term in SEARCH_TERMS
            if term.lower() in lower
        ]


        if not matched:
            continue


        cell_type = cell.get(
            "cell_type",
            "unknown"
        )


        defs = (
            function_defs(text)
            if cell_type == "code"
            else []
        )


        assigns = (
            assignment_names(text)
            if cell_type == "code"
            else []
        )


        # ----------------------------------------------------
        # Extra score for actual implementation evidence.
        # ----------------------------------------------------

        score = len(
            matched
        )


        if defs:
            score += 8


        if any(
            key in lower
            for key in [
                "baseline-v9",
                "baseline_v9",
                "score",
                "evaluate",
                "apply",
            ]
        ):
            score += 4


        if all(
            key in lower
            for key in [
                "single",
                "two",
                "cleanup",
            ]
        ):
            score += 5


        first_nonempty = ""

        for line in text.splitlines():

            if line.strip():

                first_nonempty = (
                    line.strip()
                )

                break


        rows_1010.append({

            "notebook":
                nb_path.name,

            "cell":
                cell_index,

            "type":
                cell_type,

            "score":
                score,

            "matched_terms":
                ", ".join(
                    matched
                ),

            "function_defs":
                ", ".join(
                    defs
                ),

            "assignments":
                ", ".join(
                    assigns[:15]
                ),

            "first_line":
                first_nonempty[:120],
        })


audit_1010 = (
    pd.DataFrame(
        rows_1010
    )
)


assert len(
    audit_1010
) > 0


audit_1010 = (
    audit_1010
    .sort_values(
        [
            "score",
            "notebook",
            "cell",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# Candidate implementation cells only
# ============================================================

code_candidates_1010 = (
    audit_1010[
        audit_1010[
            "type"
        ] == "code"
    ]
    .copy()
)


implementation_candidates_1010 = (
    code_candidates_1010[
        (
            code_candidates_1010[
                "function_defs"
            ] != ""
        )
        |
        (
            code_candidates_1010[
                "score"
            ] >= 8
        )
    ]
    .head(
        25
    )
)


# ============================================================
# Search specifically for known frozen-v9 constants / order
# ============================================================

TARGET_PATTERNS = {
    "division_p2_080":
        r"0\.80",

    "high_conf_09975":
        r"0\.9975",

    "residual_4um":
        r"4(?:\.0)?",

    "max_step_8um":
        r"8(?:\.0)?",

    "turn_60deg":
        r"60",

    "division_angle_120":
        r"120",

    "cleanup_3p5um":
        r"3\.5",

    "pair_count":
        r"pair_count",

    "conflict_free":
        r"conflict[_ -]?free",

    "mutual":
        r"mutual",
}


constant_rows_1010 = []


for nb_path in NOTEBOOKS:

    notebook = json.loads(
        nb_path.read_text(
            encoding="utf-8"
        )
    )


    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        text = source_text(
            cell
        )


        hits = [
            name
            for name, pattern in TARGET_PATTERNS.items()
            if re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
        ]


        if len(
            hits
        ) < 3:
            continue


        constant_rows_1010.append({

            "notebook":
                nb_path.name,

            "cell":
                cell_index,

            "type":
                cell.get(
                    "cell_type",
                    "unknown"
                ),

            "frozen_v9_hits":
                ", ".join(
                    hits
                ),

            "hit_count":
                len(
                    hits
                ),

            "first_line":
                next(
                    (
                        line.strip()
                        for line in text.splitlines()
                        if line.strip()
                    ),
                    "",
                )[:120],
        })


constant_candidates_1010 = (
    pd.DataFrame(
        constant_rows_1010
    )
)


if len(
    constant_candidates_1010
):

    constant_candidates_1010 = (
        constant_candidates_1010
        .sort_values(
            [
                "hit_count",
                "notebook",
                "cell",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.10 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "TOP IMPLEMENTATION CANDIDATES"
)

display(
    implementation_candidates_1010[
        [
            "notebook",
            "cell",
            "score",
            "function_defs",
            "assignments",
            "matched_terms",
            "first_line",
        ]
    ]
)


print()
print(
    "FROZEN-V9 CONSTANT / ORDER CANDIDATES"
)


if len(
    constant_candidates_1010
):

    display(
        constant_candidates_1010[
            [
                "notebook",
                "cell",
                "hit_count",
                "frozen_v9_hits",
                "first_line",
            ]
        ].head(
            25
        )
    )

else:

    print(
        "No multi-constant candidate cells found."
    )


print()
print(
    "Candidate code cells:",
    len(
        code_candidates_1010
    ),
)

print(
    "Implementation candidates shown:",
    len(
        implementation_candidates_1010
    ),
)


print()
print(
    "No graph processing was run."
)

print(
    "No GT was used."
)

print(
    "No file was modified."
)

print()
print(
    "10.10 frozen baseline-v9 implementation audit: COMPLETE"
)

In [ ]:
# ============================================================
# SETUP — 10.12 Frozen-v9 Prediction-Bound Dependency Audit
# Purpose:
#   Locate the EXACT source cells that GENERATE a baseline-v9
#   pipeline from an arbitrary prediction directory.
#
#   Critical distinction:
#
#       REUSABLE:
#           geometry / graph functions / frozen constants
#
#       NOT REUSABLE ACROSS MODELS:
#           single_policy_0915
#           strict_two_policy_0915
#           isolated_bridge_candidates_0910
#           any candidate table containing old prediction node_ids
#
#   We need to identify the upstream generators so Stage-10 can:
#
#       BASELINE_256 predictions
#           -> regenerate G2/K9 graph
#           -> regenerate single candidates
#           -> regenerate two-edge candidates
#           -> frozen V9
#
#       SPARSE_9975_256 predictions
#           -> SAME procedure independently
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Runs graph processing: NO
# Uses GT: NO
# Keep after running: YES — provenance for model-agnostic V9 rebuild
# ============================================================

from pathlib import Path
import json
import ast
import re
import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

NB07_PATH = (
    PROJECT
    / "notebooks/07_detection_threshold_sweep.ipynb"
)

NB09_PATH = (
    PROJECT
    / "notebooks/09_gap_recovery.ipynb"
)

assert NB07_PATH.exists()
assert NB09_PATH.exists()


# ============================================================
# Load notebooks
# ============================================================

NB07 = json.loads(
    NB07_PATH.read_text(
        encoding="utf-8"
    )
)

NB09 = json.loads(
    NB09_PATH.read_text(
        encoding="utf-8"
    )
)


NOTEBOOKS_1012 = {
    "07_detection_threshold_sweep.ipynb":
        NB07,

    "09_gap_recovery.ipynb":
        NB09,
}


def cell_source_1012(
    cell,
):

    src = cell.get(
        "source",
        []
    )

    if isinstance(
        src,
        list,
    ):
        return "".join(
            src
        )

    return str(
        src
    )


# ============================================================
# Build top-level symbol index
#
# We index:
#   - function definitions
#   - class definitions
#   - assignments
#
# This tells us WHERE every important V9 object originates.
# ============================================================

symbol_rows_1012 = []


for nb_name, notebook in (
    NOTEBOOKS_1012.items()
):

    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        if (
            cell.get(
                "cell_type"
            )
            != "code"
        ):
            continue


        source = (
            cell_source_1012(
                cell
            )
        )


        try:

            tree = ast.parse(
                source
            )

        except Exception:

            continue


        for node in (
            tree.body
        ):

            # ------------------------------------------------
            # Function / class definition
            # ------------------------------------------------

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            ):

                symbol_rows_1012.append({

                    "symbol":
                        node.name,

                    "kind":
                        type(
                            node
                        ).__name__,

                    "notebook":
                        nb_name,

                    "cell":
                        cell_index,
                })


            # ------------------------------------------------
            # Normal assignment
            # ------------------------------------------------

            elif isinstance(
                node,
                ast.Assign,
            ):

                for target in (
                    node.targets
                ):

                    if isinstance(
                        target,
                        ast.Name,
                    ):

                        symbol_rows_1012.append({

                            "symbol":
                                target.id,

                            "kind":
                                "Assign",

                            "notebook":
                                nb_name,

                            "cell":
                                cell_index,
                        })


            # ------------------------------------------------
            # Annotated assignment
            # ------------------------------------------------

            elif isinstance(
                node,
                ast.AnnAssign,
            ):

                if isinstance(
                    node.target,
                    ast.Name,
                ):

                    symbol_rows_1012.append({

                        "symbol":
                            node.target.id,

                        "kind":
                            "AnnAssign",

                        "notebook":
                            nb_name,

                        "cell":
                            cell_index,
                    })


symbols_1012 = (
    pd.DataFrame(
        symbol_rows_1012
    )
)


assert len(
    symbols_1012
) > 0


# ============================================================
# Exact / partial V9 targets
# ============================================================

TARGET_PATTERNS_1012 = [

    # --------------------------------------------------------
    # G2
    # --------------------------------------------------------

    r"apply_frozen_g2",

    # --------------------------------------------------------
    # K9
    # --------------------------------------------------------

    r"apply_low_conf",
    r"confidence.*prun",

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    r"apply_frozen_component_cleanup",

    # --------------------------------------------------------
    # Base pre-cleanup graph
    # --------------------------------------------------------

    r"prepare_precleanup_graph",

    # --------------------------------------------------------
    # Single-edge candidate generation / application
    # --------------------------------------------------------

    r"gap.*census",
    r"candidate.*census",
    r"build_mutual_policy",
    r"add_available_repairs",
    r"recovered_edge_dist",

    # --------------------------------------------------------
    # Two-edge candidate generation / application
    # --------------------------------------------------------

    r"isolated.*bridge",
    r"bridge.*census",
    r"turn_angle",
    r"step_balance",
    r"add_two_edge_bridges",
    r"build_strict_two_edge_policy",

    # --------------------------------------------------------
    # Metric
    # --------------------------------------------------------

    r"evaluate_graph",
]


target_mask_1012 = []


for symbol in (
    symbols_1012[
        "symbol"
    ]
):

    matched = any(
        re.search(
            pattern,
            symbol,
            flags=re.IGNORECASE,
        )
        for pattern in TARGET_PATTERNS_1012
    )

    target_mask_1012.append(
        matched
    )


target_symbols_1012 = (
    symbols_1012[
        target_mask_1012
    ]
    .sort_values(
        [
            "notebook",
            "cell",
            "symbol",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# Locate prediction-bound globals / candidate tables
# ============================================================

PREDICTION_BOUND_PATTERNS_1012 = [

    r"^PRED",
    r".*_PRED.*",

    r".*candidate.*",
    r".*policy.*",

    r"single_policy",
    r"two_policy",

    r"isolated_bridge",
    r"conservative_bridge",
]


prediction_bound_rows_1012 = []


for row in (
    symbols_1012
    .itertuples(
        index=False
    )
):

    if not any(
        re.search(
            pattern,
            row.symbol,
            flags=re.IGNORECASE,
        )
        for pattern in PREDICTION_BOUND_PATTERNS_1012
    ):
        continue


    prediction_bound_rows_1012.append({

        "symbol":
            row.symbol,

        "kind":
            row.kind,

        "notebook":
            row.notebook,

        "cell":
            row.cell,
    })


prediction_bound_1012 = (
    pd.DataFrame(
        prediction_bound_rows_1012
    )
)


if len(
    prediction_bound_1012
):

    prediction_bound_1012 = (
        prediction_bound_1012
        .sort_values(
            [
                "notebook",
                "cell",
                "symbol",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# Find cells containing the exact OLD prediction-bound tables.
# ============================================================

OLD_TABLES_1012 = [

    "single_policy_0912",
    "single_policy_0915",

    "strict_two_policy_0915",

    "isolated_bridge_candidates_0910",

    "conservative_bridge_0910",

    "gap_candidates",
]


old_table_cell_rows_1012 = []


for nb_name, notebook in (
    NOTEBOOKS_1012.items()
):

    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        if (
            cell.get(
                "cell_type"
            )
            != "code"
        ):
            continue


        source = (
            cell_source_1012(
                cell
            )
        )


        hits = [
            name
            for name in OLD_TABLES_1012
            if name in source
        ]


        if not hits:
            continue


        old_table_cell_rows_1012.append({

            "notebook":
                nb_name,

            "cell":
                cell_index,

            "tables":
                ", ".join(
                    hits
                ),

            "first_line":
                next(
                    (
                        line.strip()
                        for line in source.splitlines()
                        if line.strip()
                    ),
                    "",
                )[:120],
        })


old_table_cells_1012 = (
    pd.DataFrame(
        old_table_cell_rows_1012
    )
)


# ============================================================
# Identify cells that appear to LOAD prediction GEFFs.
#
# These are the cells that must eventually be retargeted from
# old Warm2 prediction directory to Stage-10 prediction dirs.
# ============================================================

prediction_loader_rows_1012 = []


LOAD_TERMS_1012 = [
    "PRED_",
    "prediction",
    "load_graph",
    ".geff",
]


for nb_name, notebook in (
    NOTEBOOKS_1012.items()
):

    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        )
    ):

        if (
            cell.get(
                "cell_type"
            )
            != "code"
        ):
            continue


        source = (
            cell_source_1012(
                cell
            )
        )


        lower = source.lower()


        if (
            "load_graph" not in lower
            or ".geff" not in lower
        ):
            continue


        prediction_refs = sorted(
            set(
                re.findall(
                    r"\bPRED[A-Z0-9_]*\b",
                    source,
                )
            )
        )


        first_line = next(
            (
                line.strip()
                for line in source.splitlines()
                if line.strip()
            ),
            "",
        )


        prediction_loader_rows_1012.append({

            "notebook":
                nb_name,

            "cell":
                cell_index,

            "prediction_globals":
                ", ".join(
                    prediction_refs
                ),

            "first_line":
                first_line[:120],
        })


prediction_loaders_1012 = (
    pd.DataFrame(
        prediction_loader_rows_1012
    )
)


# ============================================================
# Compact dependency cell inventory
#
# Focus specifically on Stage09 upstream producer cells.
# ============================================================

candidate_cells = set()


for df in [
    target_symbols_1012,
    prediction_bound_1012,
]:

    if len(df):

        for row in (
            df.itertuples(
                index=False
            )
        ):

            if (
                row.notebook
                ==
                "09_gap_recovery.ipynb"
            ):

                candidate_cells.add(
                    int(
                        row.cell
                    )
                )


# ============================================================
# Print FIRST LINES + function / assignment inventory of those
# candidate cells.
# ============================================================

cell_inventory_rows_1012 = []


for cell_index in sorted(
    candidate_cells
):

    cell = (
        NB09[
            "cells"
        ][cell_index]
    )


    source = (
        cell_source_1012(
            cell
        )
    )


    funcs = []

    assigns = []


    try:

        tree = ast.parse(
            source
        )


        for node in tree.body:

            if isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            ):

                funcs.append(
                    node.name
                )


            elif isinstance(
                node,
                ast.Assign,
            ):

                for target in (
                    node.targets
                ):

                    if isinstance(
                        target,
                        ast.Name,
                    ):

                        assigns.append(
                            target.id
                        )


    except Exception:
        pass


    first_line = next(
        (
            line.strip()
            for line in source.splitlines()
            if line.strip()
        ),
        "",
    )


    cell_inventory_rows_1012.append({

        "cell":
            cell_index,

        "functions":
            ", ".join(
                funcs
            ),

        "assignments":
            ", ".join(
                assigns
            ),

        "first_line":
            first_line[:140],
    })


stage09_cell_inventory_1012 = (
    pd.DataFrame(
        cell_inventory_rows_1012
    )
)


# ============================================================
# Explicit safety statement
# ============================================================

print()
print(
    "=" * 78
)

print(
    "10.12 RESULT"
)

print(
    "=" * 78
)


print()
print(
    "V9 RELEVANT SYMBOL DEFINITIONS"
)

display(
    target_symbols_1012
)


print()
print(
    "PREDICTION-BOUND SYMBOLS"
)

display(
    prediction_bound_1012
)


print()
print(
    "OLD WARM2 TABLE USAGE"
)

display(
    old_table_cells_1012
)


print()
print(
    "PREDICTION GRAPH LOAD CELLS"
)

display(
    prediction_loaders_1012
)


print()
print(
    "STAGE09 PRODUCER CELL INVENTORY"
)

display(
    stage09_cell_inventory_1012
)


print()
print(
    "IMPORTANT:"
)

print(
    "  Existing Stage09 candidate/policy tables will NOT be "
    "reused for Stage10."
)

print(
    "  Candidate geometry must be regenerated independently "
    "from each Stage10 prediction graph."
)

print()
print(
    "No notebook source was executed."
)

print(
    "No prediction graph was modified."
)

print(
    "No GT was used."
)

print(
    "No file was modified."
)

print()
print(
    "10.12 model-agnostic v9 dependency audit: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.14 Paired Fold0 High-Confidence Inference
# Purpose:
#   Generate the det=0.9975 high-confidence prediction graphs
#   required by the frozen baseline-v9 K9 pruning stage for:
#
#       A. BASELINE_256
#       B. SPARSE_9975_256
#
#   These predictions are NOT independently evaluated.
#   They are used only as the high-confidence coordinate source
#   for K9 when rebuilding baseline-v9.
#
# Frozen inference conditions:
#       det threshold = 0.9975
#       detection TTA = ON
#       pool kernel = 3.0 um
#       edge activation = softmax
#       ILP = OFF
#
# Changes files: YES
#   - 40 high-confidence GEFFs per model
#   - two inference logs
#
# Runs training: NO
# Runs inference: YES — full Fold0 × 2
# Uses GT: NO
# Keep after running: YES — required by Stage10 V9 evaluation
# ============================================================

from pathlib import Path
import os
import sys
import re
import time
import hashlib
import subprocess

import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = PROJECT / ".venv/bin/python"

PREDICT_SCRIPT = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

SCRIPTS_DIR = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

BASELINE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

SPARSE_256 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_sparse9975_256/"
      "split_0/edge_predictor_best.pth"
)

LOG_DIR = (
    PROJECT
    / "reports/stage10_inference"
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Frozen identities
# ============================================================

EXPECTED_PREDICT_SHA = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)

EXPECTED_BASELINE_SHA = (
    "8bddad68aaabf240d36eab9cda158f12d4969fa1cd6b21277570043ef83ab689"
)

EXPECTED_SPARSE_SHA = (
    "dd39a621dbf371da2707712ea6ddb4f5b94190b0ad208560f88a795bc00f61a9"
)


def sha256_file_1014(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# Preconditions
# ============================================================

assert Path(sys.executable).resolve() == PYTHON.resolve()

assert (
    sha256_file_1014(PREDICT_SCRIPT)
    == EXPECTED_PREDICT_SHA
)

assert (
    sha256_file_1014(BASELINE_256)
    == EXPECTED_BASELINE_SHA
)

assert (
    sha256_file_1014(SPARSE_256)
    == EXPECTED_SPARSE_SHA
)


# ============================================================
# Import environment
# ============================================================

env = os.environ.copy()

parts = [
    str(SCRIPTS_DIR),
    str(OFFICIAL_SCRIPTS),
]

if env.get("PYTHONPATH"):
    parts.append(
        env["PYTHONPATH"]
    )

env["PYTHONPATH"] = (
    os.pathsep.join(parts)
)


# ============================================================
# Frozen high-confidence inference configuration
# ============================================================

DET_THRESHOLD = 0.9975
UNET_BATCH_SIZE = 4


COMMON_ARGS = [
    str(PYTHON),
    str(PREDICT_SCRIPT),

    "--data-dir",
    str(TRAIN_ROOT),

    "--splits",
    str(SPLITS),

    "--split",
    "0",

    "--det-threshold",
    str(DET_THRESHOLD),

    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
]


VARIANTS_1014 = {

    "BASELINE_256_HIGH": {
        "method":
            "stage10_infer_baseline_256_det9975",
        "weights":
            BASELINE_256,
    },

    "SPARSE_9975_256_HIGH": {
        "method":
            "stage10_infer_sparse9975_256_det9975",
        "weights":
            SPARSE_256,
    },
}


# ============================================================
# Parse prediction output
# ============================================================

save_pattern = re.compile(
    r"Saved\s+(\d+)\s+predictions\s+to\s+(.+)"
)

start_pattern = re.compile(
    r"Fold\s+0:\s+40 datasets\s+\|\s+"
    r"weights=(.+?)\s+\|\s+device=(\w+)\s+\|\s+"
    r"window_size=(\d+)\s+\|\s+pool_kernel_um=([0-9.]+)"
)


# ============================================================
# Safety — no silent rerun
# ============================================================

for variant in VARIANTS_1014:

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )

    assert not log_path.exists(), (
        f"Existing log found:\n{log_path}\n"
        "Do not silently overwrite this expensive inference."
    )


# ============================================================
# Run
# ============================================================

rows_1014 = []

print()
print("=" * 72)
print("10.14 PAIRED HIGH-CONFIDENCE INFERENCE START")
print("=" * 72)

print(
    "Detection threshold:",
    DET_THRESHOLD,
)

print(
    "Expected pool kernel:",
    "3.0 um",
)

print(
    "ILP:",
    False,
)


for variant, cfg in VARIANTS_1014.items():

    log_path = (
        LOG_DIR
        / f"{variant.lower()}.log"
    )

    cmd = (
        COMMON_ARGS
        + [
            "--method",
            cfg["method"],

            "--weights",
            str(cfg["weights"]),
        ]
    )

    print()
    print(
        f"Starting {variant}..."
    )

    print(
        "  log:",
        log_path,
    )

    start_wall = time.time()

    with log_path.open(
        "w",
        encoding="utf-8",
    ) as f:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env,
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed = (
        time.time()
        - start_wall
    )

    log_text = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    if proc.returncode != 0:

        print(
            "\n".join(
                log_text.splitlines()[-80:]
            )
        )

        raise RuntimeError(
            f"{variant} inference failed."
        )


    # --------------------------------------------------------
    # Provenance
    # --------------------------------------------------------

    start_matches = list(
        start_pattern.finditer(log_text)
    )

    assert start_matches

    sm = start_matches[-1]

    logged_weights = Path(
        sm.group(1).strip()
    ).resolve()

    device = sm.group(2)

    window_size = int(
        sm.group(3)
    )

    pool_kernel_um = float(
        sm.group(4)
    )


    assert (
        logged_weights
        == cfg["weights"].resolve()
    )

    assert device == "mps"
    assert pool_kernel_um == 3.0


    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    save_matches = list(
        save_pattern.finditer(log_text)
    )

    assert save_matches

    save_match = save_matches[-1]

    n_reported = int(
        save_match.group(1)
    )

    output_dir = Path(
        save_match.group(2).strip()
    )

    if not output_dir.is_absolute():

        output_dir = (
            PROJECT
            / output_dir
        )

    output_dir = output_dir.resolve()

    geffs = sorted(
        output_dir.glob("*.geff")
    )

    assert n_reported == 40
    assert len(geffs) == 40


    rows_1014.append({
        "variant":
            variant,

        "return_code":
            proc.returncode,

        "device":
            device,

        "window_size":
            window_size,

        "pool_kernel_um":
            pool_kernel_um,

        "det_threshold":
            DET_THRESHOLD,

        "actual_geffs":
            len(geffs),

        "elapsed_sec":
            elapsed,

        "output_dir":
            str(output_dir),
    })


    print(
        f"{variant}: "
        f"saved={len(geffs)}/40, "
        f"elapsed={elapsed / 60:.1f} min"
    )


# ============================================================
# Verification
# ============================================================

highconf_results_1014 = (
    pd.DataFrame(rows_1014)
)

assert (
    highconf_results_1014[
        "actual_geffs"
    ] == 40
).all()

assert (
    highconf_results_1014[
        "det_threshold"
    ] == 0.9975
).all()

assert (
    highconf_results_1014[
        "pool_kernel_um"
    ] == 3.0
).all()

assert (
    highconf_results_1014[
        "output_dir"
    ].nunique()
    == 2
)


# ============================================================
# Result
# ============================================================

print()
print("=" * 72)
print("10.14 RESULT")
print("=" * 72)

display(
    highconf_results_1014[
        [
            "variant",
            "return_code",
            "device",
            "window_size",
            "pool_kernel_um",
            "det_threshold",
            "actual_geffs",
            "elapsed_sec",
            "output_dir",
        ]
    ]
)

print()
print(
    "High-confidence graphs required by K9: COMPLETE"
)

print(
    "No GT was used."
)

print(
    "No metric decision was made."
)

print()
print(
    "10.14 paired det=.9975 inference: COMPLETE"
)

In [ ]:
# ============================================================
# PATCH — 10.15B Exact Metric Wrapper Repair — FINAL
# Purpose:
#   Complete ONLY the final Warm2 baseline-v9 metric
#   reproduction using the already-successful 10.15A state.
#
#   10.15A has already reproduced:
#       single policy          = 3945
#       isolated nodes         = 70018
#       isolated local pairs   = 22085
#       strict two policy      = 2289
#
#   This Cell restores the exact frozen Stage09 metric path:
#
#       prediction graph
#           -> frozen V9 graph processing
#           -> official_evaluate(...)
#           -> node_recall(...)
#           -> per_sample_metrics(...)
#           -> summarise(...)
#
#   IMPORTANT:
#       - Do NOT rerun candidate generation.
#       - Do NOT reuse old Stage09 candidate tables.
#       - division_jaccard is a RUN-LEVEL summarise() output,
#         not a required per-sample metric key.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Runs graph processing: YES — Warm2 V9 reproduction only
# Uses GT: YES — frozen Fold0 metric reproduction only
# Keep after running: NO — DELETE after reproduction PASS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)


# ============================================================
# 1. Preconditions
#
# Preserve the successful 10.15A state.
# ============================================================

REQUIRED_1015B = [
    "PROJECT",

    "dataset_names_1015",
    "prefix_map_1015",

    "WARM2_0995_1015",
    "WARM2_09975_1015",

    "single_policy_1015",
    "strict_two_policy_1015",

    "prepare_precleanup_graph_1015",

    "add_available_repairs_0907",
    "add_two_edge_bridges_0912",
    "apply_frozen_component_cleanup",

    "load_graph",
    "TRAIN_DIR",

    "import_function_1015",
    "nb09_1015",
]


missing_1015b = [
    name
    for name in REQUIRED_1015B
    if name not in globals()
]


assert not missing_1015b, (
    "Required 10.15A state is missing:\n"
    + "\n".join(
        missing_1015b
    )
)


# ============================================================
# 2. Preserve candidate-count reproduction
# ============================================================

assert (
    len(
        dataset_names_1015
    )
    == 40
)


assert (
    len(
        single_policy_1015
    )
    == 3945
), (
    "10.15A single-policy state changed."
)


assert (
    len(
        strict_two_policy_1015
    )
    == 2289
), (
    "10.15A strict-two-policy state changed."
)


print(
    "10.15A preserved:",
    "PASS"
)

print(
    "  datasets:",
    len(
        dataset_names_1015
    )
)

print(
    "  single policy:",
    len(
        single_policy_1015
    )
)

print(
    "  strict two policy:",
    len(
        strict_two_policy_1015
    )
)


# ============================================================
# 3. Restore exact estimated-node metadata
#
# Stage09 per_sample_metrics() uses estimated number of nodes
# for the adjusted Edge Jaccard calculation.
# ============================================================

CV_MANIFEST_1015B = (
    PROJECT
    / "configs/cv_manifest.csv"
)


assert (
    CV_MANIFEST_1015B.exists()
), (
    f"Missing CV manifest:\n"
    f"{CV_MANIFEST_1015B}"
)


cv_meta_1015b = pd.read_csv(
    CV_MANIFEST_1015B
)


required_manifest_cols_1015b = {
    "dataset",
    "fold",
    "estimated_nodes",
}


assert (
    required_manifest_cols_1015b
    .issubset(
        cv_meta_1015b.columns
    )
), (
    "cv_manifest.csv is missing required columns:\n"
    f"{required_manifest_cols_1015b - set(cv_meta_1015b.columns)}"
)


fold0_meta_1015b = (
    cv_meta_1015b[
        cv_meta_1015b[
            "fold"
        ] == 0
    ]
    .copy()
)


assert (
    len(
        fold0_meta_1015b
    )
    == 40
), (
    f"Expected 40 Fold0 datasets, got "
    f"{len(fold0_meta_1015b)}"
)


assert (
    set(
        fold0_meta_1015b[
            "dataset"
        ]
    )
    ==
    set(
        dataset_names_1015
    )
), (
    "Fold0 manifest dataset names do not match prediction split."
)


assert (
    fold0_meta_1015b[
        "estimated_nodes"
    ]
    .notna()
    .all()
), (
    "Missing estimated_nodes value in Fold0 manifest."
)


# IMPORTANT:
# Use exact global name expected by evaluate_graph_0906.
estimated_map = dict(
    zip(
        fold0_meta_1015b[
            "dataset"
        ],
        fold0_meta_1015b[
            "estimated_nodes"
        ],
    )
)


assert (
    len(
        estimated_map
    )
    == 40
)


print(
    "estimated_map restored:",
    "PASS"
)


# ============================================================
# 4. Restore exact Stage09 metric wrapper
#
# Exact original logic:
#
#   er = official_evaluate(...)
#   recall = node_recall(...)
#   return per_sample_metrics(
#       er,
#       estimated_map[name],
#       recall,
#   )
#
# ============================================================

import_function_1015(
    nb09_1015,
    5,
    "evaluate_graph_0906",
)


metric_dependencies_1015b = [
    "official_evaluate",
    "node_recall",
    "per_sample_metrics",
    "summarise",
    "estimated_map",
    "evaluate_graph_0906",
]


missing_metric_dependencies_1015b = [
    name
    for name in metric_dependencies_1015b
    if name not in globals()
]


assert not missing_metric_dependencies_1015b, (
    "Metric wrapper dependencies missing:\n"
    + "\n".join(
        missing_metric_dependencies_1015b
    )
)


print(
    "Exact Stage09 metric wrapper:",
    "PASS"
)


# ============================================================
# 5. Re-run ONLY the Warm2 V9 metric stage
#
# Candidate generation is NOT rerun.
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.15B — WARM2 BASELINE-v9 METRIC REPRODUCTION [FINAL PATCH]"
)

print(
    "=" * 72
)


metric_records_1015 = []


single_added_total_1015 = 0


two_requested_total_1015 = 0

two_added_total_1015 = 0

two_edges_added_total_1015 = 0


two_missing_endpoint_total_1015 = 0

two_degree_conflict_total_1015 = 0

two_existing_conflict_total_1015 = 0


cleanup_removed_total_1015 = 0


# ============================================================
# 6. Fold0 evaluation
# ============================================================

for i, name in enumerate(
    dataset_names_1015,
    start=1,
):

    # --------------------------------------------------------
    # A. Fresh det=.995 graph
    #    -> G2
    #    -> K9 using matching det=.9975 graph
    # --------------------------------------------------------

    g, scale = (
        prepare_precleanup_graph_1015(
            name,
            WARM2_0995_1015,
            WARM2_09975_1015,
        )
    )


    assert (
        np.asarray(
            scale
        ).shape
        == (
            3,
        )
    )


    # --------------------------------------------------------
    # B. Frozen mutual single-edge recovery FIRST
    # --------------------------------------------------------

    single_sub = (
        single_policy_1015[
            single_policy_1015[
                "dataset"
            ] == name
        ]
        .copy()
    )


    single_stats = (
        add_available_repairs_0907(
            g,
            single_sub,
        )
    )


    assert (
        int(
            single_stats[
                "candidate_edges"
            ]
        )
        ==
        len(
            single_sub
        )
    )


    assert (
        int(
            single_stats[
                "added"
            ]
        )
        ==
        len(
            single_sub
        )
    ), (
        f"{name}: not all frozen single-edge "
        f"repairs were addable.\n"
        f"{single_stats}"
    )


    single_added_total_1015 += int(
        single_stats[
            "added"
        ]
    )


    # --------------------------------------------------------
    # C. Frozen strict two-edge recovery SECOND
    # --------------------------------------------------------

    two_sub = (
        strict_two_policy_1015[
            strict_two_policy_1015[
                "dataset"
            ] == name
        ]
        .copy()
    )


    two_stats = (
        add_two_edge_bridges_0912(
            g,
            two_sub,
        )
    )


    two_requested_total_1015 += int(
        two_stats[
            "requested_bridges"
        ]
    )


    two_added_total_1015 += int(
        two_stats[
            "added_bridges"
        ]
    )


    two_edges_added_total_1015 += int(
        two_stats[
            "edges_added"
        ]
    )


    two_missing_endpoint_total_1015 += int(
        two_stats[
            "missing_endpoint"
        ]
    )


    two_degree_conflict_total_1015 += int(
        two_stats[
            "degree_conflict"
        ]
    )


    two_existing_conflict_total_1015 += int(
        two_stats[
            "existing_edge_conflict"
        ]
    )


    # --------------------------------------------------------
    # D. Frozen component cleanup
    #
    # singleton removal
    # +
    # 2-node component removal if displacement > 3.5 um
    # --------------------------------------------------------

    cleanup_stats = (
        apply_frozen_component_cleanup(
            g,
            scale,
        )
    )


    cleanup_removed_total_1015 += int(
        cleanup_stats[
            "nodes_removed"
        ]
    )


    # --------------------------------------------------------
    # E. Locked official Fold0 metric
    #
    # IMPORTANT:
    # evaluate_graph_0906 returns PER-SAMPLE metric dict.
    #
    # division_jaccard is NOT required here. It is generated
    # later by summarise(metric_records_1015).
    # --------------------------------------------------------

    gt_g = load_graph(
        TRAIN_DIR
        / f"{name}.geff"
    )


    m = (
        evaluate_graph_0906(
            g,
            gt_g,
            scale,
            name,
        )
    )


    assert isinstance(
        m,
        dict,
    ), (
        f"{name}: evaluate_graph_0906 returned "
        f"{type(m)}, expected dict."
    )


    # --------------------------------------------------------
    # Exact minimum schema needed by frozen Stage09 aggregation
    # --------------------------------------------------------

    required_per_sample_keys_1015b = {
        "edge_tp",
        "edge_jaccard",
        "adj_edge_jaccard",
        "node_recall",
        "total_node_ratio",
    }


    missing_per_sample_keys_1015b = (
        required_per_sample_keys_1015b
        - set(
            m.keys()
        )
    )


    assert (
        not missing_per_sample_keys_1015b
    ), (
        f"Missing per-sample metric keys for {name}:\n"
        f"{missing_per_sample_keys_1015b}\n\n"
        f"Available keys:\n"
        f"{sorted(m.keys())}"
    )


    metric_records_1015.append(
        m
    )


    # --------------------------------------------------------
    # Concise progress only
    # --------------------------------------------------------

    if (
        i == 1
        or i % 10 == 0
        or i == len(
            dataset_names_1015
        )
    ):

        print(
            f"Evaluated "
            f"{i:02d}/"
            f"{len(dataset_names_1015)}"
        )


# ============================================================
# 7. Per-sample record integrity
# ============================================================

assert (
    len(
        metric_records_1015
    )
    == 40
)


assert all(
    isinstance(
        record,
        dict,
    )
    for record in metric_records_1015
)


# Show schema once for provenance.
sample_metric_keys_1015b = sorted(
    metric_records_1015[
        0
    ].keys()
)


print()
print(
    "Per-sample metric schema:"
)

print(
    sample_metric_keys_1015b
)


# ============================================================
# 8. Aggregate EXACTLY using official summarise()
#
# division_jaccard belongs here at run level.
# ============================================================

summary_1015 = (
    summarise(
        metric_records_1015
    )
)


required_summary_keys_1015b = {
    "edge_jaccard",
    "adj_edge_jaccard",
    "division_jaccard",
    "node_recall",
    "score",
}


missing_summary_keys_1015b = (
    required_summary_keys_1015b
    - set(
        summary_1015.keys()
    )
)


assert (
    not missing_summary_keys_1015b
), (
    "Run-level summary missing expected keys:\n"
    f"{missing_summary_keys_1015b}\n\n"
    f"Available summary keys:\n"
    f"{sorted(summary_1015.keys())}"
)


warm2_v9_score_1015 = float(
    summary_1015[
        "score"
    ]
)


# ============================================================
# 9. Frozen structural reproduction gates
# ============================================================

assert (
    single_added_total_1015
    == 3945
), (
    "Single-edge added-count reproduction failed:\n"
    f"{single_added_total_1015} != 3945"
)


assert (
    two_requested_total_1015
    == 2289
), (
    "Strict two-edge requested-count reproduction failed:\n"
    f"{two_requested_total_1015} != 2289"
)


assert (
    two_added_total_1015
    == 2111
), (
    "Strict two-edge added-count reproduction failed:\n"
    f"{two_added_total_1015} != 2111"
)


assert (
    two_edges_added_total_1015
    == 4222
), (
    "Two-edge association-count reproduction failed:\n"
    f"{two_edges_added_total_1015} != 4222"
)


# ============================================================
# 10. Frozen score reproduction gate
# ============================================================

EXPECTED_WARM2_V9_SCORE_1015B = (
    0.770921
)


score_delta_1015 = (
    warm2_v9_score_1015
    - EXPECTED_WARM2_V9_SCORE_1015B
)


assert (
    abs(
        score_delta_1015
    )
    < 5e-6
), (
    "\n"
    "FROZEN BASELINE-v9 SCORE REPRODUCTION FAILED\n"
    "\n"
    f"Observed: "
    f"{warm2_v9_score_1015:.9f}\n"
    f"Expected: "
    f"{EXPECTED_WARM2_V9_SCORE_1015B:.9f}\n"
    f"Delta:    "
    f"{score_delta_1015:+.9f}\n"
)


# ============================================================
# 11. Prefix diagnostics
#
# These are diagnostics, not additional selection gates.
# ============================================================

prefix_rows_1015b = []


for prefix in [
    "44b6",
    "6bba",
]:

    recs = [
        record
        for dataset_name, record
        in zip(
            dataset_names_1015,
            metric_records_1015,
        )
        if (
            prefix_map_1015[
                dataset_name
            ]
            == prefix
        )
    ]


    assert len(
        recs
    ) > 0


    s = summarise(
        recs
    )


    prefix_rows_1015b.append({
        "prefix":
            prefix,

        "datasets":
            len(
                recs
            ),

        "edge_jaccard":
            float(
                s[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                s[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                s[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                s[
                    "node_recall"
                ]
            ),

        "score":
            float(
                s[
                    "score"
                ]
            ),
    })


prefix_1015b = (
    pd.DataFrame(
        prefix_rows_1015b
    )
)


# ============================================================
# 12. Final result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.15 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "CANDIDATE REPRODUCTION"
)

print(
    "  single policy:",
    len(
        single_policy_1015
    ),
    "[expected 3945]",
)

print(
    "  strict two policy:",
    len(
        strict_two_policy_1015
    ),
    "[expected 2289]",
)


print()
print(
    "REPAIR INTERACTION"
)

print(
    "  single edges added:",
    single_added_total_1015,
    "[expected 3945]",
)

print(
    "  two bridges requested:",
    two_requested_total_1015,
    "[expected 2289]",
)

print(
    "  two bridges added:",
    two_added_total_1015,
    "[expected 2111]",
)

print(
    "  two associations added:",
    two_edges_added_total_1015,
    "[expected 4222]",
)


print()
print(
    "TWO-EDGE SKIPS"
)

print(
    "  missing endpoint:",
    two_missing_endpoint_total_1015,
)

print(
    "  degree conflict:",
    two_degree_conflict_total_1015,
)

print(
    "  existing edge conflict:",
    two_existing_conflict_total_1015,
)


print()
print(
    "CLEANUP"
)

print(
    "  nodes removed:",
    cleanup_removed_total_1015,
)


print()
print(
    "LOCKED OVERALL METRIC"
)


for key in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "division_jaccard",
    "node_recall",
    "score",
]:

    print(
        f"  {key}: "
        f"{float(summary_1015[key]):.6f}"
    )


print()
print(
    "BY PREFIX"
)

display(
    prefix_1015b
)


print()
print(
    "Expected frozen score:",
    f"{EXPECTED_WARM2_V9_SCORE_1015B:.6f}"
)

print(
    "Observed frozen score:",
    f"{warm2_v9_score_1015:.6f}"
)

print(
    "Score delta:",
    f"{score_delta_1015:+.9f}"
)


print()
print(
    "MODEL-AGNOSTIC BASELINE-v9 REPRODUCTION:",
    "PASS"
)


print()
print(
    "No prediction GEFF was modified."
)

print(
    "No candidate table was regenerated."
)

print(
    "No Stage09 prediction-bound policy table was reused."
)

print(
    "GT was used only for frozen Fold0 metric reproduction."
)


print()
print(
    "10.15 Warm2 reproduction gate: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.16 Paired Stage10 Frozen Baseline-v9 Evaluation
# Purpose:
#   Compare the two Stage-10 256-iteration pilot checkpoints
#   using the EXACT model-agnostic baseline-v9 pipeline whose
#   Warm2 reproduction passed in 10.15.
#
#   Variants:
#
#       A. BASELINE_256
#           det=.995 predictions
#           det=.9975 high-confidence predictions
#
#       B. SPARSE_9975_256
#           det=.995 predictions
#           det=.9975 high-confidence predictions
#
#   For EACH model independently:
#
#       raw predictions
#           -> G2
#           -> K9
#           -> regenerate single-edge candidates
#           -> mutual single-edge policy r<=4 um
#           -> regenerate isolated-middle candidates
#           -> strict two-edge policy r<=4 um
#           -> apply single-edge recovery
#           -> apply strict two-edge recovery
#           -> frozen cleanup
#           -> locked official Fold0 metric
#
#   No Stage09 / Warm2 candidate table is reused.
#
# Changes files: NO — all graph processing is in memory
# Runs training: NO
# Runs inference: NO
# Runs graph processing: YES
# Uses GT: YES — ONLY for final Fold0 metric evaluation
# Keep after running: YES — Stage10 model-selection evidence
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)


# ============================================================
# 1. Preconditions
#
# Reuse the model-agnostic functions already validated by
# 10.15 Warm2 reproduction.
# ============================================================

REQUIRED_1016 = [
    "PROJECT",
    "TRAIN_DIR",

    "dataset_names_1015",
    "prefix_map_1015",

    "census_single_edge_gaps_1015",
    "build_mutual_policy_1015",

    "census_isolated_bridges_1015",
    "build_strict_two_edge_policy_1015",

    "prepare_precleanup_graph_1015",

    "add_available_repairs_0907",
    "add_two_edge_bridges_0912",
    "apply_frozen_component_cleanup",

    "load_graph",
]


missing_1016 = [
    name
    for name in REQUIRED_1016
    if name not in globals()
]


assert not missing_1016, (
    "Required validated 10.15 state is missing:\n"
    + "\n".join(missing_1016)
)


assert len(
    dataset_names_1015
) == 40


# ============================================================
# 2. Stage10 prediction directories
# ============================================================

PRED_ROOT_1016 = (
    PROJECT
    / "predictions/heqiuyan"
)


VARIANTS_1016 = {

    "BASELINE_256": {

        "pred_995":
            PRED_ROOT_1016
            / "stage10_infer_baseline_256_det995"
            / "split_0",

        "pred_9975":
            PRED_ROOT_1016
            / "stage10_infer_baseline_256_det9975"
            / "split_0",
    },


    "SPARSE_9975_256": {

        "pred_995":
            PRED_ROOT_1016
            / "stage10_infer_sparse9975_256_det995"
            / "split_0",

        "pred_9975":
            PRED_ROOT_1016
            / "stage10_infer_sparse9975_256_det9975"
            / "split_0",
    },
}


# ============================================================
# 3. Prediction completeness / split guards
# ============================================================

expected_names_1016 = set(
    dataset_names_1015
)


for variant, cfg in (
    VARIANTS_1016.items()
):

    for threshold_name in [
        "pred_995",
        "pred_9975",
    ]:

        pred_dir = cfg[
            threshold_name
        ]

        assert pred_dir.exists(), (
            f"{variant}: missing prediction directory:\n"
            f"{pred_dir}"
        )


        geffs = sorted(
            pred_dir.glob(
                "*.geff"
            )
        )


        assert len(
            geffs
        ) == 40, (
            f"{variant} {threshold_name}: "
            f"{len(geffs)} GEFFs, expected 40"
        )


        names = {
            p.name.removesuffix(
                ".geff"
            )
            for p in geffs
        }


        assert names == expected_names_1016, (
            f"{variant} {threshold_name}: "
            "Fold0 dataset mismatch."
        )


print(
    "Stage10 prediction completeness:",
    "PASS"
)


# ============================================================
# 4. Restore exact estimated-node metadata
# ============================================================

CV_MANIFEST_1016 = (
    PROJECT
    / "configs/cv_manifest.csv"
)


cv_meta_1016 = pd.read_csv(
    CV_MANIFEST_1016
)


fold0_meta_1016 = (
    cv_meta_1016[
        cv_meta_1016[
            "fold"
        ] == 0
    ]
    .copy()
)


assert len(
    fold0_meta_1016
) == 40


assert set(
    fold0_meta_1016[
        "dataset"
    ]
) == expected_names_1016


estimated_map_1016 = dict(
    zip(
        fold0_meta_1016[
            "dataset"
        ],
        fold0_meta_1016[
            "estimated_nodes"
        ],
    )
)


# ============================================================
# 5. Exact frozen metric wrapper
# ============================================================

def evaluate_graph_1016(
    graph,
    gt_graph,
    scale,
    name,
):

    er = official_evaluate(
        graph,
        gt_graph,
        scale=tuple(
            scale
        ),
        max_distance=7.0,
    )


    recall = node_recall(
        graph,
        gt_graph,
    )


    return per_sample_metrics(
        er,
        float(
            estimated_map_1016[
                name
            ]
        ),
        recall,
    )


# ============================================================
# 6. Model-agnostic candidate builder
# ============================================================

def build_stage10_v9_policies_1016(
    pred_995_dir,
    pred_9975_dir,
    label,
):

    single_parts = []

    isolated_parts = []

    isolated_scope_rows = []


    print()
    print(
        f"Generating candidates: {label}"
    )


    for i, name in enumerate(
        dataset_names_1015,
        start=1,
    ):

        # ----------------------------------------------------
        # Single-edge census
        # ----------------------------------------------------

        single_df = (
            census_single_edge_gaps_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        if len(
            single_df
        ):
            single_parts.append(
                single_df
            )


        # ----------------------------------------------------
        # Existing isolated-middle census
        # ----------------------------------------------------

        isolated_df, scope = (
            census_isolated_bridges_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        if len(
            isolated_df
        ):
            isolated_parts.append(
                isolated_df
            )


        isolated_scope_rows.append(
            scope
        )


        if (
            i == 1
            or i % 10 == 0
            or i == 40
        ):

            print(
                f"  candidates "
                f"{i:02d}/40"
            )


    assert single_parts, (
        f"{label}: no single-edge candidates."
    )

    assert isolated_parts, (
        f"{label}: no isolated bridge candidates."
    )


    gap_candidates = pd.concat(
        single_parts,
        ignore_index=True,
    )


    isolated_candidates = pd.concat(
        isolated_parts,
        ignore_index=True,
    )


    isolated_scope = pd.DataFrame(
        isolated_scope_rows
    )


    # --------------------------------------------------------
    # Frozen V9 single-edge policy
    # --------------------------------------------------------

    single_policy = (
        build_mutual_policy_1015(
            gap_candidates,
            residual_radius_um=4.0,
        )
    )


    # --------------------------------------------------------
    # Frozen V9 strict two-edge policy
    # --------------------------------------------------------

    strict_two_policy = (
        build_strict_two_edge_policy_1015(
            isolated_candidates,
            residual_um=4.0,
        )
    )


    return {
        "gap_candidates":
            gap_candidates,

        "single_policy":
            single_policy,

        "isolated_candidates":
            isolated_candidates,

        "isolated_scope":
            isolated_scope,

        "strict_two_policy":
            strict_two_policy,
    }


# ============================================================
# 7. Exact frozen-V9 evaluator for one model
# ============================================================

def evaluate_stage10_v9_1016(
    pred_995_dir,
    pred_9975_dir,
    policies,
    label,
):

    records = []

    dataset_rows = []


    single_added_total = 0


    two_requested_total = 0

    two_added_total = 0

    two_edges_added_total = 0


    two_missing_total = 0

    two_degree_conflict_total = 0

    two_existing_conflict_total = 0


    cleanup_removed_total = 0


    print()
    print(
        f"Evaluating frozen V9: {label}"
    )


    for i, name in enumerate(
        dataset_names_1015,
        start=1,
    ):

        # ----------------------------------------------------
        # Fresh model-specific G2 + K9 graph
        # ----------------------------------------------------

        g, scale = (
            prepare_precleanup_graph_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        # ----------------------------------------------------
        # Frozen single-edge recovery FIRST
        # ----------------------------------------------------

        single_sub = (
            policies[
                "single_policy"
            ][
                policies[
                    "single_policy"
                ][
                    "dataset"
                ] == name
            ]
            .copy()
        )


        single_stats = (
            add_available_repairs_0907(
                g,
                single_sub,
            )
        )


        # Same structural expectation as V9:
        # generated single policy is directly derived from the
        # same G2+K9 state and should therefore be addable.
        assert (
            int(
                single_stats[
                    "added"
                ]
            )
            ==
            len(
                single_sub
            )
        ), (
            f"{label} / {name}: "
            "single-edge policy unexpectedly not fully addable.\n"
            f"{single_stats}"
        )


        single_added_total += int(
            single_stats[
                "added"
            ]
        )


        # ----------------------------------------------------
        # Frozen strict two-edge recovery SECOND
        # ----------------------------------------------------

        two_sub = (
            policies[
                "strict_two_policy"
            ][
                policies[
                    "strict_two_policy"
                ][
                    "dataset"
                ] == name
            ]
            .copy()
        )


        two_stats = (
            add_two_edge_bridges_0912(
                g,
                two_sub,
            )
        )


        two_requested_total += int(
            two_stats[
                "requested_bridges"
            ]
        )


        two_added_total += int(
            two_stats[
                "added_bridges"
            ]
        )


        two_edges_added_total += int(
            two_stats[
                "edges_added"
            ]
        )


        two_missing_total += int(
            two_stats[
                "missing_endpoint"
            ]
        )


        two_degree_conflict_total += int(
            two_stats[
                "degree_conflict"
            ]
        )


        two_existing_conflict_total += int(
            two_stats[
                "existing_edge_conflict"
            ]
        )


        # ----------------------------------------------------
        # Frozen cleanup
        # ----------------------------------------------------

        cleanup_stats = (
            apply_frozen_component_cleanup(
                g,
                scale,
            )
        )


        cleanup_removed_total += int(
            cleanup_stats[
                "nodes_removed"
            ]
        )


        # ----------------------------------------------------
        # Locked metric
        # ----------------------------------------------------

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )


        m = evaluate_graph_1016(
            g,
            gt_g,
            scale,
            name,
        )


        assert isinstance(
            m,
            dict,
        )


        records.append(
            m
        )


        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map_1015[
                    name
                ],

            "edge_jaccard":
                float(
                    m[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    m[
                        "adj_edge_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    m[
                        "node_recall"
                    ]
                ),

            "total_node_ratio":
                float(
                    m[
                        "total_node_ratio"
                    ]
                ),

            "single_added":
                int(
                    single_stats[
                        "added"
                    ]
                ),

            "two_requested":
                int(
                    two_stats[
                        "requested_bridges"
                    ]
                ),

            "two_added":
                int(
                    two_stats[
                        "added_bridges"
                    ]
                ),

            "cleanup_removed":
                int(
                    cleanup_stats[
                        "nodes_removed"
                    ]
                ),
        })


        if (
            i == 1
            or i % 10 == 0
            or i == 40
        ):

            print(
                f"  evaluated "
                f"{i:02d}/40"
            )


    summary = summarise(
        records
    )


    return {
        "records":
            records,

        "dataset_df":
            pd.DataFrame(
                dataset_rows
            ),

        "summary":
            summary,

        "single_added":
            single_added_total,

        "two_requested":
            two_requested_total,

        "two_added":
            two_added_total,

        "two_edges_added":
            two_edges_added_total,

        "two_missing":
            two_missing_total,

        "two_degree_conflict":
            two_degree_conflict_total,

        "two_existing_conflict":
            two_existing_conflict_total,

        "cleanup_removed":
            cleanup_removed_total,
    }


# ============================================================
# 8. Build + evaluate both variants independently
# ============================================================

stage10_v9_policies_1016 = {}

stage10_v9_results_1016 = {}


for label, cfg in (
    VARIANTS_1016.items()
):

    policies = (
        build_stage10_v9_policies_1016(
            cfg[
                "pred_995"
            ],
            cfg[
                "pred_9975"
            ],
            label,
        )
    )


    stage10_v9_policies_1016[
        label
    ] = policies


    print()
    print(
        f"{label} policy counts:"
    )

    print(
        "  single opportunities:",
        len(
            policies[
                "gap_candidates"
            ]
        ),
    )

    print(
        "  single policy:",
        len(
            policies[
                "single_policy"
            ]
        ),
    )

    print(
        "  isolated nodes:",
        int(
            policies[
                "isolated_scope"
            ][
                "isolated_nodes"
            ].sum()
        ),
    )

    print(
        "  isolated with pair:",
        int(
            policies[
                "isolated_scope"
            ][
                "isolated_with_local_pair"
            ].sum()
        ),
    )

    print(
        "  strict two policy:",
        len(
            policies[
                "strict_two_policy"
            ]
        ),
    )


    results = (
        evaluate_stage10_v9_1016(
            cfg[
                "pred_995"
            ],
            cfg[
                "pred_9975"
            ],
            policies,
            label,
        )
    )


    stage10_v9_results_1016[
        label
    ] = results


# ============================================================
# 9. Overall comparison
# ============================================================

overall_rows_1016 = []


for label in [
    "BASELINE_256",
    "SPARSE_9975_256",
]:

    policies = (
        stage10_v9_policies_1016[
            label
        ]
    )

    result = (
        stage10_v9_results_1016[
            label
        ]
    )

    s = result[
        "summary"
    ]


    overall_rows_1016.append({
        "variant":
            label,

        "single_opportunities":
            len(
                policies[
                    "gap_candidates"
                ]
            ),

        "single_policy":
            len(
                policies[
                    "single_policy"
                ]
            ),

        "isolated_nodes":
            int(
                policies[
                    "isolated_scope"
                ][
                    "isolated_nodes"
                ].sum()
            ),

        "isolated_with_pair":
            int(
                policies[
                    "isolated_scope"
                ][
                    "isolated_with_local_pair"
                ].sum()
            ),

        "strict_two_policy":
            len(
                policies[
                    "strict_two_policy"
                ]
            ),

        "single_added":
            result[
                "single_added"
            ],

        "two_added":
            result[
                "two_added"
            ],

        "associations_added":
            result[
                "two_edges_added"
            ],

        "cleanup_removed":
            result[
                "cleanup_removed"
            ],

        "edge_jaccard":
            float(
                s[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                s[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                s[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                s[
                    "node_recall"
                ]
            ),

        "score":
            float(
                s[
                    "score"
                ]
            ),
    })


overall_1016 = pd.DataFrame(
    overall_rows_1016
)


# ============================================================
# 10. Direct sparse-minus-baseline deltas
# ============================================================

baseline_row_1016 = (
    overall_1016[
        overall_1016[
            "variant"
        ] == "BASELINE_256"
    ]
    .iloc[0]
)


sparse_row_1016 = (
    overall_1016[
        overall_1016[
            "variant"
        ] == "SPARSE_9975_256"
    ]
    .iloc[0]
)


delta_score_1016 = (
    float(
        sparse_row_1016[
            "score"
        ]
    )
    -
    float(
        baseline_row_1016[
            "score"
        ]
    )
)


delta_edge_1016 = (
    float(
        sparse_row_1016[
            "edge_jaccard"
        ]
    )
    -
    float(
        baseline_row_1016[
            "edge_jaccard"
        ]
    )
)


delta_adj_1016 = (
    float(
        sparse_row_1016[
            "adj_edge_jaccard"
        ]
    )
    -
    float(
        baseline_row_1016[
            "adj_edge_jaccard"
        ]
    )
)


delta_div_1016 = (
    float(
        sparse_row_1016[
            "division_jaccard"
        ]
    )
    -
    float(
        baseline_row_1016[
            "division_jaccard"
        ]
    )
)


delta_recall_1016 = (
    float(
        sparse_row_1016[
            "node_recall"
        ]
    )
    -
    float(
        baseline_row_1016[
            "node_recall"
        ]
    )
)


# ============================================================
# 11. Paired dataset-level robustness
# ============================================================

base_df_1016 = (
    stage10_v9_results_1016[
        "BASELINE_256"
    ][
        "dataset_df"
    ]
    .copy()
)


sparse_df_1016 = (
    stage10_v9_results_1016[
        "SPARSE_9975_256"
    ][
        "dataset_df"
    ]
    .copy()
)


paired_1016 = (
    base_df_1016[
        [
            "dataset",
            "prefix",
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "total_node_ratio",
        ]
    ]
    .merge(
        sparse_df_1016[
            [
                "dataset",
                "edge_jaccard",
                "adj_edge_jaccard",
                "node_recall",
                "total_node_ratio",
            ]
        ],
        on="dataset",
        suffixes=(
            "_baseline",
            "_sparse",
        ),
        validate="one_to_one",
    )
)


paired_1016[
    "delta_edge_jaccard"
] = (
    paired_1016[
        "edge_jaccard_sparse"
    ]
    -
    paired_1016[
        "edge_jaccard_baseline"
    ]
)


paired_1016[
    "delta_adj_edge_jaccard"
] = (
    paired_1016[
        "adj_edge_jaccard_sparse"
    ]
    -
    paired_1016[
        "adj_edge_jaccard_baseline"
    ]
)


paired_1016[
    "delta_node_recall"
] = (
    paired_1016[
        "node_recall_sparse"
    ]
    -
    paired_1016[
        "node_recall_baseline"
    ]
)


paired_1016[
    "delta_node_ratio"
] = (
    paired_1016[
        "total_node_ratio_sparse"
    ]
    -
    paired_1016[
        "total_node_ratio_baseline"
    ]
)


# ============================================================
# 12. Paired robustness summary
# ============================================================

paired_summary_rows_1016 = []


for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired_1016
        if prefix == "ALL"
        else paired_1016[
            paired_1016[
                "prefix"
            ] == prefix
        ]
    )


    paired_summary_rows_1016.append({
        "prefix":
            prefix,

        "datasets":
            len(
                sub
            ),

        "edgeJ_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "recall_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_node_recall"
                    ] > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_node_recall"
                ].mean()
            ),

        "median_delta_recall":
            float(
                sub[
                    "delta_node_recall"
                ].median()
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_node_ratio"
                ].mean()
            ),
    })


paired_summary_1016 = pd.DataFrame(
    paired_summary_rows_1016
)


# ============================================================
# 13. Prefix aggregate metric
# ============================================================

prefix_rows_1016 = []


for label in [
    "BASELINE_256",
    "SPARSE_9975_256",
]:

    records = (
        stage10_v9_results_1016[
            label
        ][
            "records"
        ]
    )


    for prefix in [
        "44b6",
        "6bba",
    ]:

        prefix_records = [
            record
            for name, record
            in zip(
                dataset_names_1015,
                records,
            )
            if (
                prefix_map_1015[
                    name
                ]
                == prefix
            )
        ]


        s = summarise(
            prefix_records
        )


        prefix_rows_1016.append({
            "variant":
                label,

            "prefix":
                prefix,

            "datasets":
                len(
                    prefix_records
                ),

            "edge_jaccard":
                float(
                    s[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    s[
                        "adj_edge_jaccard"
                    ]
                ),

            "division_jaccard":
                float(
                    s[
                        "division_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    s[
                        "node_recall"
                    ]
                ),

            "score":
                float(
                    s[
                        "score"
                    ]
                ),
        })


prefix_1016 = pd.DataFrame(
    prefix_rows_1016
)


# ============================================================
# 14. Warm2 reference
# ============================================================

WARM2_V9_SCORE_1016 = (
    0.770921
)


overall_1016[
    "delta_score_vs_warm2_v9"
] = (
    overall_1016[
        "score"
    ]
    -
    WARM2_V9_SCORE_1016
)


# ============================================================
# 15. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.16 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "OVERALL FROZEN-V9 COMPARISON"
)

display(
    overall_1016
)


print()
print(
    "BY PREFIX"
)

display(
    prefix_1016
)


print()
print(
    "PAIRED DATASET ROBUSTNESS"
)

display(
    paired_summary_1016
)


print()
print(
    "SPARSE - BASELINE"
)

print(
    "  delta score:",
    f"{delta_score_1016:+.6f}"
)

print(
    "  delta Edge J:",
    f"{delta_edge_1016:+.6f}"
)

print(
    "  delta Adj Edge J:",
    f"{delta_adj_1016:+.6f}"
)

print(
    "  delta Division J:",
    f"{delta_div_1016:+.6f}"
)

print(
    "  delta node recall:",
    f"{delta_recall_1016:+.6f}"
)


print()
print(
    "WARM2 frozen-v9 reference:",
    f"{WARM2_V9_SCORE_1016:.6f}"
)


print()
print(
    "IMPORTANT:"
)

print(
    "  Candidate tables were regenerated independently "
    "for each Stage10 model."
)

print(
    "  Frozen V9 geometry and ordering were unchanged."
)

print(
    "  No threshold sweep was performed."
)

print(
    "  No prediction GEFF was modified."
)

print(
    "  GT was used only for final Fold0 evaluation."
)


print()
print(
    "10.16 paired Stage10 frozen-v9 evaluation: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.17 Sparse .9995 256-Iteration Fold0 Pilot
# Purpose:
#   Test ONE weaker sparse-label intervention:
#
#       det_ignore_prob = 0.9995
#
#   against the already-frozen BASELINE_256 control.
#
#   Exact training conditions retained from 10.07:
#       init checkpoint   = Warm2
#       Fold              = 0
#       epochs            = 1
#       max iterations    = 256
#       batch size        = 8
#       learning rate     = 1e-5
#       seed              = 1640
#       det loss weight   = 1.0
#       det neg weight    = 0.01
#       downsample        = 1,4,4
#       window size       = 2
#       training pool     = 5.0 um
#
#   ONLY experimental change relative to BASELINE_256:
#       det_ignore_prob: None -> 0.9995
#
#   This is NOT a full epoch experiment.
#   This is NOT a threshold sweep.
#
# Changes files: YES
#   - Stage10 .9995 pilot checkpoint/config
#   - reports/stage10_pilot/sparse_9995_256.log
#
# Runs training: YES — 256 optimizer iterations
# Runs inference: NO
# Uses GT: YES — normal sparse training/evaluation supervision
# Keep after running: YES until Stage10 decision is recorded
# ============================================================

from pathlib import Path
import hashlib
import os
import re
import subprocess
import sys
import time

import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

METHOD_1017 = (
    "stage10_pilot_sparse9995_256"
)

OUTPUT_DIR_1017 = (
    PROJECT
    / "models/official_baseline"
    / METHOD_1017
    / "split_0"
)

CHECKPOINT_1017 = (
    OUTPUT_DIR_1017
    / "edge_predictor_best.pth"
)

LOG_DIR_1017 = (
    PROJECT
    / "reports/stage10_pilot"
)

LOG_DIR_1017.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_PATH_1017 = (
    LOG_DIR_1017
    / "sparse_9995_256.log"
)

LOCAL_SCRIPTS = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)


# ============================================================
# 2. Frozen hashes
# ============================================================

EXPECTED_TRAIN_SCRIPT_SHA_1017 = (
    "ce4fe8b3a16771039e1c687c02e71cad2747ee48d94417f0a26c929192f1be7b"
)

EXPECTED_WARM2_SHA_1017 = (
    "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"
)


def sha256_1017(
    path,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# 3. Preconditions
# ============================================================

assert PROJECT.exists()

assert PYTHON.exists()

assert TRAIN_SCRIPT.exists()

assert TRAIN_ROOT.exists()

assert SPLITS.exists()

assert WARM2.exists()


# Correct project Python only.
# Never use stale CONDA_PREFIX.
assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON.resolve()
), (
    "Notebook kernel is not using the Biohub project .venv."
)


train_script_sha_1017 = (
    sha256_1017(
        TRAIN_SCRIPT
    )
)

warm2_sha_1017 = (
    sha256_1017(
        WARM2
    )
)


assert (
    train_script_sha_1017
    ==
    EXPECTED_TRAIN_SCRIPT_SHA_1017
), (
    "Training script SHA changed.\n"
    f"Observed: {train_script_sha_1017}"
)


assert (
    warm2_sha_1017
    ==
    EXPECTED_WARM2_SHA_1017
), (
    "Warm2 checkpoint SHA changed.\n"
    f"Observed: {warm2_sha_1017}"
)


print(
    "Training script SHA:",
    "PASS"
)

print(
    "Warm2 checkpoint SHA:",
    "PASS"
)

print(
    "Python:",
    PYTHON
)


# ============================================================
# 4. No silent overwrite
# ============================================================

assert not LOG_PATH_1017.exists(), (
    "Pilot log already exists:\n"
    f"{LOG_PATH_1017}\n"
    "Do not silently overwrite an expensive run."
)

assert not CHECKPOINT_1017.exists(), (
    "Pilot checkpoint already exists:\n"
    f"{CHECKPOINT_1017}\n"
    "Do not silently overwrite it."
)


# ============================================================
# 5. Environment
#
# Explicitly expose BOTH local and official script modules.
# ============================================================

env_1017 = os.environ.copy()

pythonpath_parts_1017 = [
    str(
        LOCAL_SCRIPTS
    ),
    str(
        OFFICIAL_SCRIPTS
    ),
]


if env_1017.get(
    "PYTHONPATH"
):

    pythonpath_parts_1017.append(
        env_1017[
            "PYTHONPATH"
        ]
    )


env_1017[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts_1017
)


# ============================================================
# 6. CLI contract audit
# ============================================================

help_proc_1017 = subprocess.run(
    [
        str(
            PYTHON
        ),
        str(
            TRAIN_SCRIPT
        ),
        "--help",
    ],
    cwd=PROJECT,
    env=env_1017,
    capture_output=True,
    text=True,
)


assert (
    help_proc_1017.returncode
    == 0
), (
    help_proc_1017.stderr
)


help_text_1017 = (
    help_proc_1017.stdout
    +
    help_proc_1017.stderr
)


required_cli_1017 = [
    "--data-dir",
    "--splits",
    "--split",
    "--method",
    "--epochs",
    "--lr",
    "--batch-size",
    "--num-workers",
    "--init-weights",
    "--downsample",
    "--det-loss-weight",
    "--det-neg-weight",
    "--det-ignore-prob",
    "--max-iters",
    "--seed",
    "--window-size",
    "--pool-kernel-um",
]


missing_cli_1017 = [
    flag
    for flag in required_cli_1017
    if flag not in help_text_1017
]


assert not missing_cli_1017, (
    "Missing expected CLI flags:\n"
    + "\n".join(
        missing_cli_1017
    )
)


print(
    "Training CLI contract:",
    "PASS"
)


# ============================================================
# 7. Frozen pilot configuration
# ============================================================

DET_IGNORE_PROB_1017 = (
    0.9995
)

MAX_ITERS_1017 = (
    256
)

LR_1017 = (
    1e-5
)

BATCH_SIZE_1017 = (
    8
)

NUM_WORKERS_1017 = (
    8
)

SEED_1017 = (
    1640
)

DET_LOSS_WEIGHT_1017 = (
    1.0
)

DET_NEG_WEIGHT_1017 = (
    0.01
)

POOL_KERNEL_UM_1017 = (
    5.0
)


# ============================================================
# 8. Exact training command
# ============================================================

cmd_1017 = [
    str(
        PYTHON
    ),

    str(
        TRAIN_SCRIPT
    ),

    "--data-dir",
    str(
        TRAIN_ROOT
    ),

    "--splits",
    str(
        SPLITS
    ),

    "--split",
    "0",

    "--method",
    METHOD_1017,

    "--epochs",
    "1",

    "--lr",
    str(
        LR_1017
    ),

    "--batch-size",
    str(
        BATCH_SIZE_1017
    ),

    "--num-workers",
    str(
        NUM_WORKERS_1017
    ),

    "--init-weights",
    str(
        WARM2
    ),

    "--downsample",
    "1,4,4",

    "--det-loss-weight",
    str(
        DET_LOSS_WEIGHT_1017
    ),

    "--det-neg-weight",
    str(
        DET_NEG_WEIGHT_1017
    ),

    "--det-ignore-prob",
    str(
        DET_IGNORE_PROB_1017
    ),

    "--max-iters",
    str(
        MAX_ITERS_1017
    ),

    "--seed",
    str(
        SEED_1017
    ),

    "--window-size",
    "2",

    "--pool-kernel-um",
    str(
        POOL_KERNEL_UM_1017
    ),
]


# ============================================================
# 9. Concise pre-run summary
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.17 SPARSE .9995 PILOT START"
)

print(
    "=" * 72
)

print(
    "Warm start:",
    WARM2
)

print(
    "Fold:",
    0
)

print(
    "Iterations:",
    MAX_ITERS_1017
)

print(
    "Batch size:",
    BATCH_SIZE_1017
)

print(
    "LR:",
    LR_1017
)

print(
    "Seed:",
    SEED_1017
)

print(
    "det_loss_weight:",
    DET_LOSS_WEIGHT_1017
)

print(
    "det_neg_weight:",
    DET_NEG_WEIGHT_1017
)

print(
    "det_ignore_prob:",
    DET_IGNORE_PROB_1017
)

print(
    "Log:",
    LOG_PATH_1017
)

print()
print(
    "Training output is redirected to the log file."
)


# ============================================================
# 10. Train
# ============================================================

start_1017 = time.time()


with LOG_PATH_1017.open(
    "w",
    encoding="utf-8",
) as log_file:

    proc_1017 = subprocess.run(
        cmd_1017,
        cwd=PROJECT,
        env=env_1017,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )


elapsed_1017 = (
    time.time()
    - start_1017
)


# ============================================================
# 11. Failure handling
# ============================================================

log_text_1017 = (
    LOG_PATH_1017.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


if (
    proc_1017.returncode
    != 0
):

    print()
    print(
        "LAST 100 LOG LINES"
    )

    print(
        "\n".join(
            log_text_1017
            .splitlines()[
                -100:
            ]
        )
    )

    raise RuntimeError(
        "10.17 .9995 pilot training failed."
    )


# ============================================================
# 12. Provenance checks
# ============================================================

assert (
    "Fold 0: 159 train, 40 test"
    in log_text_1017
), (
    "Fold0 split provenance not found in training log."
)


assert (
    str(
        WARM2
    )
    in log_text_1017
), (
    "Warm2 initialization path not found in training log."
)


assert (
    "Full-model init weights loaded"
    in log_text_1017
), (
    "Full-model warm start was not confirmed."
)


# Flexible float-text checks.
assert re.search(
    r"Detection loss:\s*"
    r"weight=1(?:\.0+)?,\s*"
    r"neg_weight=0\.01,\s*"
    r"ignore_prob=0\.9995",
    log_text_1017,
), (
    "Expected sparse .9995 detection-loss configuration "
    "was not found in the log."
)


assert (
    "Starting training for 1 epochs (batch_size=8)"
    in log_text_1017
), (
    "Unexpected epoch/batch-size configuration."
)


# max_iters=256 produces the 256-iteration progress loop.
assert (
    "/256"
    in log_text_1017
), (
    "Could not confirm 256-iteration training loop."
)


# ============================================================
# 13. Checkpoint verification
# ============================================================

assert CHECKPOINT_1017.exists(), (
    "Expected pilot checkpoint was not created:\n"
    f"{CHECKPOINT_1017}"
)


checkpoint_sha_1017 = (
    sha256_1017(
        CHECKPOINT_1017
    )
)


# ============================================================
# 14. Extract final training diagnostic line
# ============================================================

epoch_pattern_1017 = re.compile(
    r"Epoch\s+\d+/\d+\s+\|\s+"
    r"edge=([0-9.eE+-]+)\s+\|\s+"
    r"det=([0-9.eE+-]+)\s+\|\s+"
    r"test_loss=([0-9.eE+-]+)\s+\|\s+"
    r"acc=([0-9.eE+-]+)\s+\|\s+"
    r"recall=([0-9.eE+-]+)"
)


epoch_matches_1017 = list(
    epoch_pattern_1017.finditer(
        log_text_1017
    )
)


assert epoch_matches_1017, (
    "Could not parse final epoch diagnostics."
)


last_epoch_1017 = (
    epoch_matches_1017[
        -1
    ]
)


edge_loss_1017 = float(
    last_epoch_1017.group(
        1
    )
)

det_loss_1017 = float(
    last_epoch_1017.group(
        2
    )
)

test_loss_1017 = float(
    last_epoch_1017.group(
        3
    )
)

test_acc_1017 = float(
    last_epoch_1017.group(
        4
    )
)

test_recall_1017 = float(
    last_epoch_1017.group(
        5
    )
)

acc_recall_1017 = (
    test_acc_1017
    *
    test_recall_1017
)


# ============================================================
# 15. Compare ONLY training diagnostics with existing controls
#
# These are NOT competition metrics.
# ============================================================

BASELINE_256_DIAGNOSTIC_1017 = {
    "edge_loss":
        0.0005,

    "det_loss":
        0.0106,

    "test_loss":
        0.0004,

    "test_acc":
        0.9997,

    "test_recall":
        0.9170,
}


SPARSE_9975_DIAGNOSTIC_1017 = {
    "edge_loss":
        0.0005,

    "det_loss":
        0.0087,

    "test_loss":
        0.0004,

    "test_acc":
        0.9997,

    "test_recall":
        0.9193,
}


diagnostic_1017 = pd.DataFrame([
    {
        "variant":
            "BASELINE_256",

        **BASELINE_256_DIAGNOSTIC_1017,
    },

    {
        "variant":
            "SPARSE_9975_256",

        **SPARSE_9975_DIAGNOSTIC_1017,
    },

    {
        "variant":
            "SPARSE_9995_256",

        "edge_loss":
            edge_loss_1017,

        "det_loss":
            det_loss_1017,

        "test_loss":
            test_loss_1017,

        "test_acc":
            test_acc_1017,

        "test_recall":
            test_recall_1017,
    },
])


diagnostic_1017[
    "acc_x_recall"
] = (
    diagnostic_1017[
        "test_acc"
    ]
    *
    diagnostic_1017[
        "test_recall"
    ]
)


# ============================================================
# 16. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.17 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "TRAINING"
)

print(
    "  return code:",
    proc_1017.returncode
)

print(
    "  elapsed:",
    f"{elapsed_1017 / 60:.1f} min"
)

print(
    "  det_ignore_prob:",
    DET_IGNORE_PROB_1017
)

print(
    "  max_iters:",
    MAX_ITERS_1017
)


print()
print(
    "CHECKPOINT"
)

print(
    "  path:",
    CHECKPOINT_1017
)

print(
    "  SHA256:",
    checkpoint_sha_1017
)


print()
print(
    "TRAINING DIAGNOSTICS — NOT COMPETITION METRICS"
)

display(
    diagnostic_1017
)


print()
print(
    "SPARSE_9995 - BASELINE diagnostic deltas"
)

print(
    "  det loss:",
    f"{det_loss_1017 - 0.0106:+.6f}"
)

print(
    "  test recall:",
    f"{test_recall_1017 - 0.9170:+.6f}"
)

print(
    "  acc*recall:",
    f"{acc_recall_1017 - (0.9997 * 0.9170):+.6f}"
)


print()
print(
    "No inference was run."
)

print(
    "No competition metric was evaluated."
)

print(
    "No model-selection decision is made from this Cell."
)

print()
print(
    "10.17 sparse .9995 256-iteration pilot: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.18 Sparse .9995 Paired Fold0 Inference
# Purpose:
#   Generate BOTH prediction sets required by the frozen
#   baseline-v9 evaluator for SPARSE_9995_256:
#
#       A. det = .995
#          main prediction graph
#
#       B. det = .9975
#          high-confidence graph used ONLY by K9 pruning
#
#   Everything else is frozen to the validated Stage10
#   inference configuration:
#
#       predictor SHA     = frozen local MPS predictor
#       Fold              = 0
#       TTA               = ON
#       pool kernel       = 3.0 um
#       window size       = checkpoint config
#       edge policy       = unchanged
#       ILP               = OFF
#
# Changes files: YES
#   - 40 GEFFs at det=.995
#   - 40 GEFFs at det=.9975
#   - two inference logs
#
# Runs training: NO
# Runs inference: YES — Fold0 x 2 thresholds
# Uses GT: NO
# Keep after running: YES — required for 10.19 V9 evaluation
# ============================================================

from pathlib import Path
import hashlib
import os
import re
import subprocess
import sys
import time

import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

PREDICT_SCRIPT = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

LOCAL_SCRIPTS = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

CHECKPOINT_1018 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_pilot_sparse9995_256/"
      "split_0/edge_predictor_best.pth"
)

LOG_DIR_1018 = (
    PROJECT
    / "reports/stage10_inference"
)

LOG_DIR_1018.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Frozen identities
# ============================================================

EXPECTED_PREDICT_SHA_1018 = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)

EXPECTED_CHECKPOINT_SHA_1018 = (
    "ab7a75268f43e2807167913f3267a10055b1f21941af916b34962f544ccb5699"
)


def sha256_1018(
    path,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# 3. Preconditions
# ============================================================

assert PROJECT.exists()
assert PYTHON.exists()
assert PREDICT_SCRIPT.exists()
assert TRAIN_ROOT.exists()
assert SPLITS.exists()
assert CHECKPOINT_1018.exists()


assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON.resolve()
), (
    "Notebook kernel is not using the Biohub project .venv."
)


predict_sha_1018 = (
    sha256_1018(
        PREDICT_SCRIPT
    )
)

checkpoint_sha_1018 = (
    sha256_1018(
        CHECKPOINT_1018
    )
)


assert (
    predict_sha_1018
    ==
    EXPECTED_PREDICT_SHA_1018
), (
    "Prediction script SHA changed.\n"
    f"Observed: {predict_sha_1018}"
)


assert (
    checkpoint_sha_1018
    ==
    EXPECTED_CHECKPOINT_SHA_1018
), (
    "Sparse .9995 checkpoint SHA changed.\n"
    f"Observed: {checkpoint_sha_1018}"
)


print(
    "Predictor SHA:",
    "PASS"
)

print(
    "Sparse .9995 checkpoint SHA:",
    "PASS"
)

print(
    "Python:",
    PYTHON
)


# ============================================================
# 4. PYTHONPATH
# ============================================================

env_1018 = os.environ.copy()

pythonpath_parts_1018 = [
    str(
        LOCAL_SCRIPTS
    ),
    str(
        OFFICIAL_SCRIPTS
    ),
]


if env_1018.get(
    "PYTHONPATH"
):

    pythonpath_parts_1018.append(
        env_1018[
            "PYTHONPATH"
        ]
    )


env_1018[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts_1018
)


# ============================================================
# 5. Frozen inference variants
# ============================================================

UNET_BATCH_SIZE_1018 = (
    4
)


VARIANTS_1018 = {

    "SPARSE_9995_256_DET995": {
        "det_threshold":
            0.995,

        "method":
            "stage10_infer_sparse9995_256_det995",

        "log":
            LOG_DIR_1018
            / "sparse_9995_256_det995.log",
    },


    "SPARSE_9995_256_DET9975": {
        "det_threshold":
            0.9975,

        "method":
            "stage10_infer_sparse9995_256_det9975",

        "log":
            LOG_DIR_1018
            / "sparse_9995_256_det9975.log",
    },
}


# ============================================================
# 6. No silent overwrite
# ============================================================

for label, cfg in (
    VARIANTS_1018.items()
):

    assert not cfg[
        "log"
    ].exists(), (
        f"{label}: existing log found:\n"
        f"{cfg['log']}\n"
        "Do not silently overwrite an expensive inference."
    )


# ============================================================
# 7. Log parsers
# ============================================================

start_pattern_1018 = re.compile(
    r"Fold\s+0:\s+40 datasets\s+\|\s+"
    r"weights=(.+?)\s+\|\s+"
    r"device=(\w+)\s+\|\s+"
    r"window_size=(\d+)\s+\|\s+"
    r"pool_kernel_um=([0-9.]+)"
)


save_pattern_1018 = re.compile(
    r"Saved\s+(\d+)\s+predictions\s+to\s+(.+)"
)


# ============================================================
# 8. Run both thresholds
# ============================================================

rows_1018 = []


print()
print(
    "=" * 72
)

print(
    "10.18 SPARSE .9995 PAIRED INFERENCE START"
)

print(
    "=" * 72
)

print(
    "Checkpoint:",
    CHECKPOINT_1018
)

print(
    "Pool kernel:",
    "3.0 um"
)

print(
    "TTA:",
    "default ON"
)

print(
    "ILP:",
    False
)


for label, cfg in (
    VARIANTS_1018.items()
):

    det_threshold = float(
        cfg[
            "det_threshold"
        ]
    )

    method = cfg[
        "method"
    ]

    log_path = cfg[
        "log"
    ]


    cmd = [
        str(
            PYTHON
        ),

        str(
            PREDICT_SCRIPT
        ),

        "--data-dir",
        str(
            TRAIN_ROOT
        ),

        "--splits",
        str(
            SPLITS
        ),

        "--split",
        "0",

        "--method",
        method,

        "--weights",
        str(
            CHECKPOINT_1018
        ),

        "--det-threshold",
        str(
            det_threshold
        ),

        "--unet-batch-size",
        str(
            UNET_BATCH_SIZE_1018
        ),
    ]


    print()
    print(
        f"Starting {label}"
    )

    print(
        "  det threshold:",
        det_threshold
    )

    print(
        "  log:",
        log_path
    )


    start_time = (
        time.time()
    )


    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_file:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env_1018,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )


    elapsed = (
        time.time()
        - start_time
    )


    log_text = (
        log_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )


    # --------------------------------------------------------
    # Failure handling
    # --------------------------------------------------------

    if (
        proc.returncode
        != 0
    ):

        print()
        print(
            "LAST 100 LOG LINES"
        )

        print(
            "\n".join(
                log_text
                .splitlines()[
                    -100:
                ]
            )
        )

        raise RuntimeError(
            f"{label} inference failed."
        )


    # --------------------------------------------------------
    # Verify frozen inference configuration
    # --------------------------------------------------------

    start_matches = list(
        start_pattern_1018.finditer(
            log_text
        )
    )


    assert start_matches, (
        f"{label}: could not parse inference configuration."
    )


    sm = (
        start_matches[
            -1
        ]
    )


    logged_weights = Path(
        sm.group(
            1
        ).strip()
    ).resolve()


    device = (
        sm.group(
            2
        )
    )


    window_size = int(
        sm.group(
            3
        )
    )


    pool_kernel_um = float(
        sm.group(
            4
        )
    )


    assert (
        logged_weights
        ==
        CHECKPOINT_1018.resolve()
    )


    assert (
        device
        == "mps"
    )


    assert (
        window_size
        == 2
    )


    assert (
        pool_kernel_um
        == 3.0
    )


    # --------------------------------------------------------
    # Verify output
    # --------------------------------------------------------

    save_matches = list(
        save_pattern_1018.finditer(
            log_text
        )
    )


    assert save_matches, (
        f"{label}: no final save message found."
    )


    save_match = (
        save_matches[
            -1
        ]
    )


    reported_count = int(
        save_match.group(
            1
        )
    )


    output_dir = Path(
        save_match.group(
            2
        ).strip()
    )


    if not output_dir.is_absolute():

        output_dir = (
            PROJECT
            / output_dir
        )


    output_dir = (
        output_dir.resolve()
    )


    geffs = sorted(
        output_dir.glob(
            "*.geff"
        )
    )


    assert (
        reported_count
        == 40
    )


    assert (
        len(
            geffs
        )
        == 40
    )


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    rows_1018.append({
        "variant":
            label,

        "return_code":
            proc.returncode,

        "device":
            device,

        "window_size":
            window_size,

        "pool_kernel_um":
            pool_kernel_um,

        "det_threshold":
            det_threshold,

        "actual_geffs":
            len(
                geffs
            ),

        "elapsed_sec":
            elapsed,

        "output_dir":
            str(
                output_dir
            ),

        "log":
            str(
                log_path
            ),
    })


    print(
        f"{label}: "
        f"saved={len(geffs)}/40, "
        f"elapsed={elapsed / 60:.1f} min"
    )


# ============================================================
# 9. Final verification
# ============================================================

inference_1018 = pd.DataFrame(
    rows_1018
)


assert (
    len(
        inference_1018
    )
    == 2
)


assert (
    inference_1018[
        "actual_geffs"
    ]
    == 40
).all()


assert (
    inference_1018[
        "pool_kernel_um"
    ]
    == 3.0
).all()


assert set(
    inference_1018[
        "det_threshold"
    ].tolist()
) == {
    0.995,
    0.9975,
}


assert (
    inference_1018[
        "output_dir"
    ].nunique()
    == 2
)


# ============================================================
# 10. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.18 RESULT"
)

print(
    "=" * 72
)


display(
    inference_1018[
        [
            "variant",
            "return_code",
            "device",
            "window_size",
            "pool_kernel_um",
            "det_threshold",
            "actual_geffs",
            "elapsed_sec",
            "output_dir",
        ]
    ]
)


print()
print(
    "Sparse .9995 det=.995 predictions:",
    "COMPLETE"
)

print(
    "Sparse .9995 det=.9975 predictions:",
    "COMPLETE"
)

print()
print(
    "No GT was used."
)

print(
    "No metric was evaluated."
)

print(
    "No model-selection decision was made."
)

print()
print(
    "10.18 sparse .9995 paired inference: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.19 Sparse .9995 Frozen-v9 Metric Evaluation
# Purpose:
#   Evaluate SPARSE_9995_256 with the EXACT model-agnostic
#   frozen baseline-v9 pipeline validated in 10.15.
#
#   Primary comparison:
#
#       SPARSE_9995_256
#       vs
#       BASELINE_256
#
#   Secondary reference:
#
#       SPARSE_9975_256
#
#   For SPARSE_9995_256 independently:
#
#       det=.995 graph
#       -> G2
#       -> K9 using det=.9975 graph
#       -> regenerate single-edge candidates
#       -> mutual single-edge policy r<=4 um
#       -> regenerate isolated-middle candidates
#       -> strict two-edge policy r<=4 um
#       -> single-edge recovery
#       -> strict two-edge recovery
#       -> frozen cleanup
#       -> locked official Fold0 metric
#
#   No old candidate table is reused.
#   No threshold sweep is performed.
#
# Changes files: NO — in-memory graph processing only
# Runs training: NO
# Runs inference: NO
# Runs graph processing: YES
# Uses GT: YES — final Fold0 evaluation only
# Keep after running: YES — formal Stage10 model-selection evidence
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Preconditions
#
# Reuse the model-agnostic builder/evaluator already validated
# by the exact Warm2 0.770921 reproduction gate.
# ============================================================

REQUIRED_1019 = [
    "PROJECT",

    "dataset_names_1015",
    "prefix_map_1015",

    "build_stage10_v9_policies_1016",
    "evaluate_stage10_v9_1016",

    "stage10_v9_results_1016",
    "stage10_v9_policies_1016",

    "summarise",
]


missing_1019 = [
    name
    for name in REQUIRED_1019
    if name not in globals()
]


assert not missing_1019, (
    "Required 10.15/10.16 validated state is missing:\n"
    + "\n".join(
        missing_1019
    )
)


assert (
    "BASELINE_256"
    in stage10_v9_results_1016
)


assert (
    "SPARSE_9975_256"
    in stage10_v9_results_1016
)


assert len(
    dataset_names_1015
) == 40


print(
    "Validated V9 evaluator state:",
    "PASS"
)


# ============================================================
# 2. Sparse .9995 prediction directories
# ============================================================

PRED_ROOT_1019 = (
    PROJECT
    / "predictions/heqiuyan"
)


SPARSE_9995_995_1019 = (
    PRED_ROOT_1019
    / "stage10_infer_sparse9995_256_det995"
    / "split_0"
)


SPARSE_9995_9975_1019 = (
    PRED_ROOT_1019
    / "stage10_infer_sparse9995_256_det9975"
    / "split_0"
)


for pred_dir in [
    SPARSE_9995_995_1019,
    SPARSE_9995_9975_1019,
]:

    assert pred_dir.exists(), (
        f"Missing prediction directory:\n"
        f"{pred_dir}"
    )

    geffs = sorted(
        pred_dir.glob(
            "*.geff"
        )
    )

    assert len(
        geffs
    ) == 40, (
        f"{pred_dir}: "
        f"{len(geffs)} GEFFs, expected 40."
    )

    names = {
        p.name.removesuffix(
            ".geff"
        )
        for p in geffs
    }

    assert names == set(
        dataset_names_1015
    ), (
        f"Fold0 dataset mismatch:\n"
        f"{pred_dir}"
    )


print(
    "Sparse .9995 prediction completeness:",
    "PASS"
)


# ============================================================
# 3. Regenerate .9995 model-specific V9 policies
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.19A — SPARSE .9995 V9 CANDIDATE REGENERATION"
)

print(
    "=" * 72
)


sparse9995_policies_1019 = (
    build_stage10_v9_policies_1016(
        SPARSE_9995_995_1019,
        SPARSE_9995_9975_1019,
        "SPARSE_9995_256",
    )
)


single_opportunities_1019 = len(
    sparse9995_policies_1019[
        "gap_candidates"
    ]
)


single_policy_count_1019 = len(
    sparse9995_policies_1019[
        "single_policy"
    ]
)


isolated_nodes_1019 = int(
    sparse9995_policies_1019[
        "isolated_scope"
    ][
        "isolated_nodes"
    ].sum()
)


isolated_with_pair_1019 = int(
    sparse9995_policies_1019[
        "isolated_scope"
    ][
        "isolated_with_local_pair"
    ].sum()
)


strict_two_policy_count_1019 = len(
    sparse9995_policies_1019[
        "strict_two_policy"
    ]
)


print()
print(
    "SPARSE_9995_256 policy counts:"
)

print(
    "  single opportunities:",
    single_opportunities_1019
)

print(
    "  single policy:",
    single_policy_count_1019
)

print(
    "  isolated nodes:",
    isolated_nodes_1019
)

print(
    "  isolated with pair:",
    isolated_with_pair_1019
)

print(
    "  strict two policy:",
    strict_two_policy_count_1019
)


# ============================================================
# 4. Evaluate exact frozen V9
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.19B — SPARSE .9995 FROZEN-v9 EVALUATION"
)

print(
    "=" * 72
)


sparse9995_result_1019 = (
    evaluate_stage10_v9_1016(
        SPARSE_9995_995_1019,
        SPARSE_9995_9975_1019,
        sparse9995_policies_1019,
        "SPARSE_9995_256",
    )
)


assert len(
    sparse9995_result_1019[
        "records"
    ]
) == 40


assert len(
    sparse9995_result_1019[
        "dataset_df"
    ]
) == 40


sparse9995_summary_1019 = (
    sparse9995_result_1019[
        "summary"
    ]
)


# ============================================================
# 5. Existing Stage10 control results
# ============================================================

baseline_result_1019 = (
    stage10_v9_results_1016[
        "BASELINE_256"
    ]
)


sparse9975_result_1019 = (
    stage10_v9_results_1016[
        "SPARSE_9975_256"
    ]
)


baseline_summary_1019 = (
    baseline_result_1019[
        "summary"
    ]
)


sparse9975_summary_1019 = (
    sparse9975_result_1019[
        "summary"
    ]
)


# ============================================================
# 6. Overall three-way comparison
# ============================================================

def overall_row_1019(
    label,
    result,
    policies=None,
):
    s = result[
        "summary"
    ]

    row = {
        "variant":
            label,

        "edge_jaccard":
            float(
                s[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                s[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                s[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                s[
                    "node_recall"
                ]
            ),

        "score":
            float(
                s[
                    "score"
                ]
            ),

        "single_added":
            int(
                result[
                    "single_added"
                ]
            ),

        "two_added":
            int(
                result[
                    "two_added"
                ]
            ),

        "associations_added":
            int(
                result[
                    "two_edges_added"
                ]
            ),

        "cleanup_removed":
            int(
                result[
                    "cleanup_removed"
                ]
            ),
    }


    if policies is not None:

        row.update({
            "single_opportunities":
                len(
                    policies[
                        "gap_candidates"
                    ]
                ),

            "single_policy":
                len(
                    policies[
                        "single_policy"
                    ]
                ),

            "isolated_nodes":
                int(
                    policies[
                        "isolated_scope"
                    ][
                        "isolated_nodes"
                    ].sum()
                ),

            "isolated_with_pair":
                int(
                    policies[
                        "isolated_scope"
                    ][
                        "isolated_with_local_pair"
                    ].sum()
                ),

            "strict_two_policy":
                len(
                    policies[
                        "strict_two_policy"
                    ]
                ),
        })


    return row


overall_1019 = pd.DataFrame([
    overall_row_1019(
        "BASELINE_256",
        baseline_result_1019,
        stage10_v9_policies_1016[
            "BASELINE_256"
        ],
    ),

    overall_row_1019(
        "SPARSE_9975_256",
        sparse9975_result_1019,
        stage10_v9_policies_1016[
            "SPARSE_9975_256"
        ],
    ),

    overall_row_1019(
        "SPARSE_9995_256",
        sparse9995_result_1019,
        sparse9995_policies_1019,
    ),
])


baseline_score_1019 = float(
    baseline_summary_1019[
        "score"
    ]
)


overall_1019[
    "delta_score_vs_baseline"
] = (
    overall_1019[
        "score"
    ]
    -
    baseline_score_1019
)


# ============================================================
# 7. Prefix aggregate comparison
# ============================================================

prefix_rows_1019 = []


variant_records_1019 = {
    "BASELINE_256":
        baseline_result_1019[
            "records"
        ],

    "SPARSE_9975_256":
        sparse9975_result_1019[
            "records"
        ],

    "SPARSE_9995_256":
        sparse9995_result_1019[
            "records"
        ],
}


for label, records in (
    variant_records_1019.items()
):

    for prefix in [
        "44b6",
        "6bba",
    ]:

        recs = [
            record
            for name, record
            in zip(
                dataset_names_1015,
                records,
            )
            if (
                prefix_map_1015[
                    name
                ]
                == prefix
            )
        ]


        assert recs


        s = summarise(
            recs
        )


        prefix_rows_1019.append({
            "variant":
                label,

            "prefix":
                prefix,

            "datasets":
                len(
                    recs
                ),

            "edge_jaccard":
                float(
                    s[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    s[
                        "adj_edge_jaccard"
                    ]
                ),

            "division_jaccard":
                float(
                    s[
                        "division_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    s[
                        "node_recall"
                    ]
                ),

            "score":
                float(
                    s[
                        "score"
                    ]
                ),
        })


prefix_1019 = pd.DataFrame(
    prefix_rows_1019
)


# ============================================================
# 8. Primary paired comparison:
#    SPARSE_9995_256 - BASELINE_256
# ============================================================

baseline_df_1019 = (
    baseline_result_1019[
        "dataset_df"
    ]
    .copy()
)


sparse9995_df_1019 = (
    sparse9995_result_1019[
        "dataset_df"
    ]
    .copy()
)


paired_1019 = (
    baseline_df_1019[
        [
            "dataset",
            "prefix",
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "total_node_ratio",
        ]
    ]
    .merge(
        sparse9995_df_1019[
            [
                "dataset",
                "edge_jaccard",
                "adj_edge_jaccard",
                "node_recall",
                "total_node_ratio",
            ]
        ],
        on="dataset",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_baseline",
            "_sparse9995",
        ),
    )
)


assert len(
    paired_1019
) == 40


paired_1019[
    "delta_edge_jaccard"
] = (
    paired_1019[
        "edge_jaccard_sparse9995"
    ]
    -
    paired_1019[
        "edge_jaccard_baseline"
    ]
)


paired_1019[
    "delta_adj_edge_jaccard"
] = (
    paired_1019[
        "adj_edge_jaccard_sparse9995"
    ]
    -
    paired_1019[
        "adj_edge_jaccard_baseline"
    ]
)


paired_1019[
    "delta_node_recall"
] = (
    paired_1019[
        "node_recall_sparse9995"
    ]
    -
    paired_1019[
        "node_recall_baseline"
    ]
)


paired_1019[
    "delta_node_ratio"
] = (
    paired_1019[
        "total_node_ratio_sparse9995"
    ]
    -
    paired_1019[
        "total_node_ratio_baseline"
    ]
)


# ============================================================
# 9. Paired robustness summary
# ============================================================

paired_summary_rows_1019 = []


for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        paired_1019
        if prefix == "ALL"
        else paired_1019[
            paired_1019[
                "prefix"
            ] == prefix
        ]
    )


    paired_summary_rows_1019.append({
        "prefix":
            prefix,

        "datasets":
            len(
                sub
            ),

        "edgeJ_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_edge_jaccard"
                    ]
                    > 0
                ).mean()
            ),

        "adjEdgeJ_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ]
                    > 0
                ).mean()
            ),

        "recall_sparse_better_rate":
            float(
                (
                    sub[
                        "delta_node_recall"
                    ]
                    > 0
                ).mean()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_node_recall"
                ].mean()
            ),

        "median_delta_recall":
            float(
                sub[
                    "delta_node_recall"
                ].median()
            ),

        "mean_delta_node_ratio":
            float(
                sub[
                    "delta_node_ratio"
                ].mean()
            ),
    })


paired_summary_1019 = pd.DataFrame(
    paired_summary_rows_1019
)


# ============================================================
# 10. Primary aggregate deltas
# ============================================================

delta_score_1019 = (
    float(
        sparse9995_summary_1019[
            "score"
        ]
    )
    -
    float(
        baseline_summary_1019[
            "score"
        ]
    )
)


delta_edge_1019 = (
    float(
        sparse9995_summary_1019[
            "edge_jaccard"
        ]
    )
    -
    float(
        baseline_summary_1019[
            "edge_jaccard"
        ]
    )
)


delta_adj_1019 = (
    float(
        sparse9995_summary_1019[
            "adj_edge_jaccard"
        ]
    )
    -
    float(
        baseline_summary_1019[
            "adj_edge_jaccard"
        ]
    )
)


delta_div_1019 = (
    float(
        sparse9995_summary_1019[
            "division_jaccard"
        ]
    )
    -
    float(
        baseline_summary_1019[
            "division_jaccard"
        ]
    )
)


delta_recall_1019 = (
    float(
        sparse9995_summary_1019[
            "node_recall"
        ]
    )
    -
    float(
        baseline_summary_1019[
            "node_recall"
        ]
    )
)


# ============================================================
# 11. Comparison to rejected .9975 sparse pilot
# ============================================================

delta_score_vs_9975_1019 = (
    float(
        sparse9995_summary_1019[
            "score"
        ]
    )
    -
    float(
        sparse9975_summary_1019[
            "score"
        ]
    )
)


delta_adj_vs_9975_1019 = (
    float(
        sparse9995_summary_1019[
            "adj_edge_jaccard"
        ]
    )
    -
    float(
        sparse9975_summary_1019[
            "adj_edge_jaccard"
        ]
    )
)


# ============================================================
# 12. Final result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.19 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "OVERALL FROZEN-v9 COMPARISON"
)

display(
    overall_1019[
        [
            "variant",

            "single_opportunities",
            "single_policy",

            "isolated_nodes",
            "isolated_with_pair",
            "strict_two_policy",

            "single_added",
            "two_added",
            "associations_added",
            "cleanup_removed",

            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "score",

            "delta_score_vs_baseline",
        ]
    ]
)


print()
print(
    "BY PREFIX"
)

display(
    prefix_1019
)


print()
print(
    "PAIRED ROBUSTNESS — SPARSE .9995 vs BASELINE"
)

display(
    paired_summary_1019
)


print()
print(
    "SPARSE .9995 - BASELINE"
)

print(
    "  delta score:",
    f"{delta_score_1019:+.6f}"
)

print(
    "  delta Edge J:",
    f"{delta_edge_1019:+.6f}"
)

print(
    "  delta Adj Edge J:",
    f"{delta_adj_1019:+.6f}"
)

print(
    "  delta Division J:",
    f"{delta_div_1019:+.6f}"
)

print(
    "  delta node recall:",
    f"{delta_recall_1019:+.6f}"
)


print()
print(
    "SPARSE .9995 - SPARSE .9975"
)

print(
    "  delta score:",
    f"{delta_score_vs_9975_1019:+.6f}"
)

print(
    "  delta Adj Edge J:",
    f"{delta_adj_vs_9975_1019:+.6f}"
)


print()
print(
    "No prediction GEFF was modified."
)

print(
    "Candidate tables were regenerated specifically "
    "for SPARSE_9995_256."
)

print(
    "Frozen V9 geometry/order were unchanged."
)

print(
    "No detection-threshold sweep was performed."
)

print(
    "GT was used only for final Fold0 evaluation."
)

print()
print(
    "10.19 sparse .9995 frozen-v9 evaluation: COMPLETE"
)

In [ ]:
# ============================================================
# RESULT — 10.20 Domain-Routed Stage10 Hybrid Evaluation
# Purpose:
#   Test whether the Stage10 sparse-label intervention should
#   be ROUTED BY DATASET DOMAIN rather than applied globally.
#
#   Primary hybrid:
#
#       44b6 -> SPARSE_9995_256
#       6bba -> BASELINE_256
#
#   Controls:
#
#       BASELINE_ALL
#           all datasets -> BASELINE_256
#
#       SPARSE9995_ALL
#           all datasets -> SPARSE_9995_256
#
#       REVERSE_HYBRID
#           44b6 -> BASELINE_256
#           6bba -> SPARSE_9995_256
#
#   This Cell reuses already-computed per-dataset metric records.
#   It performs NO new graph evaluation.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Runs graph processing: NO
# Uses GT: INDIRECTLY ONLY — reuses previously computed metric
#          records; no GT file is accessed in this Cell.
# Keep after running: YES — formal Stage10 routing evidence
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Preconditions
# ============================================================

REQUIRED_1020 = [
    "dataset_names_1015",
    "prefix_map_1015",

    "stage10_v9_results_1016",
    "sparse9995_result_1019",

    "summarise",
]


missing_1020 = [
    name
    for name in REQUIRED_1020
    if name not in globals()
]


assert not missing_1020, (
    "Required 10.16 / 10.19 state is missing:\n"
    + "\n".join(
        missing_1020
    )
)


baseline_records_1020 = (
    stage10_v9_results_1016[
        "BASELINE_256"
    ][
        "records"
    ]
)


sparse9995_records_1020 = (
    sparse9995_result_1019[
        "records"
    ]
)


assert len(
    baseline_records_1020
) == 40


assert len(
    sparse9995_records_1020
) == 40


assert len(
    dataset_names_1015
) == 40


print(
    "Cached Stage10 metric records:",
    "PASS"
)


# ============================================================
# 2. Build exact per-dataset routing
# ============================================================

baseline_all_1020 = []

sparse_all_1020 = []

domain_hybrid_1020 = []

reverse_hybrid_1020 = []


routing_rows_1020 = []


for (
    name,
    base_record,
    sparse_record,
) in zip(
    dataset_names_1015,
    baseline_records_1020,
    sparse9995_records_1020,
):

    prefix = (
        prefix_map_1015[
            name
        ]
    )


    assert prefix in {
        "44b6",
        "6bba",
    }


    baseline_all_1020.append(
        base_record
    )


    sparse_all_1020.append(
        sparse_record
    )


    # --------------------------------------------------------
    # Primary routing hypothesis
    #
    # Sparse-label loss only where annotation sparsity appears
    # severe: 44b6.
    # --------------------------------------------------------

    if prefix == "44b6":

        hybrid_record = (
            sparse_record
        )

        reverse_record = (
            base_record
        )

        selected_model = (
            "SPARSE_9995_256"
        )

    else:

        hybrid_record = (
            base_record
        )

        reverse_record = (
            sparse_record
        )

        selected_model = (
            "BASELINE_256"
        )


    domain_hybrid_1020.append(
        hybrid_record
    )


    reverse_hybrid_1020.append(
        reverse_record
    )


    routing_rows_1020.append({
        "dataset":
            name,

        "prefix":
            prefix,

        "selected_model":
            selected_model,

        "baseline_adj_edge_jaccard":
            float(
                base_record[
                    "adj_edge_jaccard"
                ]
            ),

        "selected_adj_edge_jaccard":
            float(
                hybrid_record[
                    "adj_edge_jaccard"
                ]
            ),

        "delta_adj_edge_jaccard":
            float(
                hybrid_record[
                    "adj_edge_jaccard"
                ]
                -
                base_record[
                    "adj_edge_jaccard"
                ]
            ),

        "baseline_edge_jaccard":
            float(
                base_record[
                    "edge_jaccard"
                ]
            ),

        "selected_edge_jaccard":
            float(
                hybrid_record[
                    "edge_jaccard"
                ]
            ),

        "delta_edge_jaccard":
            float(
                hybrid_record[
                    "edge_jaccard"
                ]
                -
                base_record[
                    "edge_jaccard"
                ]
            ),

        "baseline_recall":
            float(
                base_record[
                    "node_recall"
                ]
            ),

        "selected_recall":
            float(
                hybrid_record[
                    "node_recall"
                ]
            ),

        "delta_recall":
            float(
                hybrid_record[
                    "node_recall"
                ]
                -
                base_record[
                    "node_recall"
                ]
            ),
    })


routing_df_1020 = pd.DataFrame(
    routing_rows_1020
)


# ============================================================
# 3. Exact official aggregation
# ============================================================

summary_baseline_1020 = (
    summarise(
        baseline_all_1020
    )
)


summary_sparse_1020 = (
    summarise(
        sparse_all_1020
    )
)


summary_hybrid_1020 = (
    summarise(
        domain_hybrid_1020
    )
)


summary_reverse_1020 = (
    summarise(
        reverse_hybrid_1020
    )
)


# ============================================================
# 4. Overall comparison
# ============================================================

def summary_row_1020(
    policy,
    summary,
):

    return {
        "policy":
            policy,

        "edge_jaccard":
            float(
                summary[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                summary[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                summary[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                summary[
                    "node_recall"
                ]
            ),

        "score":
            float(
                summary[
                    "score"
                ]
            ),
    }


overall_1020 = pd.DataFrame([
    summary_row_1020(
        "BASELINE_ALL",
        summary_baseline_1020,
    ),

    summary_row_1020(
        "SPARSE9995_ALL",
        summary_sparse_1020,
    ),

    summary_row_1020(
        "DOMAIN_HYBRID_44b6_SPARSE",
        summary_hybrid_1020,
    ),

    summary_row_1020(
        "REVERSE_HYBRID_6bba_SPARSE",
        summary_reverse_1020,
    ),
])


baseline_score_1020 = float(
    summary_baseline_1020[
        "score"
    ]
)


overall_1020[
    "delta_score_vs_baseline"
] = (
    overall_1020[
        "score"
    ]
    -
    baseline_score_1020
)


# ============================================================
# 5. Prefix aggregate verification
# ============================================================

prefix_rows_1020 = []


for prefix in [
    "44b6",
    "6bba",
]:

    idx = [
        i
        for i, name in enumerate(
            dataset_names_1015
        )
        if (
            prefix_map_1015[
                name
            ]
            == prefix
        )
    ]


    base_recs = [
        baseline_all_1020[
            i
        ]
        for i in idx
    ]


    hybrid_recs = [
        domain_hybrid_1020[
            i
        ]
        for i in idx
    ]


    base_summary = (
        summarise(
            base_recs
        )
    )


    hybrid_summary = (
        summarise(
            hybrid_recs
        )
    )


    for label, s in [
        (
            "BASELINE_ALL",
            base_summary,
        ),
        (
            "DOMAIN_HYBRID",
            hybrid_summary,
        ),
    ]:

        prefix_rows_1020.append({
            "prefix":
                prefix,

            "policy":
                label,

            "datasets":
                len(
                    idx
                ),

            "edge_jaccard":
                float(
                    s[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    s[
                        "adj_edge_jaccard"
                    ]
                ),

            "division_jaccard":
                float(
                    s[
                        "division_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    s[
                        "node_recall"
                    ]
                ),

            "score":
                float(
                    s[
                        "score"
                    ]
                ),
        })


prefix_1020 = pd.DataFrame(
    prefix_rows_1020
)


# ============================================================
# 6. Paired robustness versus BASELINE_ALL
#
# For 6bba the primary hybrid intentionally uses baseline,
# therefore delta = 0 by construction.
# ============================================================

paired_rows_1020 = []


for prefix in [
    "ALL",
    "44b6",
    "6bba",
]:

    sub = (
        routing_df_1020
        if prefix == "ALL"
        else routing_df_1020[
            routing_df_1020[
                "prefix"
            ] == prefix
        ]
    )


    paired_rows_1020.append({
        "prefix":
            prefix,

        "datasets":
            len(
                sub
            ),

        "AdjEdgeJ_improved":
            int(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).sum()
            ),

        "AdjEdgeJ_worse":
            int(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] < 0
                ).sum()
            ),

        "AdjEdgeJ_equal":
            int(
                (
                    sub[
                        "delta_adj_edge_jaccard"
                    ] == 0
                ).sum()
            ),

        "mean_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                sub[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                sub[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                sub[
                    "delta_recall"
                ].mean()
            ),
    })


paired_1020 = pd.DataFrame(
    paired_rows_1020
)


# ============================================================
# 7. Primary deltas
# ============================================================

hybrid_score_1020 = float(
    summary_hybrid_1020[
        "score"
    ]
)


delta_score_1020 = (
    hybrid_score_1020
    -
    baseline_score_1020
)


delta_edge_1020 = (
    float(
        summary_hybrid_1020[
            "edge_jaccard"
        ]
    )
    -
    float(
        summary_baseline_1020[
            "edge_jaccard"
        ]
    )
)


delta_adj_1020 = (
    float(
        summary_hybrid_1020[
            "adj_edge_jaccard"
        ]
    )
    -
    float(
        summary_baseline_1020[
            "adj_edge_jaccard"
        ]
    )
)


delta_div_1020 = (
    float(
        summary_hybrid_1020[
            "division_jaccard"
        ]
    )
    -
    float(
        summary_baseline_1020[
            "division_jaccard"
        ]
    )
)


delta_recall_1020 = (
    float(
        summary_hybrid_1020[
            "node_recall"
        ]
    )
    -
    float(
        summary_baseline_1020[
            "node_recall"
        ]
    )
)


# ============================================================
# 8. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.20 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "OVERALL ROUTING COMPARISON"
)

display(
    overall_1020
)


print()
print(
    "BY PREFIX"
)

display(
    prefix_1020
)


print()
print(
    "PAIRED ROBUSTNESS VS BASELINE"
)

display(
    paired_1020
)


print()
print(
    "DOMAIN HYBRID - BASELINE"
)

print(
    "  delta score:",
    f"{delta_score_1020:+.6f}"
)

print(
    "  delta Edge J:",
    f"{delta_edge_1020:+.6f}"
)

print(
    "  delta Adj Edge J:",
    f"{delta_adj_1020:+.6f}"
)

print(
    "  delta Division J:",
    f"{delta_div_1020:+.6f}"
)

print(
    "  delta node recall:",
    f"{delta_recall_1020:+.6f}"
)


print()
print(
    "Routing:"
)

print(
    "  44b6 -> SPARSE_9995_256"
)

print(
    "  6bba -> BASELINE_256"
)


print()
print(
    "No graph was processed."
)

print(
    "No GT file was accessed."
)

print(
    "No prediction was modified."
)

print(
    "No training or inference was run."
)

print()
print(
    "10.20 domain-routed hybrid evaluation: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.21 Second-Seed Paired 256-Iteration Training
# Purpose:
#   Confirm whether the 44b6 benefit of sparse-ignore=.9995
#   survives an independent training seed.
#
#   Train TWO paired controls from the exact same Warm2:
#
#       A. BASELINE_SEED2_256
#           det_ignore_prob = None
#
#       B. SPARSE9995_SEED2_256
#           det_ignore_prob = .9995
#
#   Frozen relative to first Stage10 pilot:
#       Fold              = 0
#       Warm start        = Warm2
#       epochs            = 1
#       max_iters         = 256
#       batch size        = 8
#       learning rate     = 1e-5
#       det loss weight   = 1.0
#       det neg weight    = .01
#       downsample        = 1,4,4
#       window size       = 2
#       training pool     = 5.0 um
#
#   NEW predeclared seed:
#       seed = 1641
#
# Changes files: YES
#   - two checkpoint directories
#   - two training logs
# Runs training: YES — 256 iterations x 2
# Runs inference: NO
# Uses GT: YES — normal sparse training supervision
# Keep after running: YES until second-seed decision closes
# ============================================================

from pathlib import Path
import hashlib
import os
import re
import subprocess
import sys
import time

import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts/train_unet_transformer_mps.py"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

LOCAL_SCRIPTS = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

LOG_DIR_1021 = (
    PROJECT
    / "reports/stage10_seed2"
)

LOG_DIR_1021.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Frozen hashes
# ============================================================

EXPECTED_TRAIN_SCRIPT_SHA_1021 = (
    "ce4fe8b3a16771039e1c687c02e71cad2747ee48d94417f0a26c929192f1be7b"
)

EXPECTED_WARM2_SHA_1021 = (
    "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"
)


def sha256_1021(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


assert (
    Path(sys.executable).resolve()
    ==
    PYTHON.resolve()
)

assert (
    sha256_1021(TRAIN_SCRIPT)
    ==
    EXPECTED_TRAIN_SCRIPT_SHA_1021
)

assert (
    sha256_1021(WARM2)
    ==
    EXPECTED_WARM2_SHA_1021
)

print("Training script SHA: PASS")
print("Warm2 checkpoint SHA: PASS")
print("Python:", PYTHON)


# ============================================================
# 3. Environment
# ============================================================

env_1021 = os.environ.copy()

pythonpath_parts = [
    str(LOCAL_SCRIPTS),
    str(OFFICIAL_SCRIPTS),
]

if env_1021.get("PYTHONPATH"):
    pythonpath_parts.append(
        env_1021["PYTHONPATH"]
    )

env_1021["PYTHONPATH"] = (
    os.pathsep.join(
        pythonpath_parts
    )
)


# ============================================================
# 4. Frozen second-seed configuration
# ============================================================

SEED_1021 = 1641
MAX_ITERS_1021 = 256
LR_1021 = 1e-5
BATCH_SIZE_1021 = 8
NUM_WORKERS_1021 = 8
DET_LOSS_WEIGHT_1021 = 1.0
DET_NEG_WEIGHT_1021 = 0.01
POOL_KERNEL_UM_1021 = 5.0


VARIANTS_1021 = {

    "BASELINE_SEED2_256": {
        "method":
            "stage10_seed2_baseline_256",

        "ignore_prob":
            None,

        "log":
            LOG_DIR_1021
            / "baseline_seed1641_256.log",
    },

    "SPARSE9995_SEED2_256": {
        "method":
            "stage10_seed2_sparse9995_256",

        "ignore_prob":
            0.9995,

        "log":
            LOG_DIR_1021
            / "sparse9995_seed1641_256.log",
    },
}


# ============================================================
# 5. No silent overwrite
# ============================================================

for label, cfg in VARIANTS_1021.items():

    checkpoint = (
        PROJECT
        / "models/official_baseline"
        / cfg["method"]
        / "split_0/edge_predictor_best.pth"
    )

    assert not cfg["log"].exists(), (
        f"{label}: log already exists:\n"
        f"{cfg['log']}"
    )

    assert not checkpoint.exists(), (
        f"{label}: checkpoint already exists:\n"
        f"{checkpoint}"
    )


# ============================================================
# 6. Train paired variants
# ============================================================

rows_1021 = []


print()
print("=" * 72)
print("10.21 SECOND-SEED PAIRED PILOT")
print("=" * 72)

print("Seed:", SEED_1021)
print("Warm start:", WARM2)
print("Iterations per model:", MAX_ITERS_1021)


for label, cfg in VARIANTS_1021.items():

    cmd = [
        str(PYTHON),
        str(TRAIN_SCRIPT),

        "--data-dir",
        str(TRAIN_ROOT),

        "--splits",
        str(SPLITS),

        "--split",
        "0",

        "--method",
        cfg["method"],

        "--epochs",
        "1",

        "--lr",
        str(LR_1021),

        "--batch-size",
        str(BATCH_SIZE_1021),

        "--num-workers",
        str(NUM_WORKERS_1021),

        "--init-weights",
        str(WARM2),

        "--downsample",
        "1,4,4",

        "--det-loss-weight",
        str(DET_LOSS_WEIGHT_1021),

        "--det-neg-weight",
        str(DET_NEG_WEIGHT_1021),

        "--max-iters",
        str(MAX_ITERS_1021),

        "--seed",
        str(SEED_1021),

        "--window-size",
        "2",

        "--pool-kernel-um",
        str(POOL_KERNEL_UM_1021),
    ]


    # --------------------------------------------------------
    # ONLY experimental difference
    # --------------------------------------------------------

    if cfg["ignore_prob"] is not None:

        cmd += [
            "--det-ignore-prob",
            str(
                cfg["ignore_prob"]
            ),
        ]


    print()
    print("Starting:", label)

    print(
        "  det_ignore_prob:",
        cfg["ignore_prob"]
    )

    print(
        "  log:",
        cfg["log"]
    )


    start = time.time()


    with cfg["log"].open(
        "w",
        encoding="utf-8",
    ) as log_file:

        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env_1021,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )


    elapsed = (
        time.time()
        - start
    )


    log_text = cfg["log"].read_text(
        encoding="utf-8",
        errors="replace",
    )


    if proc.returncode != 0:

        print(
            "\n".join(
                log_text.splitlines()[-100:]
            )
        )

        raise RuntimeError(
            f"{label} training failed."
        )


    # --------------------------------------------------------
    # Provenance
    # --------------------------------------------------------

    assert (
        "Fold 0: 159 train, 40 test"
        in log_text
    )

    assert (
        "Full-model init weights loaded"
        in log_text
    )

    assert (
        str(WARM2)
        in log_text
    )

    assert (
        "Starting training for 1 epochs (batch_size=8)"
        in log_text
    )

    assert (
        "/256"
        in log_text
    )


    if cfg["ignore_prob"] is None:

        assert re.search(
            r"Detection loss:\s*"
            r"weight=1(?:\.0+)?,\s*"
            r"neg_weight=0\.01,\s*"
            r"ignore_prob=None",
            log_text,
        )

    else:

        assert re.search(
            r"Detection loss:\s*"
            r"weight=1(?:\.0+)?,\s*"
            r"neg_weight=0\.01,\s*"
            r"ignore_prob=0\.9995",
            log_text,
        )


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    checkpoint = (
        PROJECT
        / "models/official_baseline"
        / cfg["method"]
        / "split_0/edge_predictor_best.pth"
    )

    assert checkpoint.exists()


    checkpoint_sha = (
        sha256_1021(
            checkpoint
        )
    )


    # --------------------------------------------------------
    # Parse final diagnostic
    # --------------------------------------------------------

    epoch_pattern = re.compile(
        r"Epoch\s+\d+/\d+\s+\|\s+"
        r"edge=([0-9.eE+-]+)\s+\|\s+"
        r"det=([0-9.eE+-]+)\s+\|\s+"
        r"test_loss=([0-9.eE+-]+)\s+\|\s+"
        r"acc=([0-9.eE+-]+)\s+\|\s+"
        r"recall=([0-9.eE+-]+)"
    )


    matches = list(
        epoch_pattern.finditer(
            log_text
        )
    )

    assert matches


    m = matches[-1]


    edge_loss = float(
        m.group(1)
    )

    det_loss = float(
        m.group(2)
    )

    test_loss = float(
        m.group(3)
    )

    test_acc = float(
        m.group(4)
    )

    test_recall = float(
        m.group(5)
    )


    rows_1021.append({
        "variant":
            label,

        "seed":
            SEED_1021,

        "det_ignore_prob":
            (
                np.nan
                if cfg["ignore_prob"] is None
                else cfg["ignore_prob"]
            ),

        "return_code":
            proc.returncode,

        "elapsed_min":
            elapsed / 60.0,

        "edge_loss":
            edge_loss,

        "det_loss":
            det_loss,

        "test_loss":
            test_loss,

        "test_acc":
            test_acc,

        "test_recall":
            test_recall,

        "acc_x_recall":
            test_acc * test_recall,

        "checkpoint":
            str(checkpoint),

        "checkpoint_sha256":
            checkpoint_sha,
    })


    print(
        f"{label}: "
        f"COMPLETE, "
        f"{elapsed / 60:.1f} min"
    )


# ============================================================
# 7. Paired training diagnostic
# ============================================================

result_1021 = pd.DataFrame(
    rows_1021
)


assert len(
    result_1021
) == 2


base_1021 = (
    result_1021[
        result_1021[
            "variant"
        ] == "BASELINE_SEED2_256"
    ]
    .iloc[0]
)


sparse_1021 = (
    result_1021[
        result_1021[
            "variant"
        ] == "SPARSE9995_SEED2_256"
    ]
    .iloc[0]
)


# ============================================================
# 8. Result
# ============================================================

print()
print("=" * 72)
print("10.21 RESULT")
print("=" * 72)

print()
print(
    "SECOND-SEED TRAINING DIAGNOSTICS "
    "— NOT COMPETITION METRICS"
)

display(
    result_1021[
        [
            "variant",
            "seed",
            "det_ignore_prob",
            "elapsed_min",
            "edge_loss",
            "det_loss",
            "test_loss",
            "test_acc",
            "test_recall",
            "acc_x_recall",
            "checkpoint_sha256",
        ]
    ]
)


print()
print(
    "SPARSE9995_SEED2 - BASELINE_SEED2 diagnostics"
)

print(
    "  det loss:",
    f"{float(sparse_1021['det_loss'] - base_1021['det_loss']):+.6f}"
)

print(
    "  test recall:",
    f"{float(sparse_1021['test_recall'] - base_1021['test_recall']):+.6f}"
)

print(
    "  acc*recall:",
    f"{float(sparse_1021['acc_x_recall'] - base_1021['acc_x_recall']):+.6f}"
)


print()
print(
    "No inference was run."
)

print(
    "No competition metric was evaluated."
)

print()
print(
    "10.21 second-seed paired training: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.22 Seed2 44b6-Only Paired Inference
# Purpose:
#   Generate the four prediction sets required to test whether
#   the first-seed 44b6 benefit of det_ignore_prob=.9995
#   reproduces under independent seed=1641.
#
#   Evaluate ONLY the 15 Fold0 44b6 validation datasets:
#
#       BASELINE_SEED2:
#           det=.995
#           det=.9975
#
#       SPARSE9995_SEED2:
#           det=.995
#           det=.9975
#
#   A temporary 44b6-only split manifest is created while
#   preserving the exact original Fold0 dataset identities.
#
# Changes files: YES
#   - reports/stage10_seed2/split_44b6_only.json
#   - four inference logs
#   - 4 x 15 prediction GEFFs
# Runs training: NO
# Runs inference: YES
# Uses GT: NO
# Keep after running: YES — required for 10.23 validation
# ============================================================

from pathlib import Path

import copy
import hashlib
import json
import os
import re
import subprocess
import sys
import time

import pandas as pd


# ============================================================
# 1. Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON = (
    PROJECT
    / ".venv/bin/python"
)

PREDICT_SCRIPT = (
    PROJECT
    / "scripts/predict_unet_transformer_mps.py"
)

LOCAL_SCRIPTS = (
    PROJECT
    / "scripts"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

ORIGINAL_SPLITS = (
    PROJECT
    / "configs/dataset_splits.json"
)

REPORT_DIR_1022 = (
    PROJECT
    / "reports/stage10_seed2"
)

REPORT_DIR_1022.mkdir(
    parents=True,
    exist_ok=True,
)

SPLITS_44B6_1022 = (
    REPORT_DIR_1022
    / "split_44b6_only.json"
)

PRED_ROOT_1022 = (
    PROJECT
    / "predictions/heqiuyan"
)


BASELINE_CKPT_1022 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_seed2_baseline_256/"
      "split_0/edge_predictor_best.pth"
)

SPARSE_CKPT_1022 = (
    PROJECT
    / "models/official_baseline/"
      "stage10_seed2_sparse9995_256/"
      "split_0/edge_predictor_best.pth"
)


# ============================================================
# 2. Frozen predictor identity
# ============================================================

EXPECTED_PREDICT_SHA_1022 = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)


def sha256_1022(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ============================================================
# 3. Preconditions
# ============================================================

assert PROJECT.exists()
assert PYTHON.exists()
assert PREDICT_SCRIPT.exists()
assert ORIGINAL_SPLITS.exists()
assert TRAIN_ROOT.exists()

assert BASELINE_CKPT_1022.exists()
assert SPARSE_CKPT_1022.exists()


assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON.resolve()
), (
    "Notebook is not running from the Biohub project .venv."
)


predict_sha_1022 = (
    sha256_1022(
        PREDICT_SCRIPT
    )
)


assert (
    predict_sha_1022
    ==
    EXPECTED_PREDICT_SHA_1022
), (
    "Frozen prediction adapter changed.\n"
    f"Observed SHA: {predict_sha_1022}"
)


baseline_ckpt_sha_1022 = (
    sha256_1022(
        BASELINE_CKPT_1022
    )
)

sparse_ckpt_sha_1022 = (
    sha256_1022(
        SPARSE_CKPT_1022
    )
)


assert (
    baseline_ckpt_sha_1022
    != sparse_ckpt_sha_1022
), (
    "Seed2 baseline and sparse checkpoints are unexpectedly identical."
)


print(
    "Predictor SHA:",
    "PASS"
)

print(
    "Baseline seed2 checkpoint:",
    baseline_ckpt_sha_1022
)

print(
    "Sparse seed2 checkpoint:",
    sparse_ckpt_sha_1022
)


# ============================================================
# 4. Recover EXACT 15 Fold0 44b6 datasets
# ============================================================

assert (
    "dataset_names_1015"
    in globals()
)

assert (
    "prefix_map_1015"
    in globals()
)


fold0_44b6_1022 = [
    name
    for name in dataset_names_1015
    if (
        prefix_map_1015[
            name
        ]
        == "44b6"
    )
]


assert (
    len(
        fold0_44b6_1022
    )
    == 15
), (
    f"Expected 15 44b6 Fold0 datasets, "
    f"got {len(fold0_44b6_1022)}"
)


assert all(
    name.startswith(
        "44b6"
    )
    for name in fold0_44b6_1022
)


print(
    "Fold0 44b6 subset:",
    len(
        fold0_44b6_1022
    ),
    "datasets",
)


# ============================================================
# 5. Create 44b6-only inference split
#
# Preserve the original JSON structure exactly; change ONLY
# Fold0's test list from 40 datasets -> 15 44b6 datasets.
# ============================================================

with ORIGINAL_SPLITS.open(
    "r",
    encoding="utf-8",
) as f:

    split_payload_1022 = (
        json.load(
            f
        )
    )


subset_payload_1022 = (
    copy.deepcopy(
        split_payload_1022
    )
)


if (
    isinstance(
        subset_payload_1022,
        dict,
    )
    and
    "folds"
    in subset_payload_1022
):

    folds_1022 = (
        subset_payload_1022[
            "folds"
        ]
    )

else:

    folds_1022 = (
        subset_payload_1022
    )


assert isinstance(
    folds_1022,
    list,
)


assert len(
    folds_1022
) > 0


original_fold0_test_1022 = list(
    folds_1022[
        0
    ][
        "test"
    ]
)


assert len(
    original_fold0_test_1022
) == 40


assert set(
    fold0_44b6_1022
).issubset(
    set(
        original_fold0_test_1022
    )
)


folds_1022[
    0
][
    "test"
] = list(
    fold0_44b6_1022
)


with SPLITS_44B6_1022.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        subset_payload_1022,
        f,
        indent=2,
    )


# Re-read verification.
with SPLITS_44B6_1022.open(
    "r",
    encoding="utf-8",
) as f:

    verify_payload_1022 = (
        json.load(
            f
        )
    )


if (
    isinstance(
        verify_payload_1022,
        dict,
    )
    and
    "folds"
    in verify_payload_1022
):

    verify_fold0_1022 = (
        verify_payload_1022[
            "folds"
        ][
            0
        ]
    )

else:

    verify_fold0_1022 = (
        verify_payload_1022[
            0
        ]
    )


assert (
    verify_fold0_1022[
        "test"
    ]
    ==
    fold0_44b6_1022
)


print(
    "44b6-only split manifest:",
    "PASS"
)

print(
    SPLITS_44B6_1022
)


# ============================================================
# 6. Check checkpoint architecture configs
# ============================================================

baseline_config_path_1022 = (
    BASELINE_CKPT_1022.parent
    / "config.json"
)

sparse_config_path_1022 = (
    SPARSE_CKPT_1022.parent
    / "config.json"
)


assert baseline_config_path_1022.exists()
assert sparse_config_path_1022.exists()


with baseline_config_path_1022.open(
    "r",
    encoding="utf-8",
) as f:

    baseline_config_1022 = (
        json.load(
            f
        )
    )


with sparse_config_path_1022.open(
    "r",
    encoding="utf-8",
) as f:

    sparse_config_1022 = (
        json.load(
            f
        )
    )


architecture_keys_1022 = [
    "unet_out_channels",
    "unet_layers",
    "downsample",
    "window_size",
    "pool_kernel_um",
]


baseline_arch_1022 = {
    key:
        baseline_config_1022.get(
            key
        )
    for key in architecture_keys_1022
}


sparse_arch_1022 = {
    key:
        sparse_config_1022.get(
            key
        )
    for key in architecture_keys_1022
}


assert (
    baseline_arch_1022
    ==
    sparse_arch_1022
), (
    "Seed2 checkpoint architecture/config mismatch."
)


print(
    "Checkpoint architecture parity:",
    "PASS"
)

print(
    baseline_arch_1022
)


# ============================================================
# 7. PYTHONPATH
# ============================================================

env_1022 = (
    os.environ.copy()
)


pythonpath_parts_1022 = [
    str(
        LOCAL_SCRIPTS
    ),
    str(
        OFFICIAL_SCRIPTS
    ),
]


if env_1022.get(
    "PYTHONPATH"
):

    pythonpath_parts_1022.append(
        env_1022[
            "PYTHONPATH"
        ]
    )


env_1022[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts_1022
)


# ============================================================
# 8. Frozen inference matrix
# ============================================================

UNET_BATCH_SIZE_1022 = (
    4
)


RUNS_1022 = [

    {
        "variant":
            "BASELINE_SEED2_DET995",

        "checkpoint":
            BASELINE_CKPT_1022,

        "det_threshold":
            0.995,

        "method":
            "stage10_seed2_baseline1641_det995_44b6",

        "log":
            REPORT_DIR_1022
            / "baseline1641_det995_44b6.log",
    },


    {
        "variant":
            "BASELINE_SEED2_DET9975",

        "checkpoint":
            BASELINE_CKPT_1022,

        "det_threshold":
            0.9975,

        "method":
            "stage10_seed2_baseline1641_det9975_44b6",

        "log":
            REPORT_DIR_1022
            / "baseline1641_det9975_44b6.log",
    },


    {
        "variant":
            "SPARSE9995_SEED2_DET995",

        "checkpoint":
            SPARSE_CKPT_1022,

        "det_threshold":
            0.995,

        "method":
            "stage10_seed2_sparse9995_1641_det995_44b6",

        "log":
            REPORT_DIR_1022
            / "sparse9995_1641_det995_44b6.log",
    },


    {
        "variant":
            "SPARSE9995_SEED2_DET9975",

        "checkpoint":
            SPARSE_CKPT_1022,

        "det_threshold":
            0.9975,

        "method":
            "stage10_seed2_sparse9995_1641_det9975_44b6",

        "log":
            REPORT_DIR_1022
            / "sparse9995_1641_det9975_44b6.log",
    },
]


# ============================================================
# 9. No silent overwrite
# ============================================================

for run in RUNS_1022:

    output_dir = (
        PRED_ROOT_1022
        / run[
            "method"
        ]
        / "split_0"
    )


    assert not run[
        "log"
    ].exists(), (
        f"Existing log:\n"
        f"{run['log']}"
    )


    existing_geffs = (
        list(
            output_dir.glob(
                "*.geff"
            )
        )
        if output_dir.exists()
        else []
    )


    assert not existing_geffs, (
        f"Existing prediction GEFFs found:\n"
        f"{output_dir}"
    )


# ============================================================
# 10. Log parser
# ============================================================

start_pattern_1022 = re.compile(
    r"Fold\s+0:\s+"
    r"(\d+)\s+datasets\s+\|\s+"
    r"weights=(.+?)\s+\|\s+"
    r"device=(\w+)\s+\|\s+"
    r"window_size=(\d+)\s+\|\s+"
    r"pool_kernel_um=([0-9.]+)"
)


save_pattern_1022 = re.compile(
    r"Saved\s+(\d+)\s+predictions\s+to\s+(.+)"
)


# ============================================================
# 11. Run four targeted inference jobs
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.22 SEED2 44b6-ONLY PAIRED INFERENCE"
)

print(
    "=" * 72
)

print(
    "Datasets:",
    15
)

print(
    "Runs:",
    4
)

print(
    "Pool:",
    "3.0 um"
)

print(
    "TTA:",
    "default ON"
)

print(
    "ILP:",
    False
)


rows_1022 = []


for run in RUNS_1022:

    checkpoint = (
        run[
            "checkpoint"
        ]
    )

    det_threshold = float(
        run[
            "det_threshold"
        ]
    )

    method = (
        run[
            "method"
        ]
    )

    log_path = (
        run[
            "log"
        ]
    )


    cmd_1022 = [
        str(
            PYTHON
        ),

        str(
            PREDICT_SCRIPT
        ),

        "--data-dir",
        str(
            TRAIN_ROOT
        ),

        "--splits",
        str(
            SPLITS_44B6_1022
        ),

        "--split",
        "0",

        "--method",
        method,

        "--weights",
        str(
            checkpoint
        ),

        "--det-threshold",
        str(
            det_threshold
        ),

        "--unet-batch-size",
        str(
            UNET_BATCH_SIZE_1022
        ),
    ]


    print()
    print(
        "Starting:",
        run[
            "variant"
        ]
    )

    print(
        "  threshold:",
        det_threshold
    )

    print(
        "  log:",
        log_path
    )


    start_1022 = (
        time.time()
    )


    with log_path.open(
        "w",
        encoding="utf-8",
    ) as log_file:

        proc_1022 = subprocess.run(
            cmd_1022,
            cwd=PROJECT,
            env=env_1022,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )


    elapsed_1022 = (
        time.time()
        - start_1022
    )


    log_text_1022 = (
        log_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )


    if (
        proc_1022.returncode
        != 0
    ):

        print()
        print(
            "LAST 100 LOG LINES"
        )

        print(
            "\n".join(
                log_text_1022
                .splitlines()[
                    -100:
                ]
            )
        )

        raise RuntimeError(
            f"{run['variant']} failed."
        )


    # --------------------------------------------------------
    # Parse frozen inference configuration
    # --------------------------------------------------------

    start_matches_1022 = list(
        start_pattern_1022.finditer(
            log_text_1022
        )
    )


    assert start_matches_1022, (
        f"{run['variant']}: "
        "could not parse inference configuration."
    )


    sm_1022 = (
        start_matches_1022[
            -1
        ]
    )


    dataset_count_1022 = int(
        sm_1022.group(
            1
        )
    )

    logged_weights_1022 = Path(
        sm_1022.group(
            2
        ).strip()
    ).resolve()

    device_1022 = (
        sm_1022.group(
            3
        )
    )

    window_1022 = int(
        sm_1022.group(
            4
        )
    )

    pool_1022 = float(
        sm_1022.group(
            5
        )
    )


    assert (
        dataset_count_1022
        == 15
    )

    assert (
        logged_weights_1022
        ==
        checkpoint.resolve()
    )

    assert (
        device_1022
        == "mps"
    )

    assert (
        window_1022
        == 2
    )

    assert (
        pool_1022
        == 3.0
    )


    # --------------------------------------------------------
    # Verify saved predictions
    # --------------------------------------------------------

    save_matches_1022 = list(
        save_pattern_1022.finditer(
            log_text_1022
        )
    )


    assert save_matches_1022


    save_match_1022 = (
        save_matches_1022[
            -1
        ]
    )


    reported_count_1022 = int(
        save_match_1022.group(
            1
        )
    )


    output_dir_1022 = Path(
        save_match_1022.group(
            2
        ).strip()
    )


    if not output_dir_1022.is_absolute():

        output_dir_1022 = (
            PROJECT
            / output_dir_1022
        )


    output_dir_1022 = (
        output_dir_1022.resolve()
    )


    geffs_1022 = sorted(
        output_dir_1022.glob(
            "*.geff"
        )
    )


    assert (
        reported_count_1022
        == 15
    )

    assert (
        len(
            geffs_1022
        )
        == 15
    )


    actual_names_1022 = {
        p.name.removesuffix(
            ".geff"
        )
        for p in geffs_1022
    }


    assert (
        actual_names_1022
        ==
        set(
            fold0_44b6_1022
        )
    )


    rows_1022.append({
        "variant":
            run[
                "variant"
            ],

        "checkpoint_sha":
            sha256_1022(
                checkpoint
            ),

        "det_threshold":
            det_threshold,

        "datasets":
            dataset_count_1022,

        "return_code":
            proc_1022.returncode,

        "device":
            device_1022,

        "window_size":
            window_1022,

        "pool_kernel_um":
            pool_1022,

        "geffs":
            len(
                geffs_1022
            ),

        "elapsed_min":
            elapsed_1022
            / 60.0,

        "output_dir":
            str(
                output_dir_1022
            ),

        "log":
            str(
                log_path
            ),
    })


    print(
        f"{run['variant']}: "
        f"COMPLETE — "
        f"{len(geffs_1022)}/15, "
        f"{elapsed_1022 / 60:.1f} min"
    )


# ============================================================
# 12. Final verification
# ============================================================

inference_1022 = pd.DataFrame(
    rows_1022
)


assert (
    len(
        inference_1022
    )
    == 4
)


assert (
    inference_1022[
        "geffs"
    ]
    == 15
).all()


assert (
    inference_1022[
        "datasets"
    ]
    == 15
).all()


assert (
    inference_1022[
        "pool_kernel_um"
    ]
    == 3.0
).all()


assert (
    inference_1022[
        "window_size"
    ]
    == 2
).all()


assert (
    inference_1022[
        "device"
    ]
    == "mps"
).all()


# Two thresholds x two models.
threshold_counts_1022 = (
    inference_1022[
        "det_threshold"
    ]
    .value_counts()
    .to_dict()
)


assert (
    threshold_counts_1022
    ==
    {
        0.995:
            2,

        0.9975:
            2,
    }
)


# ============================================================
# 13. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.22 RESULT"
)

print(
    "=" * 72
)


display(
    inference_1022[
        [
            "variant",
            "det_threshold",
            "datasets",
            "device",
            "window_size",
            "pool_kernel_um",
            "geffs",
            "elapsed_min",
            "output_dir",
        ]
    ]
)


print()
print(
    "Seed2 baseline .995:",
    "COMPLETE"
)

print(
    "Seed2 baseline .9975:",
    "COMPLETE"
)

print(
    "Seed2 sparse .9995 .995:",
    "COMPLETE"
)

print(
    "Seed2 sparse .9995 .9975:",
    "COMPLETE"
)


print()
print(
    "All four runs contain exactly the same "
    "15 Fold0 44b6 datasets."
)

print(
    "No GT was used."
)

print(
    "No metric was evaluated."
)

print()
print(
    "10.22 seed2 44b6 paired inference: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.23 Seed2 44b6 Frozen-v9 Reproduction Gate
# Purpose:
#   Test whether the 44b6 benefit of sparse-ignore=.9995
#   reproduces under independent training seed=1641.
#
#   Evaluate ONLY the exact 15 Fold0 44b6 validation datasets.
#
#   For EACH seed2 checkpoint independently:
#
#       det=.995 graph
#       -> frozen G2
#       -> frozen K9 using its OWN det=.9975 graph
#       -> regenerate single-edge candidates
#       -> mutual single-edge policy, residual<=4 um
#       -> regenerate isolated-middle candidates
#       -> strict two-edge policy, residual<=4 um
#       -> single-edge recovery
#       -> strict two-edge recovery
#       -> frozen cleanup
#       -> locked official metric
#
#   Primary comparison:
#
#       SPARSE9995_SEED2 - BASELINE_SEED2
#
#   First-seed 44b6 result is shown only as a replication
#   reference; it is NOT used to alter any policy.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Runs graph processing: YES — 15 datasets x 2 variants
# Uses GT: YES — final metric evaluation only
# Keep after running: YES — formal Stage10 seed-robustness evidence
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Preconditions
# ============================================================

REQUIRED_1023 = [
    "PROJECT",
    "TRAIN_DIR",

    "dataset_names_1015",
    "prefix_map_1015",

    "census_single_edge_gaps_1015",
    "build_mutual_policy_1015",

    "census_isolated_bridges_1015",
    "build_strict_two_edge_policy_1015",

    "prepare_precleanup_graph_1015",

    "add_available_repairs_0907",
    "add_two_edge_bridges_0912",
    "apply_frozen_component_cleanup",

    "load_graph",
    "evaluate_graph_1016",
    "summarise",
]


missing_1023 = [
    name
    for name in REQUIRED_1023
    if name not in globals()
]


assert not missing_1023, (
    "Required validated V9 state is missing:\n"
    + "\n".join(
        missing_1023
    )
)


# Exact 15 Fold0 44b6 datasets.
datasets_44b6_1023 = [
    name
    for name in dataset_names_1015
    if (
        prefix_map_1015[
            name
        ]
        == "44b6"
    )
]


assert (
    len(
        datasets_44b6_1023
    )
    == 15
)


assert all(
    name.startswith(
        "44b6"
    )
    for name in datasets_44b6_1023
)


print(
    "Fold0 44b6 subset:",
    len(
        datasets_44b6_1023
    ),
    "datasets"
)


# ============================================================
# 2. Seed2 prediction directories
# ============================================================

PRED_ROOT_1023 = (
    PROJECT
    / "predictions/heqiuyan"
)


VARIANTS_1023 = {

    "BASELINE_SEED2_256": {

        "pred_995":
            PRED_ROOT_1023
            / "stage10_seed2_baseline1641_det995_44b6"
            / "split_0",

        "pred_9975":
            PRED_ROOT_1023
            / "stage10_seed2_baseline1641_det9975_44b6"
            / "split_0",
    },


    "SPARSE9995_SEED2_256": {

        "pred_995":
            PRED_ROOT_1023
            / "stage10_seed2_sparse9995_1641_det995_44b6"
            / "split_0",

        "pred_9975":
            PRED_ROOT_1023
            / "stage10_seed2_sparse9995_1641_det9975_44b6"
            / "split_0",
    },
}


# ============================================================
# 3. Prediction completeness guards
# ============================================================

expected_44b6_1023 = set(
    datasets_44b6_1023
)


for label, cfg in (
    VARIANTS_1023.items()
):

    for key in [
        "pred_995",
        "pred_9975",
    ]:

        pred_dir = cfg[
            key
        ]


        assert pred_dir.exists(), (
            f"{label}: missing directory:\n"
            f"{pred_dir}"
        )


        geffs = sorted(
            pred_dir.glob(
                "*.geff"
            )
        )


        assert (
            len(
                geffs
            )
            == 15
        ), (
            f"{label} {key}: "
            f"{len(geffs)} GEFFs, expected 15."
        )


        names = {
            p.name.removesuffix(
                ".geff"
            )
            for p in geffs
        }


        assert (
            names
            ==
            expected_44b6_1023
        ), (
            f"{label} {key}: "
            "44b6 subset mismatch."
        )


print(
    "Seed2 prediction completeness:",
    "PASS"
)


# ============================================================
# 4. Subset-aware V9 policy builder
#
# IMPORTANT:
# 10.16's builder loops all 40 datasets, so we intentionally
# define a subset-aware version here rather than mutating it.
# ============================================================

def build_v9_policies_subset_1023(
    dataset_names,
    pred_995_dir,
    pred_9975_dir,
    label,
):

    single_parts = []

    isolated_parts = []

    isolated_scope_rows = []


    print()
    print(
        f"Regenerating candidates: {label}"
    )


    for i, name in enumerate(
        dataset_names,
        start=1,
    ):

        # ----------------------------------------------------
        # Single-edge candidate census
        # ----------------------------------------------------

        single_df = (
            census_single_edge_gaps_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        if len(
            single_df
        ):
            single_parts.append(
                single_df
            )


        # ----------------------------------------------------
        # Existing isolated-middle census
        # ----------------------------------------------------

        isolated_df, scope = (
            census_isolated_bridges_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        if len(
            isolated_df
        ):
            isolated_parts.append(
                isolated_df
            )


        isolated_scope_rows.append(
            scope
        )


        if (
            i == 1
            or i % 5 == 0
            or i == len(
                dataset_names
            )
        ):

            print(
                f"  candidates "
                f"{i:02d}/"
                f"{len(dataset_names):02d}"
            )


    # --------------------------------------------------------
    # Concatenate candidate tables
    # --------------------------------------------------------

    if single_parts:

        gap_candidates = pd.concat(
            single_parts,
            ignore_index=True,
        )

    else:

        gap_candidates = pd.DataFrame()


    if isolated_parts:

        isolated_candidates = pd.concat(
            isolated_parts,
            ignore_index=True,
        )

    else:

        isolated_candidates = pd.DataFrame()


    isolated_scope = pd.DataFrame(
        isolated_scope_rows
    )


    assert len(
        gap_candidates
    ) > 0, (
        f"{label}: no single-edge candidates."
    )


    assert len(
        isolated_candidates
    ) > 0, (
        f"{label}: no isolated-middle candidates."
    )


    # --------------------------------------------------------
    # Frozen single-edge policy
    # --------------------------------------------------------

    single_policy = (
        build_mutual_policy_1015(
            gap_candidates,
            residual_radius_um=4.0,
        )
    )


    # --------------------------------------------------------
    # Frozen strict two-edge policy
    # --------------------------------------------------------

    strict_two_policy = (
        build_strict_two_edge_policy_1015(
            isolated_candidates,
            residual_um=4.0,
        )
    )


    return {
        "gap_candidates":
            gap_candidates,

        "single_policy":
            single_policy,

        "isolated_candidates":
            isolated_candidates,

        "isolated_scope":
            isolated_scope,

        "strict_two_policy":
            strict_two_policy,
    }


# ============================================================
# 5. Subset-aware exact V9 evaluator
# ============================================================

def evaluate_v9_subset_1023(
    dataset_names,
    pred_995_dir,
    pred_9975_dir,
    policies,
    label,
):

    records = []

    dataset_rows = []


    single_added_total = 0


    two_requested_total = 0

    two_added_total = 0

    two_edges_added_total = 0


    two_missing_total = 0

    two_degree_conflict_total = 0

    two_existing_conflict_total = 0


    cleanup_removed_total = 0


    print()
    print(
        f"Evaluating frozen V9: {label}"
    )


    for i, name in enumerate(
        dataset_names,
        start=1,
    ):

        # ----------------------------------------------------
        # Fresh G2 + K9 graph
        #
        # Uses this model's own .995 and .9975 predictions.
        # ----------------------------------------------------

        g, scale = (
            prepare_precleanup_graph_1015(
                name,
                pred_995_dir,
                pred_9975_dir,
            )
        )


        # ----------------------------------------------------
        # Frozen single-edge recovery FIRST
        # ----------------------------------------------------

        single_sub = (
            policies[
                "single_policy"
            ][
                policies[
                    "single_policy"
                ][
                    "dataset"
                ]
                == name
            ]
            .copy()
        )


        single_stats = (
            add_available_repairs_0907(
                g,
                single_sub,
            )
        )


        assert (
            int(
                single_stats[
                    "added"
                ]
            )
            ==
            len(
                single_sub
            )
        ), (
            f"{label}/{name}: "
            "single-edge policy not fully addable.\n"
            f"{single_stats}"
        )


        single_added_total += int(
            single_stats[
                "added"
            ]
        )


        # ----------------------------------------------------
        # Frozen strict two-edge recovery SECOND
        # ----------------------------------------------------

        two_sub = (
            policies[
                "strict_two_policy"
            ][
                policies[
                    "strict_two_policy"
                ][
                    "dataset"
                ]
                == name
            ]
            .copy()
        )


        two_stats = (
            add_two_edge_bridges_0912(
                g,
                two_sub,
            )
        )


        two_requested_total += int(
            two_stats[
                "requested_bridges"
            ]
        )


        two_added_total += int(
            two_stats[
                "added_bridges"
            ]
        )


        two_edges_added_total += int(
            two_stats[
                "edges_added"
            ]
        )


        two_missing_total += int(
            two_stats[
                "missing_endpoint"
            ]
        )


        two_degree_conflict_total += int(
            two_stats[
                "degree_conflict"
            ]
        )


        two_existing_conflict_total += int(
            two_stats[
                "existing_edge_conflict"
            ]
        )


        # ----------------------------------------------------
        # Frozen cleanup
        # ----------------------------------------------------

        cleanup_stats = (
            apply_frozen_component_cleanup(
                g,
                scale,
            )
        )


        cleanup_removed_total += int(
            cleanup_stats[
                "nodes_removed"
            ]
        )


        # ----------------------------------------------------
        # Locked official metric
        # ----------------------------------------------------

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )


        m = evaluate_graph_1016(
            g,
            gt_g,
            scale,
            name,
        )


        assert isinstance(
            m,
            dict,
        )


        records.append(
            m
        )


        dataset_rows.append({
            "dataset":
                name,

            "edge_jaccard":
                float(
                    m[
                        "edge_jaccard"
                    ]
                ),

            "adj_edge_jaccard":
                float(
                    m[
                        "adj_edge_jaccard"
                    ]
                ),

            "node_recall":
                float(
                    m[
                        "node_recall"
                    ]
                ),

            "total_node_ratio":
                float(
                    m[
                        "total_node_ratio"
                    ]
                ),

            "single_added":
                int(
                    single_stats[
                        "added"
                    ]
                ),

            "two_requested":
                int(
                    two_stats[
                        "requested_bridges"
                    ]
                ),

            "two_added":
                int(
                    two_stats[
                        "added_bridges"
                    ]
                ),

            "cleanup_removed":
                int(
                    cleanup_stats[
                        "nodes_removed"
                    ]
                ),
        })


        if (
            i == 1
            or i % 5 == 0
            or i == len(
                dataset_names
            )
        ):

            print(
                f"  evaluated "
                f"{i:02d}/"
                f"{len(dataset_names):02d}"
            )


    assert (
        len(
            records
        )
        ==
        len(
            dataset_names
        )
    )


    summary = summarise(
        records
    )


    return {
        "records":
            records,

        "dataset_df":
            pd.DataFrame(
                dataset_rows
            ),

        "summary":
            summary,

        "single_added":
            single_added_total,

        "two_requested":
            two_requested_total,

        "two_added":
            two_added_total,

        "two_edges_added":
            two_edges_added_total,

        "two_missing":
            two_missing_total,

        "two_degree_conflict":
            two_degree_conflict_total,

        "two_existing_conflict":
            two_existing_conflict_total,

        "cleanup_removed":
            cleanup_removed_total,
    }


# ============================================================
# 6. Regenerate + evaluate BOTH seed2 variants independently
# ============================================================

policies_1023 = {}

results_1023 = {}


for label, cfg in (
    VARIANTS_1023.items()
):

    p = build_v9_policies_subset_1023(
        datasets_44b6_1023,
        cfg[
            "pred_995"
        ],
        cfg[
            "pred_9975"
        ],
        label,
    )


    policies_1023[
        label
    ] = p


    print()
    print(
        f"{label} policy counts:"
    )


    print(
        "  single opportunities:",
        len(
            p[
                "gap_candidates"
            ]
        )
    )


    print(
        "  single policy:",
        len(
            p[
                "single_policy"
            ]
        )
    )


    print(
        "  isolated nodes:",
        int(
            p[
                "isolated_scope"
            ][
                "isolated_nodes"
            ].sum()
        )
    )


    print(
        "  isolated with pair:",
        int(
            p[
                "isolated_scope"
            ][
                "isolated_with_local_pair"
            ].sum()
        )
    )


    print(
        "  strict two policy:",
        len(
            p[
                "strict_two_policy"
            ]
        )
    )


    r = evaluate_v9_subset_1023(
        datasets_44b6_1023,
        cfg[
            "pred_995"
        ],
        cfg[
            "pred_9975"
        ],
        p,
        label,
    )


    results_1023[
        label
    ] = r


# ============================================================
# 7. Aggregate seed2 comparison
# ============================================================

overall_rows_1023 = []


for label in [
    "BASELINE_SEED2_256",
    "SPARSE9995_SEED2_256",
]:

    p = policies_1023[
        label
    ]

    r = results_1023[
        label
    ]

    s = r[
        "summary"
    ]


    overall_rows_1023.append({
        "variant":
            label,

        "single_opportunities":
            len(
                p[
                    "gap_candidates"
                ]
            ),

        "single_policy":
            len(
                p[
                    "single_policy"
                ]
            ),

        "isolated_nodes":
            int(
                p[
                    "isolated_scope"
                ][
                    "isolated_nodes"
                ].sum()
            ),

        "isolated_with_pair":
            int(
                p[
                    "isolated_scope"
                ][
                    "isolated_with_local_pair"
                ].sum()
            ),

        "strict_two_policy":
            len(
                p[
                    "strict_two_policy"
                ]
            ),

        "single_added":
            int(
                r[
                    "single_added"
                ]
            ),

        "two_added":
            int(
                r[
                    "two_added"
                ]
            ),

        "associations_added":
            int(
                r[
                    "two_edges_added"
                ]
            ),

        "cleanup_removed":
            int(
                r[
                    "cleanup_removed"
                ]
            ),

        "edge_jaccard":
            float(
                s[
                    "edge_jaccard"
                ]
            ),

        "adj_edge_jaccard":
            float(
                s[
                    "adj_edge_jaccard"
                ]
            ),

        "division_jaccard":
            float(
                s[
                    "division_jaccard"
                ]
            ),

        "node_recall":
            float(
                s[
                    "node_recall"
                ]
            ),

        "score":
            float(
                s[
                    "score"
                ]
            ),
    })


overall_1023 = pd.DataFrame(
    overall_rows_1023
)


# ============================================================
# 8. Paired per-dataset seed2 comparison
# ============================================================

base_df_1023 = (
    results_1023[
        "BASELINE_SEED2_256"
    ][
        "dataset_df"
    ]
    .copy()
)


sparse_df_1023 = (
    results_1023[
        "SPARSE9995_SEED2_256"
    ][
        "dataset_df"
    ]
    .copy()
)


paired_1023 = (
    base_df_1023[
        [
            "dataset",
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "total_node_ratio",
        ]
    ]
    .merge(
        sparse_df_1023[
            [
                "dataset",
                "edge_jaccard",
                "adj_edge_jaccard",
                "node_recall",
                "total_node_ratio",
            ]
        ],
        on="dataset",
        validate="one_to_one",
        suffixes=(
            "_baseline",
            "_sparse",
        ),
    )
)


assert (
    len(
        paired_1023
    )
    == 15
)


paired_1023[
    "delta_edge_jaccard"
] = (
    paired_1023[
        "edge_jaccard_sparse"
    ]
    -
    paired_1023[
        "edge_jaccard_baseline"
    ]
)


paired_1023[
    "delta_adj_edge_jaccard"
] = (
    paired_1023[
        "adj_edge_jaccard_sparse"
    ]
    -
    paired_1023[
        "adj_edge_jaccard_baseline"
    ]
)


paired_1023[
    "delta_node_recall"
] = (
    paired_1023[
        "node_recall_sparse"
    ]
    -
    paired_1023[
        "node_recall_baseline"
    ]
)


paired_1023[
    "delta_node_ratio"
] = (
    paired_1023[
        "total_node_ratio_sparse"
    ]
    -
    paired_1023[
        "total_node_ratio_baseline"
    ]
)


# ============================================================
# 9. Paired robustness summary
# ============================================================

paired_summary_1023 = pd.DataFrame([
    {
        "seed":
            1641,

        "datasets":
            len(
                paired_1023
            ),

        "edgeJ_sparse_better":
            int(
                (
                    paired_1023[
                        "delta_edge_jaccard"
                    ] > 0
                ).sum()
            ),

        "edgeJ_sparse_better_rate":
            float(
                (
                    paired_1023[
                        "delta_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "adjEdgeJ_sparse_better":
            int(
                (
                    paired_1023[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).sum()
            ),

        "adjEdgeJ_sparse_better_rate":
            float(
                (
                    paired_1023[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "recall_sparse_better":
            int(
                (
                    paired_1023[
                        "delta_node_recall"
                    ] > 0
                ).sum()
            ),

        "mean_delta_edgeJ":
            float(
                paired_1023[
                    "delta_edge_jaccard"
                ].mean()
            ),

        "median_delta_edgeJ":
            float(
                paired_1023[
                    "delta_edge_jaccard"
                ].median()
            ),

        "mean_delta_adjEdgeJ":
            float(
                paired_1023[
                    "delta_adj_edge_jaccard"
                ].mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                paired_1023[
                    "delta_adj_edge_jaccard"
                ].median()
            ),

        "mean_delta_recall":
            float(
                paired_1023[
                    "delta_node_recall"
                ].mean()
            ),

        "median_delta_recall":
            float(
                paired_1023[
                    "delta_node_recall"
                ].median()
            ),

        "mean_delta_node_ratio":
            float(
                paired_1023[
                    "delta_node_ratio"
                ].mean()
            ),
    }
])


# ============================================================
# 10. Aggregate deltas
# ============================================================

base_summary_1023 = (
    results_1023[
        "BASELINE_SEED2_256"
    ][
        "summary"
    ]
)


sparse_summary_1023 = (
    results_1023[
        "SPARSE9995_SEED2_256"
    ][
        "summary"
    ]
)


delta_score_1023 = (
    float(
        sparse_summary_1023[
            "score"
        ]
    )
    -
    float(
        base_summary_1023[
            "score"
        ]
    )
)


delta_edge_1023 = (
    float(
        sparse_summary_1023[
            "edge_jaccard"
        ]
    )
    -
    float(
        base_summary_1023[
            "edge_jaccard"
        ]
    )
)


delta_adj_1023 = (
    float(
        sparse_summary_1023[
            "adj_edge_jaccard"
        ]
    )
    -
    float(
        base_summary_1023[
            "adj_edge_jaccard"
        ]
    )
)


delta_div_1023 = (
    float(
        sparse_summary_1023[
            "division_jaccard"
        ]
    )
    -
    float(
        base_summary_1023[
            "division_jaccard"
        ]
    )
)


delta_recall_1023 = (
    float(
        sparse_summary_1023[
            "node_recall"
        ]
    )
    -
    float(
        base_summary_1023[
            "node_recall"
        ]
    )
)


# ============================================================
# 11. First-seed replication reference
# ============================================================

FIRST_SEED_44B6_1023 = {
    "seed":
        1640,

    "baseline_score":
        0.767588,

    "sparse9995_score":
        0.773458,

    "delta_score":
        0.005870,

    "adj_better":
        9,

    "datasets":
        15,

    "adj_better_rate":
        0.60,

    "median_delta_adjEdgeJ":
        0.002645,
}


replication_1023 = pd.DataFrame([
    {
        "seed":
            1640,

        "baseline_score":
            FIRST_SEED_44B6_1023[
                "baseline_score"
            ],

        "sparse9995_score":
            FIRST_SEED_44B6_1023[
                "sparse9995_score"
            ],

        "delta_score":
            FIRST_SEED_44B6_1023[
                "delta_score"
            ],

        "adjEdgeJ_sparse_better":
            FIRST_SEED_44B6_1023[
                "adj_better"
            ],

        "adjEdgeJ_sparse_better_rate":
            FIRST_SEED_44B6_1023[
                "adj_better_rate"
            ],

        "median_delta_adjEdgeJ":
            FIRST_SEED_44B6_1023[
                "median_delta_adjEdgeJ"
            ],
    },

    {
        "seed":
            1641,

        "baseline_score":
            float(
                base_summary_1023[
                    "score"
                ]
            ),

        "sparse9995_score":
            float(
                sparse_summary_1023[
                    "score"
                ]
            ),

        "delta_score":
            delta_score_1023,

        "adjEdgeJ_sparse_better":
            int(
                (
                    paired_1023[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).sum()
            ),

        "adjEdgeJ_sparse_better_rate":
            float(
                (
                    paired_1023[
                        "delta_adj_edge_jaccard"
                    ] > 0
                ).mean()
            ),

        "median_delta_adjEdgeJ":
            float(
                paired_1023[
                    "delta_adj_edge_jaccard"
                ].median()
            ),
    },
])


# ============================================================
# 12. Formal replication indicators
#
# These do NOT select/tune anything. They summarize whether
# the predeclared second-seed test reproduces the direction.
# ============================================================

seed2_aggregate_positive_1023 = (
    delta_score_1023
    > 0
)


seed2_median_adj_positive_1023 = (
    float(
        paired_1023[
            "delta_adj_edge_jaccard"
        ].median()
    )
    > 0
)


seed2_majority_adj_positive_1023 = (
    float(
        (
            paired_1023[
                "delta_adj_edge_jaccard"
            ] > 0
        ).mean()
    )
    > 0.5
)


# ============================================================
# 13. Result
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.23 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "SEED2 44b6 OVERALL FROZEN-v9"
)

display(
    overall_1023
)


print()
print(
    "SEED2 PAIRED ROBUSTNESS"
)

display(
    paired_summary_1023
)


print()
print(
    "CROSS-SEED REPLICATION"
)

display(
    replication_1023
)


print()
print(
    "SPARSE9995_SEED2 - BASELINE_SEED2"
)

print(
    "  delta score:",
    f"{delta_score_1023:+.6f}"
)

print(
    "  delta Edge J:",
    f"{delta_edge_1023:+.6f}"
)

print(
    "  delta Adj Edge J:",
    f"{delta_adj_1023:+.6f}"
)

print(
    "  delta Division J:",
    f"{delta_div_1023:+.6f}"
)

print(
    "  delta node recall:",
    f"{delta_recall_1023:+.6f}"
)


print()
print(
    "REPLICATION INDICATORS"
)

print(
    "  aggregate score positive:",
    seed2_aggregate_positive_1023
)

print(
    "  median paired AdjEdgeJ positive:",
    seed2_median_adj_positive_1023
)

print(
    "  majority datasets AdjEdgeJ positive:",
    seed2_majority_adj_positive_1023
)


print()
print(
    "No prediction GEFF was modified."
)

print(
    "Candidates were regenerated independently for each seed2 model."
)

print(
    "Frozen V9 geometry/order were unchanged."
)

print(
    "No threshold was tuned in this Cell."
)

print(
    "GT was used only for final 44b6 metric evaluation."
)

print()
print(
    "10.23 second-seed 44b6 frozen-v9 test: COMPLETE"
)

In [ ]:
# ============================================================
# PATCH — 10.25 Domain-Aware Sparse Detection Loss
#          SOURCE-LOCKED FINAL VERSION
#
# Purpose:
#   Add per-sample domain-aware sparse-label detection loss
#   using the EXACT source structure audited in 10.25D.
#
#   Frozen intended policy:
#
#       44b6 -> det_ignore_prob = 0.9995
#       6bba -> original baseline negative BCE
#
#   Dataflow:
#
#       VideoMeta.zarr_path
#           -> FrameWindowDataset.__getitem__()
#           -> batch["dataset_name"]
#           -> per-sample prefix mask
#           -> compute_detection_loss(
#                  ignore_sample_mask=...
#              )
#
# Backward compatibility:
#
#   ignore_prob=None
#       -> exact original baseline loss
#
#   ignore_prob=.9995, prefix=None
#       -> exact existing GLOBAL sparse-ignore
#
#   ignore_prob=.9995, prefix="44b6"
#       -> new domain-aware mixed-batch behaviour
#
# Safety:
#   - candidate source is fully AST-parsed/compiled BEFORE write
#   - post-write tests run inside rollback guard
#   - any post-write failure automatically restores backup
#
# Changes files: YES
#   scripts/train_unet_transformer_mps.py
#
# Backup:
#   scripts/train_unet_transformer_mps.py.pre_domain_aware_backup
#
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: NO
#   DELETE this PATCH Cell after PASS.
# ============================================================

from pathlib import Path

import ast
import hashlib
import os
import shutil
import subprocess
import sys

import torch
import torch.nn.functional as F
from torch.utils.data._utils.collate import default_collate


# ============================================================
# 1. Frozen paths / source identity
# ============================================================

PROJECT_1025 = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

SCRIPT_1025 = (
    PROJECT_1025
    / "scripts/train_unet_transformer_mps.py"
)

BACKUP_1025 = (
    PROJECT_1025
    / "scripts/train_unet_transformer_mps.py.pre_domain_aware_backup"
)

PYTHON_1025 = (
    PROJECT_1025
    / ".venv/bin/python"
)

LOCAL_SCRIPTS_1025 = (
    PROJECT_1025
    / "scripts"
)

OFFICIAL_SCRIPTS_1025 = (
    PROJECT_1025
    / "external/official/scripts"
)

PREDICT_SCRIPT_1025 = (
    PROJECT_1025
    / "scripts/predict_unet_transformer_mps.py"
)


EXPECTED_PREPATCH_SHA_1025 = (
    "ce4fe8b3a16771039e1c687c02e71cad2747ee48d94417f0a26c929192f1be7b"
)

EXPECTED_PREDICT_SHA_1025 = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)


# ============================================================
# 2. Helpers
# ============================================================

def sha256_1025(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()


def replace_once_1025(
    text,
    old,
    new,
    label,
):

    count = text.count(
        old
    )

    assert count == 1, (
        f"\n{label}\n"
        f"Expected exactly one match, got {count}.\n"
        "Nothing has been written to disk."
    )

    return text.replace(
        old,
        new,
        1,
    )


def top_function_1025(
    text,
    name,
):

    tree = ast.parse(
        text
    )

    nodes = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and
            node.name
            == name
        )
    ]

    assert len(
        nodes
    ) == 1, (
        f"Expected one top-level function "
        f"{name}, got {len(nodes)}."
    )

    return nodes[
        0
    ]


def function_block_1025(
    text,
    name,
):

    node = top_function_1025(
        text,
        name,
    )

    lines = text.splitlines(
        keepends=True
    )

    start = sum(
        len(line)
        for line in lines[
            :node.lineno - 1
        ]
    )

    end = sum(
        len(line)
        for line in lines[
            :node.end_lineno
        ]
    )

    return (
        start,
        end,
        text[
            start:end
        ],
    )


def replace_function_1025(
    text,
    name,
    new_block,
):

    start, end, _ = (
        function_block_1025(
            text,
            name,
        )
    )

    return (
        text[
            :start
        ]
        +
        new_block
        +
        text[
            end:
        ]
    )


def extract_function_1025(
    path,
    name,
):

    text = Path(
        path
    ).read_text(
        encoding="utf-8"
    )

    tree = ast.parse(
        text
    )

    nodes = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                ast.FunctionDef,
            )
            and
            node.name
            == name
        )
    ]

    assert len(
        nodes
    ) == 1

    module = ast.Module(
        body=[
            nodes[
                0
            ]
        ],
        type_ignores=[],
    )

    ast.fix_missing_locations(
        module
    )

    namespace = {
        "torch":
            torch,

        "F":
            F,
    }

    exec(
        compile(
            module,
            str(
                path
            ),
            "exec",
        ),
        namespace,
    )

    return namespace[
        name
    ]


def named_calls_1025(
    text,
    parent_name,
    called_name,
):

    parent = top_function_1025(
        text,
        parent_name,
    )

    calls = [
        node
        for node in ast.walk(
            parent
        )
        if (
            isinstance(
                node,
                ast.Call,
            )
            and
            isinstance(
                node.func,
                ast.Name,
            )
            and
            node.func.id
            == called_name
        )
    ]

    return calls


# ============================================================
# 3. Disk-state safety gate
# ============================================================

assert SCRIPT_1025.exists()
assert BACKUP_1025.exists()
assert PYTHON_1025.exists()
assert PREDICT_SCRIPT_1025.exists()


assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON_1025.resolve()
), (
    "Notebook is not using the Biohub project .venv."
)


observed_source_sha_1025 = (
    sha256_1025(
        SCRIPT_1025
    )
)


assert (
    observed_source_sha_1025
    ==
    EXPECTED_PREPATCH_SHA_1025
), (
    "\nTraining source is not at the expected "
    "pre-domain-aware SHA.\n"
    "\n"
    f"Observed:\n"
    f"  {observed_source_sha_1025}\n"
    "\n"
    f"Expected:\n"
    f"  {EXPECTED_PREPATCH_SHA_1025}\n"
)


assert (
    sha256_1025(
        BACKUP_1025
    )
    ==
    EXPECTED_PREPATCH_SHA_1025
)


assert (
    sha256_1025(
        PREDICT_SCRIPT_1025
    )
    ==
    EXPECTED_PREDICT_SHA_1025
)


print(
    "Pre-patch training-script SHA:",
    "PASS"
)

print(
    "Backup SHA:",
    "PASS"
)

print(
    "Frozen predictor SHA:",
    "PASS"
)


# ============================================================
# 4. Load clean source
# ============================================================

source_1025 = (
    SCRIPT_1025.read_text(
        encoding="utf-8"
    )
)


assert (
    "det_ignore_prefix"
    not in source_1025
)

assert (
    "ignore_sample_mask"
    not in source_1025
)

assert (
    '"dataset_name": vm.zarr_path.stem'
    not in source_1025
)


# ============================================================
# 5. PATCH A
#    Dataset identity -> DataLoader batch
# ============================================================

class_start_1025 = (
    source_1025.index(
        "class FrameWindowDataset"
    )
)

class_end_1025 = (
    source_1025.index(
        "\ndef load_dataset_windows",
        class_start_1025,
    )
)


class_block_1025 = (
    source_1025[
        class_start_1025:
        class_end_1025
    ]
)


class_block_1025 = (
    replace_once_1025(
        class_block_1025,

        (
            "    def __getitem__(self, idx: int) "
            "-> dict[str, torch.Tensor]:"
        ),

        (
            "    def __getitem__(self, idx: int) "
            "-> dict[str, torch.Tensor | str]:"
        ),

        "PATCH A1 — __getitem__ annotation",
    )
)


class_block_1025 = (
    replace_once_1025(
        class_block_1025,

        (
            '            "downsample": '
            'torch.tensor(vm.downsample, dtype=torch.float32),\n'
        ),

        (
            '            "downsample": '
            'torch.tensor(vm.downsample, dtype=torch.float32),\n'
            '            "dataset_name": vm.zarr_path.stem,\n'
        ),

        "PATCH A2 — dataset_name return",
    )
)


source_1025 = (
    source_1025[
        :class_start_1025
    ]
    +
    class_block_1025
    +
    source_1025[
        class_end_1025:
    ]
)


print(
    "PATCH A — dataset identity:",
    "PASS"
)


# ============================================================
# 6. PATCH B
#    Detection loss -> per-sample sparse-ignore activation
# ============================================================

_, _, det_block_1025 = (
    function_block_1025(
        source_1025,
        "compute_detection_loss",
    )
)


# ------------------------------------------------------------
# 6A. Signature
# ------------------------------------------------------------

det_block_1025 = (
    replace_once_1025(
        det_block_1025,

        (
            "    ignore_prob: float | None = None,\n"
            ") -> torch.Tensor:"
        ),

        (
            "    ignore_prob: float | None = None,\n"
            "    ignore_sample_mask: torch.Tensor | None = None,\n"
            ") -> torch.Tensor:"
        ),

        "PATCH B1 — loss signature",
    )
)


# ------------------------------------------------------------
# 6B. Exact existing ignore branch
# ------------------------------------------------------------

old_ignore_branch_1025 = """\
    else:
        if not (0.0 < ignore_prob <= 1.0):
            raise ValueError(
                f"ignore_prob must be in (0, 1], got {ignore_prob}"
            )

        # The supervision mask is a training policy, not a differentiable
        # function of the logits.
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            ignored_unlabeled = (
                (~positive)
                & (probs >= ignore_prob)
            )
            supervised_negative = (
                (~positive)
                & (~ignored_unlabeled)
            )
"""


new_ignore_branch_1025 = """\
    else:
        if not (0.0 < ignore_prob <= 1.0):
            raise ValueError(
                f"ignore_prob must be in (0, 1], got {ignore_prob}"
            )

        # None reproduces the existing GLOBAL sparse-ignore policy.
        if ignore_sample_mask is None:
            active_ignore = torch.ones(
                B,
                dtype=torch.bool,
                device=logits.device,
            )
        else:
            active_ignore = ignore_sample_mask.to(
                device=logits.device,
                dtype=torch.bool,
            )

            if (
                active_ignore.ndim != 1
                or active_ignore.shape[0] != B
            ):
                raise ValueError(
                    "ignore_sample_mask must have shape "
                    f"(B,), got {tuple(active_ignore.shape)} "
                    f"for B={B}"
                )

        active_ignore = active_ignore.reshape(
            (B,) + (1,) * len(spatial)
        )

        # The supervision mask is a training policy, not a differentiable
        # function of the logits.
        with torch.no_grad():
            probs = torch.sigmoid(logits)

            ignored_unlabeled = (
                (~positive)
                & active_ignore
                & (probs >= ignore_prob)
            )

            supervised_negative = (
                (~positive)
                & (~ignored_unlabeled)
            )
"""


det_block_1025 = (
    replace_once_1025(
        det_block_1025,
        old_ignore_branch_1025,
        new_ignore_branch_1025,
        "PATCH B2 — per-sample ignore branch",
    )
)


source_1025 = (
    replace_function_1025(
        source_1025,
        "compute_detection_loss",
        det_block_1025,
    )
)


print(
    "PATCH B — per-sample loss:",
    "PASS"
)


# ============================================================
# 7. PATCH C
#    train_epoch -> construct domain mask
# ============================================================

_, _, epoch_block_1025 = (
    function_block_1025(
        source_1025,
        "train_epoch",
    )
)


# ------------------------------------------------------------
# 7A. Signature
# ------------------------------------------------------------

epoch_block_1025 = (
    replace_once_1025(
        epoch_block_1025,

        (
            "    det_ignore_prob: float | None = None,\n"
            "    max_iters: int | None = None,\n"
        ),

        (
            "    det_ignore_prob: float | None = None,\n"
            "    det_ignore_prefix: str | None = None,\n"
            "    max_iters: int | None = None,\n"
        ),

        "PATCH C1 — train_epoch signature",
    )
)


# ------------------------------------------------------------
# 7B. Per-sample routing immediately after B,W
# ------------------------------------------------------------

epoch_block_1025 = (
    replace_once_1025(
        epoch_block_1025,

        (
            "        B, W = imgs.shape[:2]\n"
            "\n"
            "        # --- 1. Encode:"
        ),

        (
            "        B, W = imgs.shape[:2]\n"
            "\n"
            "        if (\n"
            "            det_ignore_prefix is not None\n"
            "            and det_ignore_prob is None\n"
            "        ):\n"
            "            raise ValueError(\n"
            "                \"det_ignore_prefix requires det_ignore_prob\"\n"
            "            )\n"
            "\n"
            "        # Domain-aware sparse-ignore is per sample, not per batch.\n"
            "        ignore_sample_mask = None\n"
            "\n"
            "        if (\n"
            "            det_ignore_prob is not None\n"
            "            and det_ignore_prefix is not None\n"
            "        ):\n"
            "            dataset_names = batch.get(\"dataset_name\")\n"
            "\n"
            "            if dataset_names is None:\n"
            "                raise KeyError(\n"
            "                    \"dataset_name missing from training batch while \"\n"
            "                    \"det_ignore_prefix is enabled\"\n"
            "                )\n"
            "\n"
            "            if len(dataset_names) != B:\n"
            "                raise ValueError(\n"
            "                    \"dataset_name batch length mismatch: \"\n"
            "                    f\"{len(dataset_names)} != {B}\"\n"
            "                )\n"
            "\n"
            "            ignore_sample_mask = torch.tensor(\n"
            "                [\n"
            "                    str(name).startswith(det_ignore_prefix)\n"
            "                    for name in dataset_names\n"
            "                ],\n"
            "                dtype=torch.bool,\n"
            "                device=device,\n"
            "            )\n"
            "\n"
            "        # --- 1. Encode:"
        ),

        "PATCH C2 — domain mask construction",
    )
)


# ------------------------------------------------------------
# 7C. Exact current detection-loss call
# ------------------------------------------------------------

epoch_block_1025 = (
    replace_once_1025(
        epoch_block_1025,

        (
            "            compute_detection_loss(\n"
            "                det_logits[i],\n"
            "                coords[:, i],\n"
            "                masks[:, i],\n"
            "                det_neg_weight,\n"
            "                det_ignore_prob,\n"
            "            )"
        ),

        (
            "            compute_detection_loss(\n"
            "                det_logits[i],\n"
            "                coords[:, i],\n"
            "                masks[:, i],\n"
            "                neg_weight=det_neg_weight,\n"
            "                ignore_prob=det_ignore_prob,\n"
            "                ignore_sample_mask=ignore_sample_mask,\n"
            "            )"
        ),

        "PATCH C3 — loss call propagation",
    )
)


source_1025 = (
    replace_function_1025(
        source_1025,
        "train_epoch",
        epoch_block_1025,
    )
)


print(
    "PATCH C — train_epoch routing:",
    "PASS"
)


# ============================================================
# 8. PATCH D
#    train() -> train_epoch()
#
# Exact structure from 10.25D:
#
#   train_epoch(
#       model,
#       train_loader,
#       optimizer,
#       device,
#       det_loss_weight,
#       det_neg_weight,
#       det_ignore_prob,
#       max_iters=max_iters,
#       pool_kernel_um=pool_kernel_um,
#   )
# ============================================================

_, _, train_block_1025 = (
    function_block_1025(
        source_1025,
        "train",
    )
)


# ------------------------------------------------------------
# 8A. train() signature
# ------------------------------------------------------------

train_block_1025 = (
    replace_once_1025(
        train_block_1025,

        (
            "    det_ignore_prob: float | None = None,\n"
            "    max_iters: int | None = None,\n"
        ),

        (
            "    det_ignore_prob: float | None = None,\n"
            "    det_ignore_prefix: str | None = None,\n"
            "    max_iters: int | None = None,\n"
        ),

        "PATCH D1 — train signature",
    )
)


# ------------------------------------------------------------
# 8B. Exact audited train_epoch call
# ------------------------------------------------------------

old_train_epoch_call_1025 = """\
train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            det_loss_weight,
            det_neg_weight,
            det_ignore_prob,
            max_iters=max_iters,
            pool_kernel_um=pool_kernel_um,
        )"""


new_train_epoch_call_1025 = """\
train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            det_loss_weight,
            det_neg_weight,
            det_ignore_prob,
            det_ignore_prefix=det_ignore_prefix,
            max_iters=max_iters,
            pool_kernel_um=pool_kernel_um,
        )"""


train_block_1025 = (
    replace_once_1025(
        train_block_1025,
        old_train_epoch_call_1025,
        new_train_epoch_call_1025,
        "PATCH D2 — exact train -> train_epoch call",
    )
)


source_1025 = (
    replace_function_1025(
        source_1025,
        "train",
        train_block_1025,
    )
)


print(
    "PATCH D — exact train propagation:",
    "PASS"
)


# ============================================================
# 9. PATCH E
#    CLI + main() -> train()
#
# Both structures are exact outputs from 10.25D.
# ============================================================

_, _, main_block_1025 = (
    function_block_1025(
        source_1025,
        "main",
    )
)


# ------------------------------------------------------------
# 9A. Exact existing CLI block
# ------------------------------------------------------------

old_cli_1025 = """\
    parser.add_argument(
        "--det-ignore-prob",
        type=float,
        default=None,
        help=(
            "Optional sparse-label mode: non-GT voxels with "
            "sigmoid probability >= this threshold are treated as "
            "unlabeled and excluded from negative detection BCE. "
            "Default: disabled."
        ),
    )
    parser.add_argument("--max-iters", type=int, default=None,
                        help="Max training iterations per epoch. None = full epoch.")
"""


new_cli_1025 = """\
    parser.add_argument(
        "--det-ignore-prob",
        type=float,
        default=None,
        help=(
            "Optional sparse-label mode: non-GT voxels with "
            "sigmoid probability >= this threshold are treated as "
            "unlabeled and excluded from negative detection BCE. "
            "Default: disabled."
        ),
    )
    parser.add_argument(
        "--det-ignore-prefix",
        type=str,
        default=None,
        help=(
            "Apply --det-ignore-prob only to samples whose dataset "
            "stem starts with this prefix. None preserves global "
            "sparse-ignore behaviour."
        ),
    )
    parser.add_argument("--max-iters", type=int, default=None,
                        help="Max training iterations per epoch. None = full epoch.")
"""


main_block_1025 = (
    replace_once_1025(
        main_block_1025,
        old_cli_1025,
        new_cli_1025,
        "PATCH E1 — exact CLI block",
    )
)


# ------------------------------------------------------------
# 9B. Exact main() -> train() keywords
# ------------------------------------------------------------

main_block_1025 = (
    replace_once_1025(
        main_block_1025,

        (
            "            det_ignore_prob=args.det_ignore_prob,\n"
            "            max_iters=args.max_iters,\n"
        ),

        (
            "            det_ignore_prob=args.det_ignore_prob,\n"
            "            det_ignore_prefix=args.det_ignore_prefix,\n"
            "            max_iters=args.max_iters,\n"
        ),

        "PATCH E2 — exact main -> train propagation",
    )
)


source_1025 = (
    replace_function_1025(
        source_1025,
        "main",
        main_block_1025,
    )
)


print(
    "PATCH E — CLI/main propagation:",
    "PASS"
)


# ============================================================
# 10. PRE-WRITE STATIC VALIDATION
#
# Nothing has touched the training source on disk yet.
# ============================================================

candidate_tree_1025 = ast.parse(
    source_1025
)


# ------------------------------------------------------------
# 10A. Signatures
# ------------------------------------------------------------

candidate_functions_1025 = {
    node.name:
        node

    for node in candidate_tree_1025.body

    if isinstance(
        node,
        ast.FunctionDef,
    )
}


det_args_1025 = [
    arg.arg
    for arg in candidate_functions_1025[
        "compute_detection_loss"
    ].args.args
]


epoch_args_1025 = [
    arg.arg
    for arg in candidate_functions_1025[
        "train_epoch"
    ].args.args
]


train_args_1025 = [
    arg.arg
    for arg in candidate_functions_1025[
        "train"
    ].args.args
]


assert (
    "ignore_sample_mask"
    in det_args_1025
)

assert (
    "det_ignore_prefix"
    in epoch_args_1025
)

assert (
    "det_ignore_prefix"
    in train_args_1025
)


# ------------------------------------------------------------
# 10B. train() -> train_epoch() AST contract
# ------------------------------------------------------------

train_epoch_calls_1025 = (
    named_calls_1025(
        source_1025,
        "train",
        "train_epoch",
    )
)


assert len(
    train_epoch_calls_1025
) == 1


train_epoch_call_1025 = (
    train_epoch_calls_1025[
        0
    ]
)


# Original seven positional args remain unchanged.
assert len(
    train_epoch_call_1025.args
) == 7


assert (
    ast.get_source_segment(
        source_1025,
        train_epoch_call_1025.args[
            6
        ],
    )
    ==
    "det_ignore_prob"
)


train_epoch_keyword_names_1025 = [
    kw.arg
    for kw in train_epoch_call_1025.keywords
]


assert (
    train_epoch_keyword_names_1025
    ==
    [
        "det_ignore_prefix",
        "max_iters",
        "pool_kernel_um",
    ]
)


# ------------------------------------------------------------
# 10C. main() -> train() AST contract
# ------------------------------------------------------------

main_train_calls_1025 = (
    named_calls_1025(
        source_1025,
        "main",
        "train",
    )
)


assert len(
    main_train_calls_1025
) == 1


main_train_keywords_1025 = [
    kw.arg
    for kw in main_train_calls_1025[
        0
    ].keywords
]


assert (
    "det_ignore_prob"
    in main_train_keywords_1025
)

assert (
    "det_ignore_prefix"
    in main_train_keywords_1025
)

assert (
    main_train_keywords_1025.index(
        "det_ignore_prefix"
    )
    ==
    main_train_keywords_1025.index(
        "det_ignore_prob"
    )
    + 1
)


# ------------------------------------------------------------
# 10D. Required source tokens
# ------------------------------------------------------------

required_tokens_1025 = [
    '"dataset_name": vm.zarr_path.stem',

    "ignore_sample_mask: torch.Tensor | None = None",

    "det_ignore_prefix: str | None = None",

    "str(name).startswith(det_ignore_prefix)",

    "ignore_sample_mask=ignore_sample_mask",

    "det_ignore_prefix=det_ignore_prefix",

    "--det-ignore-prefix",

    "det_ignore_prefix=args.det_ignore_prefix",

    "det_ignore_prefix requires det_ignore_prob",
]


for token_1025 in (
    required_tokens_1025
):

    assert (
        token_1025
        in source_1025
    ), (
        f"Missing candidate token:\n"
        f"{token_1025}"
    )


# ------------------------------------------------------------
# 10E. Compile entire candidate source
# ------------------------------------------------------------

compile(
    source_1025,
    str(
        SCRIPT_1025
    ),
    "exec",
)


# Disk MUST still be untouched here.
assert (
    sha256_1025(
        SCRIPT_1025
    )
    ==
    EXPECTED_PREPATCH_SHA_1025
)


print()
print(
    "STATIC VALIDATION BEFORE WRITE:",
    "PASS"
)


# ============================================================
# 11. Write + automatic rollback guard
# ============================================================

patched_sha_1025 = None


try:

    # --------------------------------------------------------
    # 11A. First and only source write
    # --------------------------------------------------------

    SCRIPT_1025.write_text(
        source_1025,
        encoding="utf-8",
    )


    patched_sha_1025 = (
        sha256_1025(
            SCRIPT_1025
        )
    )


    assert (
        patched_sha_1025
        !=
        EXPECTED_PREPATCH_SHA_1025
    )


    print(
        "Patched source write:",
        "PASS"
    )

    print(
        "Patched SHA256:",
        patched_sha_1025
    )


    # ========================================================
    # 12. Disk parse / compile
    # ========================================================

    disk_source_1025 = (
        SCRIPT_1025.read_text(
            encoding="utf-8"
        )
    )


    assert (
        disk_source_1025
        ==
        source_1025
    )


    ast.parse(
        disk_source_1025
    )


    compile(
        disk_source_1025,
        str(
            SCRIPT_1025
        ),
        "exec",
    )


    print(
        "AST / compile from disk:",
        "PASS"
    )


    # ========================================================
    # 13. PyTorch mixed-domain collation smoke
    # ========================================================

    collated_1025 = default_collate([
        {
            "dataset_name":
                "44b6_example_a",

            "x":
                torch.tensor(
                    1
                ),
        },

        {
            "dataset_name":
                "6bba_example_b",

            "x":
                torch.tensor(
                    2
                ),
        },
    ])


    names_1025 = (
        collated_1025[
            "dataset_name"
        ]
    )


    assert list(
        names_1025
    ) == [
        "44b6_example_a",
        "6bba_example_b",
    ]


    routing_mask_1025 = torch.tensor([
        str(
            name
        ).startswith(
            "44b6"
        )

        for name in names_1025
    ])


    assert (
        routing_mask_1025.tolist()
        ==
        [
            True,
            False,
        ]
    )


    print(
        "Mixed-domain DataLoader collation:",
        "PASS"
    )


    # ========================================================
    # 14. Load old/new detection loss only
    # ========================================================

    old_loss_1025 = (
        extract_function_1025(
            BACKUP_1025,
            "compute_detection_loss",
        )
    )


    new_loss_1025 = (
        extract_function_1025(
            SCRIPT_1025,
            "compute_detection_loss",
        )
    )


    # ========================================================
    # 15. Synthetic regression batch
    # ========================================================

    # B=2, spatial=(1,1,3)
    #
    # x=0: GT positive
    # x=1: p≈.99995 non-GT
    # x=2: ordinary background

    logits_base_1025 = torch.tensor(
        [
            [
                [
                    [
                        [
                            4.0,
                            10.0,
                            -2.0,
                        ]
                    ]
                ]
            ],

            [
                [
                    [
                        [
                            4.0,
                            10.0,
                            -2.0,
                        ]
                    ]
                ]
            ],
        ],
        dtype=torch.float32,
    )


    coords_1025 = torch.tensor(
        [
            [
                [
                    0.0,
                    0.0,
                    0.0,
                ]
            ],

            [
                [
                    0.0,
                    0.0,
                    0.0,
                ]
            ],
        ],
        dtype=torch.float32,
    )


    gt_mask_1025 = torch.tensor(
        [
            [
                True
            ],

            [
                True
            ],
        ],
        dtype=torch.bool,
    )


    # ========================================================
    # 16. Baseline must reproduce exactly
    # ========================================================

    old_baseline_1025 = old_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=None,
    )


    new_baseline_1025 = new_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=None,
        ignore_sample_mask=None,
    )


    torch.testing.assert_close(
        old_baseline_1025,
        new_baseline_1025,
        rtol=0.0,
        atol=0.0,
    )


    print(
        "Baseline no-ignore reproduction:",
        "EXACT PASS"
    )


    # ========================================================
    # 17. Existing GLOBAL sparse must reproduce exactly
    # ========================================================

    old_global_1025 = old_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=0.9995,
    )


    new_global_1025 = new_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=0.9995,
        ignore_sample_mask=None,
    )


    torch.testing.assert_close(
        old_global_1025,
        new_global_1025,
        rtol=0.0,
        atol=0.0,
    )


    print(
        "Global sparse .9995 reproduction:",
        "EXACT PASS"
    )


    # ========================================================
    # 18. all-False mask == baseline exactly
    # ========================================================

    all_false_1025 = new_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=0.9995,
        ignore_sample_mask=torch.tensor(
            [
                False,
                False,
            ]
        ),
    )


    torch.testing.assert_close(
        all_false_1025,
        new_baseline_1025,
        rtol=0.0,
        atol=0.0,
    )


    print(
        "All-False routing -> baseline:",
        "EXACT PASS"
    )


    # ========================================================
    # 19. all-True mask == global sparse exactly
    # ========================================================

    all_true_1025 = new_loss_1025(
        logits_base_1025.clone(),
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=0.9995,
        ignore_sample_mask=torch.tensor(
            [
                True,
                True,
            ]
        ),
    )


    torch.testing.assert_close(
        all_true_1025,
        new_global_1025,
        rtol=0.0,
        atol=0.0,
    )


    print(
        "All-True routing -> global sparse:",
        "EXACT PASS"
    )


    # ========================================================
    # 20. Mixed-domain gradient semantics
    # ========================================================

    mixed_logits_1025 = (
        logits_base_1025
        .clone()
        .requires_grad_(
            True
        )
    )


    mixed_loss_1025 = new_loss_1025(
        mixed_logits_1025,
        coords_1025,
        gt_mask_1025,
        neg_weight=0.01,
        ignore_prob=0.9995,
        ignore_sample_mask=torch.tensor(
            [
                True,   # 44b6
                False,  # 6bba
            ]
        ),
    )


    mixed_loss_1025.backward()


    high_conf_grad_1025 = (
        mixed_logits_1025.grad[
            :,
            0,
            0,
            0,
            1,
        ]
        .detach()
        .cpu()
    )


    grad_44b6_1025 = float(
        high_conf_grad_1025[
            0
        ]
    )


    grad_6bba_1025 = float(
        high_conf_grad_1025[
            1
        ]
    )


    assert abs(
        grad_44b6_1025
    ) < 1e-12


    assert (
        grad_6bba_1025
        > 0.0
    )


    print(
        "Mixed-domain gradient semantics:",
        "PASS"
    )

    print(
        "  44b6 high-conf gradient:",
        grad_44b6_1025
    )

    print(
        "  6bba high-conf gradient:",
        grad_6bba_1025
    )


    # ========================================================
    # 21. Invalid sample mask fails closed
    # ========================================================

    invalid_shape_pass_1025 = False


    try:

        new_loss_1025(
            logits_base_1025.clone(),
            coords_1025,
            gt_mask_1025,
            neg_weight=0.01,
            ignore_prob=0.9995,
            ignore_sample_mask=torch.tensor(
                [
                    True
                ]
            ),
        )


    except ValueError:

        invalid_shape_pass_1025 = True


    assert (
        invalid_shape_pass_1025
    )


    print(
        "Invalid mask shape guard:",
        "PASS"
    )


    # ========================================================
    # 22. CLI/runtime environment
    # ========================================================

    env_1025 = os.environ.copy()


    pythonpath_parts_1025 = [
        str(
            LOCAL_SCRIPTS_1025
        ),

        str(
            OFFICIAL_SCRIPTS_1025
        ),
    ]


    if env_1025.get(
        "PYTHONPATH"
    ):

        pythonpath_parts_1025.append(
            env_1025[
                "PYTHONPATH"
            ]
        )


    env_1025[
        "PYTHONPATH"
    ] = os.pathsep.join(
        pythonpath_parts_1025
    )


    # ========================================================
    # 23. CLI smoke
    # ========================================================

    help_proc_1025 = subprocess.run(
        [
            str(
                PYTHON_1025
            ),

            str(
                SCRIPT_1025
            ),

            "--help",
        ],
        cwd=PROJECT_1025,
        env=env_1025,
        capture_output=True,
        text=True,
    )


    assert (
        help_proc_1025.returncode
        == 0
    ), (
        help_proc_1025.stderr
    )


    help_text_1025 = (
        help_proc_1025.stdout
        +
        help_proc_1025.stderr
    )


    assert (
        "--det-ignore-prob"
        in help_text_1025
    )

    assert (
        "--det-ignore-prefix"
        in help_text_1025
    )


    print(
        "CLI --det-ignore-prefix:",
        "PASS"
    )


    # ========================================================
    # 24. Runtime import/signature smoke
    # ========================================================

    runtime_code_1025 = r'''
import inspect
import train_unet_transformer_mps as m

print(
    inspect.signature(
        m.compute_detection_loss
    )
)

print(
    inspect.signature(
        m.train_epoch
    )
)

print(
    inspect.signature(
        m.train
    )
)
'''


    runtime_proc_1025 = subprocess.run(
        [
            str(
                PYTHON_1025
            ),

            "-c",

            runtime_code_1025,
        ],
        cwd=PROJECT_1025,
        env=env_1025,
        capture_output=True,
        text=True,
    )


    assert (
        runtime_proc_1025.returncode
        == 0
    ), (
        runtime_proc_1025.stderr
    )


    runtime_text_1025 = (
        runtime_proc_1025.stdout
    )


    assert (
        "ignore_sample_mask"
        in runtime_text_1025
    )

    assert (
        runtime_text_1025.count(
            "det_ignore_prefix"
        )
        >= 2
    )


    print(
        "Runtime import/signatures:",
        "PASS"
    )


    # ========================================================
    # 25. Final integrity
    # ========================================================

    assert (
        sha256_1025(
            SCRIPT_1025
        )
        ==
        patched_sha_1025
    )


    assert (
        sha256_1025(
            BACKUP_1025
        )
        ==
        EXPECTED_PREPATCH_SHA_1025
    )


    assert (
        sha256_1025(
            PREDICT_SCRIPT_1025
        )
        ==
        EXPECTED_PREDICT_SHA_1025
    )


    print(
        "Final training-source integrity:",
        "PASS"
    )

    print(
        "Backup untouched:",
        "PASS"
    )

    print(
        "Predictor untouched:",
        "PASS"
    )


# ============================================================
# 26. AUTOMATIC ROLLBACK ON ANY POST-WRITE FAILURE
# ============================================================

except Exception:

    print()
    print(
        "=" * 72
    )

    print(
        "10.25 POST-WRITE VALIDATION FAILED"
    )

    print(
        "Rolling training source back to frozen backup..."
    )


    shutil.copy2(
        BACKUP_1025,
        SCRIPT_1025,
    )


    restored_sha_1025 = (
        sha256_1025(
            SCRIPT_1025
        )
    )


    assert (
        restored_sha_1025
        ==
        EXPECTED_PREPATCH_SHA_1025
    )


    print(
        "ROLLBACK:",
        "PASS"
    )

    print(
        "Restored SHA:",
        restored_sha_1025
    )


    raise


# ============================================================
# 27. RESULT
# ============================================================

print()
print(
    "=" * 72
)

print(
    "10.25 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "SOURCE"
)

print(
    "  pre-patch SHA:"
)

print(
    " ",
    EXPECTED_PREPATCH_SHA_1025
)

print(
    "  patched SHA:"
)

print(
    " ",
    patched_sha_1025
)

print(
    "  backup SHA:"
)

print(
    " ",
    sha256_1025(
        BACKUP_1025
    )
)


print()
print(
    "DATAFLOW"
)

print(
    "  VideoMeta.zarr_path -> dataset_name:",
    "PASS"
)

print(
    "  dataset_name survives mixed DataLoader collation:",
    "PASS"
)

print(
    "  per-sample prefix routing:",
    "PASS"
)


print()
print(
    "LOSS COMPATIBILITY"
)

print(
    "  baseline ignore_prob=None:",
    "EXACT PASS"
)

print(
    "  global sparse .9995:",
    "EXACT PASS"
)

print(
    "  all-False routing -> baseline:",
    "EXACT PASS"
)

print(
    "  all-True routing -> global sparse:",
    "EXACT PASS"
)


print()
print(
    "DOMAIN-AWARE SEMANTICS"
)

print(
    "  mixed 44b6/6bba gradient:",
    "PASS"
)

print(
    "  44b6 high-conf gradient:",
    grad_44b6_1025
)

print(
    "  6bba high-conf gradient:",
    grad_6bba_1025
)

print(
    "  invalid mask shape:",
    "FAIL-CLOSED PASS"
)


print()
print(
    "PROPAGATION"
)

print(
    "  train -> train_epoch:",
    "PASS"
)

print(
    "  main -> train:",
    "PASS"
)

print(
    "  CLI:",
    "PASS"
)

print(
    "  runtime signatures:",
    "PASS"
)


print()
print(
    "TARGET POLICY"
)

print(
    "  --det-ignore-prob 0.9995"
)

print(
    "  --det-ignore-prefix 44b6"
)


print()
print(
    "Training:",
    "NOT RUN"
)

print(
    "Inference:",
    "NOT RUN"
)

print(
    "Predictor:",
    "UNCHANGED"
)


print()
print(
    "10.25 domain-aware sparse-loss patch:",
    "COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.26 Domain-Aware Loss End-to-End Integration Smoke
#
# Purpose:
#   Verify the newly patched domain-aware sparse-label path on
#   REAL Biohub training data through the complete CLI pipeline.
#
#   Smoke data:
#       1 Fold0 TRAIN video from 44b6
#       1 Fold0 TRAIN video from 6bba
#
#   Policy:
#       44b6 -> det_ignore_prob=.9995
#       6bba -> baseline nominal-negative BCE
#
#   This is NOT a quality/model-selection experiment.
#
#   Frozen:
#       Warm start      = Warm2
#       batch size      = 8
#       LR              = 1e-5
#       det loss weight = 1.0
#       det neg weight  = .01
#       downsample      = 1,4,4
#       window size     = 2
#       pool kernel     = 5.0 um
#       max frames      = 12
#       optimizer iters = 12
#       seed            = 1642
#
# Changes files: YES
#   - temporary smoke split manifest
#   - smoke training log
#   - smoke checkpoint/config
#
# Runs training: YES — ONLY 12 optimizer iterations
# Runs inference: NO
# Uses GT: YES — normal training supervision only
# Keep after running: NO
#   Delete this RUN Cell after PASS; keep concise RESULT Markdown.
# ============================================================

from pathlib import Path

import copy
import hashlib
import json
import os
import re
import subprocess
import sys
import time

import torch
from torch.utils.data import DataLoader


# ============================================================
# 1. Paths
# ============================================================

PROJECT_1026 = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

PYTHON_1026 = (
    PROJECT_1026
    / ".venv/bin/python"
)

TRAIN_SCRIPT_1026 = (
    PROJECT_1026
    / "scripts/train_unet_transformer_mps.py"
)

TRAIN_ROOT_1026 = (
    PROJECT_1026
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

ORIGINAL_SPLITS_1026 = (
    PROJECT_1026
    / "configs/dataset_splits.json"
)

WARM2_1026 = (
    PROJECT_1026
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

LOCAL_SCRIPTS_1026 = (
    PROJECT_1026
    / "scripts"
)

OFFICIAL_SCRIPTS_1026 = (
    PROJECT_1026
    / "external/official/scripts"
)

REPORT_DIR_1026 = (
    PROJECT_1026
    / "reports/stage10_domain_smoke"
)

REPORT_DIR_1026.mkdir(
    parents=True,
    exist_ok=True,
)

SMOKE_SPLITS_1026 = (
    REPORT_DIR_1026
    / "mixed_domain_smoke_split.json"
)

LOG_1026 = (
    REPORT_DIR_1026
    / "domain_aware_12iter.log"
)

METHOD_1026 = (
    "stage10_domain_aware_smoke_12iter"
)

CHECKPOINT_1026 = (
    PROJECT_1026
    / "models/official_baseline"
    / METHOD_1026
    / "split_0/edge_predictor_best.pth"
)


# ============================================================
# 2. Frozen source identities
# ============================================================

EXPECTED_TRAIN_SHA_1026 = (
    "c6cd92b3562d8a8ad42d40944a545122ca23b42e85eacd04690abae8ee00cbc1"
)

EXPECTED_WARM2_SHA_1026 = (
    "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"
)


def sha256_1026(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


assert (
    Path(
        sys.executable
    ).resolve()
    ==
    PYTHON_1026.resolve()
)

assert (
    sha256_1026(
        TRAIN_SCRIPT_1026
    )
    ==
    EXPECTED_TRAIN_SHA_1026
)

assert (
    sha256_1026(
        WARM2_1026
    )
    ==
    EXPECTED_WARM2_SHA_1026
)


print(
    "Training-script SHA:",
    "PASS"
)

print(
    "Warm2 SHA:",
    "PASS"
)

print(
    "Python:",
    PYTHON_1026
)


# ============================================================
# 3. Read frozen Fold0 manifest
# ============================================================

with ORIGINAL_SPLITS_1026.open(
    "r",
    encoding="utf-8",
) as f:

    original_payload_1026 = (
        json.load(
            f
        )
    )


if (
    isinstance(
        original_payload_1026,
        dict,
    )
    and
    "folds"
    in original_payload_1026
):

    original_folds_1026 = (
        original_payload_1026[
            "folds"
        ]
    )

else:

    original_folds_1026 = (
        original_payload_1026
    )


assert isinstance(
    original_folds_1026,
    list,
)

assert len(
    original_folds_1026
) >= 1


fold0_train_1026 = list(
    original_folds_1026[
        0
    ][
        "train"
    ]
)


assert len(
    fold0_train_1026
) == 159


# ============================================================
# 4. Pick ONE real Fold0-train video from each domain
#
# No validation/test dataset is selected here.
# ============================================================

train_44b6_1026 = [
    name
    for name in fold0_train_1026
    if str(
        name
    ).startswith(
        "44b6"
    )
]


train_6bba_1026 = [
    name
    for name in fold0_train_1026
    if str(
        name
    ).startswith(
        "6bba"
    )
]


assert train_44b6_1026
assert train_6bba_1026


VIDEO_44B6_1026 = (
    sorted(
        train_44b6_1026
    )[
        0
    ]
)

VIDEO_6BBA_1026 = (
    sorted(
        train_6bba_1026
    )[
        0
    ]
)


SMOKE_VIDEOS_1026 = [
    VIDEO_44B6_1026,
    VIDEO_6BBA_1026,
]


assert all(
    name in fold0_train_1026
    for name in SMOKE_VIDEOS_1026
)


for name in SMOKE_VIDEOS_1026:

    assert (
        TRAIN_ROOT_1026
        / f"{name}.zarr"
    ).exists()

    assert (
        TRAIN_ROOT_1026
        / f"{name}.geff"
    ).exists()


print()
print(
    "Smoke 44b6 video:",
    VIDEO_44B6_1026
)

print(
    "Smoke 6bba video:",
    VIDEO_6BBA_1026
)

print(
    "Both are original Fold0 TRAIN:",
    "PASS"
)


# ============================================================
# 5. Create tiny mixed-domain split manifest
#
# Train/test contain the same two videos ONLY for integration
# smoke. This is not used for any quality/model decision.
# ============================================================

smoke_payload_1026 = (
    copy.deepcopy(
        original_payload_1026
    )
)


if (
    isinstance(
        smoke_payload_1026,
        dict,
    )
    and
    "folds"
    in smoke_payload_1026
):

    smoke_folds_1026 = (
        smoke_payload_1026[
            "folds"
        ]
    )

else:

    smoke_folds_1026 = (
        smoke_payload_1026
    )


smoke_folds_1026[
    0
][
    "train"
] = list(
    SMOKE_VIDEOS_1026
)

smoke_folds_1026[
    0
][
    "test"
] = list(
    SMOKE_VIDEOS_1026
)


with SMOKE_SPLITS_1026.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        smoke_payload_1026,
        f,
        indent=2,
    )


print(
    "Mixed-domain smoke manifest:",
    "PASS"
)


# ============================================================
# 6. Import patched training module
#
# Before the subprocess, directly verify REAL samples expose
# dataset_name through FrameWindowDataset.
# ============================================================

if str(
    LOCAL_SCRIPTS_1026
) not in sys.path:

    sys.path.insert(
        0,
        str(
            LOCAL_SCRIPTS_1026
        ),
    )


if str(
    OFFICIAL_SCRIPTS_1026
) not in sys.path:

    sys.path.insert(
        0,
        str(
            OFFICIAL_SCRIPTS_1026
        ),
    )


import importlib

import train_unet_transformer_mps as trainmod_1026

trainmod_1026 = (
    importlib.reload(
        trainmod_1026
    )
)


assert (
    Path(
        trainmod_1026.__file__
    ).resolve()
    ==
    TRAIN_SCRIPT_1026.resolve()
)


# ============================================================
# 7. REAL-data dataset identity smoke
# ============================================================

real_video_data_1026 = []


for name in (
    SMOKE_VIDEOS_1026
):

    vm_1026, windows_1026 = (
        trainmod_1026.load_dataset_windows(
            TRAIN_ROOT_1026
            / f"{name}.zarr",
            window_size=2,
            max_frames=12,
            downsample=(
                1,
                4,
                4,
            ),
        )
    )


    assert len(
        windows_1026
    ) > 0


    real_video_data_1026.append(
        (
            vm_1026,
            windows_1026,
        )
    )


all_windows_1026 = [
    w
    for _, windows in real_video_data_1026
    for w in windows
]


max_nodes_1026 = max(
    max(
        w.node_counts
    )
    for w in all_windows_1026
)


real_ds_1026 = (
    trainmod_1026.FrameWindowDataset(
        real_video_data_1026,
        max_nodes=max_nodes_1026,
        augmentations=None,
    )
)


dataset_names_real_1026 = {
    real_ds_1026[
        i
    ][
        "dataset_name"
    ]

    for i in range(
        len(
            real_ds_1026
        )
    )
}


assert (
    VIDEO_44B6_1026
    in dataset_names_real_1026
)

assert (
    VIDEO_6BBA_1026
    in dataset_names_real_1026
)


print(
    "Real-data dataset_name propagation:",
    "PASS"
)


# ============================================================
# 8. Explicit REAL mixed-batch collation verification
#
# Build one batch manually containing one sample from each
# domain. No model update here.
# ============================================================

idx_44b6_1026 = next(
    i
    for i in range(
        len(
            real_ds_1026
        )
    )
    if real_ds_1026[
        i
    ][
        "dataset_name"
    ].startswith(
        "44b6"
    )
)


idx_6bba_1026 = next(
    i
    for i in range(
        len(
            real_ds_1026
        )
    )
    if real_ds_1026[
        i
    ][
        "dataset_name"
    ].startswith(
        "6bba"
    )
)


from torch.utils.data._utils.collate import default_collate


mixed_real_batch_1026 = (
    default_collate(
        [
            real_ds_1026[
                idx_44b6_1026
            ],

            real_ds_1026[
                idx_6bba_1026
            ],
        ]
    )
)


mixed_names_1026 = list(
    mixed_real_batch_1026[
        "dataset_name"
    ]
)


mixed_mask_1026 = [
    str(
        name
    ).startswith(
        "44b6"
    )
    for name in mixed_names_1026
]


assert mixed_mask_1026 == [
    True,
    False,
]


print(
    "Real mixed-domain batch:",
    mixed_names_1026
)

print(
    "Expected routing mask:",
    mixed_mask_1026
)

print(
    "Real mixed-domain collation:",
    "PASS"
)


# ============================================================
# 9. Subprocess environment
# ============================================================

env_1026 = (
    os.environ.copy()
)


pythonpath_parts_1026 = [
    str(
        LOCAL_SCRIPTS_1026
    ),

    str(
        OFFICIAL_SCRIPTS_1026
    ),
]


if env_1026.get(
    "PYTHONPATH"
):

    pythonpath_parts_1026.append(
        env_1026[
            "PYTHONPATH"
        ]
    )


env_1026[
    "PYTHONPATH"
] = os.pathsep.join(
    pythonpath_parts_1026
)


# ============================================================
# 10. No silent overwrite
# ============================================================

assert not LOG_1026.exists(), (
    f"Smoke log already exists:\n"
    f"{LOG_1026}"
)

assert not CHECKPOINT_1026.exists(), (
    f"Smoke checkpoint already exists:\n"
    f"{CHECKPOINT_1026}"
)


# ============================================================
# 11. Exact short end-to-end training command
# ============================================================

cmd_1026 = [
    str(
        PYTHON_1026
    ),

    str(
        TRAIN_SCRIPT_1026
    ),

    "--data-dir",
    str(
        TRAIN_ROOT_1026
    ),

    "--splits",
    str(
        SMOKE_SPLITS_1026
    ),

    "--split",
    "0",

    "--method",
    METHOD_1026,

    "--epochs",
    "1",

    "--lr",
    "1e-5",

    "--batch-size",
    "8",

    "--num-workers",
    "2",

    "--init-weights",
    str(
        WARM2_1026
    ),

    "--downsample",
    "1,4,4",

    "--det-loss-weight",
    "1.0",

    "--det-neg-weight",
    "0.01",

    "--det-ignore-prob",
    "0.9995",

    "--det-ignore-prefix",
    "44b6",

    "--max-iters",
    "12",

    "--seed",
    "1642",

    "--max-frames",
    "12",

    "--window-size",
    "2",

    "--pool-kernel-um",
    "5.0",
]


print()
print(
    "=" * 72
)

print(
    "10.26 END-TO-END SMOKE"
)

print(
    "=" * 72
)

print(
    "Training videos:",
    SMOKE_VIDEOS_1026
)

print(
    "Optimizer iterations:",
    12
)

print(
    "Domain policy:",
    "44b6 -> .9995; 6bba -> baseline"
)

print(
    "Log:",
    LOG_1026
)


# ============================================================
# 12. Run — stdout/stderr only to log
# ============================================================

start_1026 = (
    time.time()
)


with LOG_1026.open(
    "w",
    encoding="utf-8",
) as log_file_1026:

    proc_1026 = subprocess.run(
        cmd_1026,
        cwd=PROJECT_1026,
        env=env_1026,
        stdout=log_file_1026,
        stderr=subprocess.STDOUT,
        text=True,
    )


elapsed_1026 = (
    time.time()
    -
    start_1026
)


log_text_1026 = (
    LOG_1026.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


if proc_1026.returncode != 0:

    print()
    print(
        "LAST 100 LOG LINES"
    )

    print(
        "\n".join(
            log_text_1026
            .splitlines()[
                -100:
            ]
        )
    )

    raise RuntimeError(
        "10.26 integration smoke failed."
    )


# ============================================================
# 13. Runtime provenance guards
# ============================================================

assert (
    "Fold 0: 2 train, 2 test"
    in log_text_1026
), (
    "Smoke split was not used as expected."
)


assert (
    "Full-model init weights loaded"
    in log_text_1026
)


assert (
    str(
        WARM2_1026
    )
    in log_text_1026
)


# Existing Stage10 log provenance for detection settings.
assert re.search(
    r"Detection loss:\s*"
    r"weight=1(?:\.0+)?,\s*"
    r"neg_weight=0\.01,\s*"
    r"ignore_prob=0\.9995",
    log_text_1026,
), (
    "Could not verify sparse detection-loss configuration "
    "from runtime log."
)


assert (
    "/12"
    in log_text_1026
), (
    "Could not verify 12-iteration training loop."
)


# ============================================================
# 14. Checkpoint verification
# ============================================================

assert CHECKPOINT_1026.exists(), (
    "Smoke checkpoint was not created."
)


checkpoint_sha_1026 = (
    sha256_1026(
        CHECKPOINT_1026
    )
)


assert (
    checkpoint_sha_1026
    !=
    EXPECTED_WARM2_SHA_1026
), (
    "Smoke checkpoint is unexpectedly identical to Warm2."
)


# ============================================================
# 15. Parse final training diagnostics
#
# NOT competition metrics.
# ============================================================

epoch_pattern_1026 = re.compile(
    r"Epoch\s+\d+/\d+\s+\|\s+"
    r"edge=([0-9.eE+-]+)\s+\|\s+"
    r"det=([0-9.eE+-]+)\s+\|\s+"
    r"test_loss=([0-9.eE+-]+)\s+\|\s+"
    r"acc=([0-9.eE+-]+)\s+\|\s+"
    r"recall=([0-9.eE+-]+)"
)


matches_1026 = list(
    epoch_pattern_1026.finditer(
        log_text_1026
    )
)


assert matches_1026


m_1026 = (
    matches_1026[
        -1
    ]
)


edge_loss_1026 = float(
    m_1026.group(
        1
    )
)

det_loss_1026 = float(
    m_1026.group(
        2
    )
)

test_loss_1026 = float(
    m_1026.group(
        3
    )
)

test_acc_1026 = float(
    m_1026.group(
        4
    )
)

test_recall_1026 = float(
    m_1026.group(
        5
    )
)


# ============================================================
# 16. Source-integrity postcondition
# ============================================================

assert (
    sha256_1026(
        TRAIN_SCRIPT_1026
    )
    ==
    EXPECTED_TRAIN_SHA_1026
)


print()
print(
    "=" * 72
)

print(
    "10.26 RESULT"
)

print(
    "=" * 72
)


print()
print(
    "REAL DATAFLOW"
)

print(
    "  selected Fold0 TRAIN videos:",
    SMOKE_VIDEOS_1026
)

print(
    "  mixed batch names:",
    mixed_names_1026
)

print(
    "  routing mask:",
    mixed_mask_1026
)

print(
    "  expected:",
    "[44b6=True, 6bba=False]"
)


print()
print(
    "END-TO-END TRAINING"
)

print(
    "  return code:",
    proc_1026.returncode
)

print(
    "  iterations:",
    12
)

print(
    "  elapsed:",
    f"{elapsed_1026:.1f}s"
)

print(
    "  warm start:",
    "PASS"
)

print(
    "  domain-aware CLI:",
    "PASS"
)

print(
    "  checkpoint created:",
    "PASS"
)


print()
print(
    "TRAINING DIAGNOSTICS — NOT COMPETITION METRICS"
)

print(
    "  edge loss:",
    edge_loss_1026
)

print(
    "  detection loss:",
    det_loss_1026
)

print(
    "  test loss:",
    test_loss_1026
)

print(
    "  test accuracy:",
    test_acc_1026
)

print(
    "  test recall:",
    test_recall_1026
)


print()
print(
    "CHECKPOINT"
)

print(
    "  path:",
    CHECKPOINT_1026
)

print(
    "  SHA256:",
    checkpoint_sha_1026
)


print()
print(
    "Training source SHA unchanged:",
    "PASS"
)

print(
    "No inference was run."
)

print(
    "No competition metric was evaluated."
)


print()
print(
    "10.26 domain-aware integration smoke:",
    "COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 10.28B Domain-Aware Fold0 Paired Inference
# Purpose:
#   Run the new domain-aware checkpoint at det=.995 and .9975.
# Changes files: YES — logs + prediction GEFFs
# Runs training: NO
# Runs inference: YES — 2 x full Fold0
# Uses GT: NO
# Keep after running: NO
# ============================================================

from pathlib import Path
import sys

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "scripts"))

from stage10_experiment_utils import (
    build_project_env,
    ensure_no_existing_outputs,
    load_split_payload,
    project_python,
    run_logged,
    sha256_file,
    verify_prediction_set,
)

PREDICT = PROJECT / "scripts/predict_unet_transformer_mps.py"
TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/biohub-cell-tracking-during-development/train"
)
SPLITS = PROJECT / "configs/dataset_splits.json"

CHECKPOINT = (
    PROJECT
    / "models/official_baseline/"
      "stage10_domain_aware_9995_seed1640_256/"
      "split_0/edge_predictor_best.pth"
)

assert sha256_file(PREDICT) == (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)
assert sha256_file(CHECKPOINT) == (
    "edce784b1625b94965d8ff7c638b255ec319e396cdc6b3cc0b543763d02a24b0"
)

_, folds = load_split_payload(SPLITS)
fold0_names = list(folds[0]["test"])
assert len(fold0_names) == 40

RUNS = [
    (
        0.995,
        "stage10_domainaware9995_1640_det995",
    ),
    (
        0.9975,
        "stage10_domainaware9995_1640_det9975",
    ),
]

results_1028b = []

for threshold, method in RUNS:
    output_dir = (
        PROJECT
        / "predictions/heqiuyan"
        / method
        / "split_0"
    )
    log = (
        PROJECT
        / "reports/stage10_domain_aware"
        / f"{method}.log"
    )

    ensure_no_existing_outputs(
        log_path=log,
        prediction_dir=output_dir,
    )

    command = [
        str(project_python(PROJECT)),
        str(PREDICT),
        "--data-dir", str(TRAIN_ROOT),
        "--splits", str(SPLITS),
        "--split", "0",
        "--method", method,
        "--weights", str(CHECKPOINT),
        "--det-threshold", str(threshold),
        "--unet-batch-size", "4",
    ]

    result = run_logged(
        command,
        cwd=PROJECT,
        env=build_project_env(PROJECT),
        log_path=log,
    )

    geffs = verify_prediction_set(
        output_dir,
        fold0_names,
    )

    results_1028b.append(
        {
            "threshold": threshold,
            "method": method,
            "elapsed_min": result.elapsed_seconds / 60,
            "geffs": len(geffs),
            "output_dir": output_dir,
        }
    )

for r in results_1028b:
    print(
        f"det={r['threshold']}: "
        f"{r['geffs']}/40 — "
        f"{r['elapsed_min']:.1f} min"
    )

print("10.28B paired Fold0 inference: COMPLETE")

In [ ]:
# ============================================================
# RUN — 10.28C Domain-Aware 256 Frozen-V9 Evaluation
# Purpose:
#   Evaluate the new domain-aware checkpoint using frozen V9.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — locked metric evaluation only
# Keep after running: YES
# ============================================================

from pathlib import Path
from dataclasses import fields, is_dataclass
import pandas as pd
import sys

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "src"))

from biohub_cell_tracking.frozen_v9 import evaluate_frozen_v9

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/biohub-cell-tracking-during-development/train"
)

CV_MANIFEST = PROJECT / "configs/cv_manifest.csv"

PRED_995 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_domainaware9995_1640_det995/split_0"
)

PRED_9975 = (
    PROJECT
    / "predictions/heqiuyan/"
      "stage10_domainaware9995_1640_det9975/split_0"
)

# Exact construction used by the successful 10.27B reproduction gate.
cv_meta = pd.read_csv(CV_MANIFEST)
fold0 = cv_meta[cv_meta["fold"] == 0].copy()

assert len(fold0) == 40

dataset_names = sorted(fold0["dataset"].tolist())

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
        strict=True,
    )
)

estimated_nodes = dict(
    zip(
        fold0["dataset"],
        fold0["estimated_nodes"],
        strict=True,
    )
)

result_1028c = evaluate_frozen_v9(
    dataset_names=dataset_names,
    pred_995_dir=PRED_995,
    pred_9975_dir=PRED_9975,
    gt_dir=TRAIN_ROOT,
    estimated_nodes=estimated_nodes,
    prefix_map=prefix_map,
)

print("10.28C Frozen-V9 evaluation: COMPLETE")

# Only show the result object's compact public structure for now.
if is_dataclass(result_1028c):
    print("Result fields:", [f.name for f in fields(result_1028c)])

In [ ]:
# ============================================================
# RESULT — 10.28D Inspect Domain-Aware Frozen-V9 Result
# Purpose:
#   Read compact metrics from the completed 10.28C evaluation.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO — reads existing evaluation result only
# Keep after running: YES
# ============================================================

print("========== SUMMARY ==========")
print(result_1028c.summary)

print("\n========== INTERVENTIONS ==========")
print(result_1028c.interventions)

print("\n========== DATASET_DF COLUMNS ==========")
print(result_1028c.dataset_df.columns.tolist())

print("\n========== FIRST 3 ROWS ==========")
display(result_1028c.dataset_df.head(3))

In [ ]:
# ============================================================
# RESULT — 10.28E Domain-Aware Prefix Diagnosis
# Purpose:
#   Diagnose whether the single-checkpoint failure comes from
#   44b6, 6bba, or both.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO — reads completed evaluation only
# Keep after running: YES
# ============================================================

prefix_1028e = (
    result_1028c.dataset_df
    .groupby("prefix")
    .agg(
        n=("dataset", "size"),
        edge=("edge_jaccard", "mean"),
        adj=("adj_edge_jaccard", "mean"),
        recall=("node_recall", "mean"),
        node_ratio=("total_node_ratio", "mean"),
    )
)

display(prefix_1028e)

### 10.28 — Single-Checkpoint Domain-Aware Sparse Loss

**Checkpoint**
`stage10_domain_aware_9995_seed1640_256`
SHA256:
`edce784b1625b94965d8ff7c638b255ec319e396cdc6b3cc0b543763d02a24b0`

**Policy**
- 44b6: sparse-ignore `p >= .9995`
- 6bba: original baseline negative BCE
- shared model / optimizer

**Frozen V9 Fold0**
- Edge Jaccard: `0.779172`
- Adjusted Edge Jaccard: `0.770912`
- Division Jaccard: `0.015873`
- Node recall: `0.958233`
- Score: **`0.772499`**

**Prefix**
- 44b6 (n=15): Edge `0.753549`, Adj `0.742756`, Recall `0.960531`
- 6bba (n=25): Edge `0.782376`, Adj `0.774236`, Recall `0.956854`

**Decision**
REJECT single-checkpoint domain-aware training.

The failure is concentrated in 44b6 rather than 6bba. The independently
trained Sparse9995 checkpoint had reproduced a positive 44b6 effect across
seeds, but that effect disappeared under shared mixed-domain optimization.

Do not run seed1641 replication.

Retain dual-checkpoint domain routing as the current Stage10 best:
- 44b6 -> Sparse9995 checkpoint
- 6bba -> Baseline256 checkpoint
- Frozen V9 score: **0.777453**

In [ ]:
# ============================================================
# DEBUG — 11.05A Association-Replay API Audit
# Purpose:
#   Inspect reusable APIs already created in Stage 11.04 so the
#   train-vs-validation division audit can reuse them directly.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: NO — delete after resolved
# ============================================================

from pathlib import Path
import inspect
import sys

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "src"))

import biohub_cell_tracking.association_replay as ar

print("PUBLIC FUNCTIONS")
for name, obj in vars(ar).items():
    if (
        not name.startswith("_")
        and inspect.isfunction(obj)
        and obj.__module__ == ar.__name__
    ):
        print(f"\n{name}{inspect.signature(obj)}")

print("\nPUBLIC DATACLASSES / CLASSES")
for name, obj in vars(ar).items():
    if (
        not name.startswith("_")
        and inspect.isclass(obj)
        and obj.__module__ == ar.__name__
    ):
        print(name)

In [ ]:
# ============================================================
# RESULT — 11.05A Division Supervision Mass Audit
# Purpose:
#   Quantify how much of the association supervision actually
#   comes from division parents under the current unweighted loss.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: YES — supervision census only
# Keep after running: YES
# ============================================================

from pathlib import Path
import pandas as pd
import sys

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "src"))

from biohub_cell_tracking.frozen_v9 import load_graph

TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/biohub-cell-tracking-during-development/train"
)

cv = pd.read_csv(PROJECT / "configs/cv_manifest.csv")

rows = []

for r in cv.itertuples(index=False):
    g = load_graph(TRAIN_ROOT / f"{r.dataset}.geff")
    edges = g.edge_attrs(
        attr_keys=["source_id", "target_id"],
        unpack=True,
    ).to_pandas()

    outdeg = edges.groupby("source_id").size()

    assert (outdeg <= 2).all()

    ordinary_parents = int((outdeg == 1).sum())
    division_parents = int((outdeg == 2).sum())

    ordinary_edges = ordinary_parents
    division_edges = 2 * division_parents
    total_edges = int(len(edges))

    rows.append({
        "dataset": r.dataset,
        "prefix": r.prefix,
        "fold": int(r.fold),
        "split_role": "val" if int(r.fold) == 0 else "train",
        "ordinary_parents": ordinary_parents,
        "division_parents": division_parents,
        "ordinary_edges": ordinary_edges,
        "division_edges": division_edges,
        "total_edges": total_edges,
    })

df_1105a = pd.DataFrame(rows)

assert df_1105a["division_parents"].sum() == 151

summary_1105a = (
    df_1105a
    .groupby(["split_role", "prefix"])
    .agg(
        datasets=("dataset", "size"),
        ordinary_parents=("ordinary_parents", "sum"),
        division_parents=("division_parents", "sum"),
        ordinary_edges=("ordinary_edges", "sum"),
        division_edges=("division_edges", "sum"),
        total_edges=("total_edges", "sum"),
    )
)

summary_1105a["division_parent_fraction"] = (
    summary_1105a["division_parents"]
    / (
        summary_1105a["ordinary_parents"]
        + summary_1105a["division_parents"]
    )
)

summary_1105a["division_positive_edge_fraction"] = (
    summary_1105a["division_edges"]
    / summary_1105a["total_edges"]
)

summary_1105a["extra_second_daughter_fraction"] = (
    summary_1105a["division_parents"]
    / summary_1105a["total_edges"]
)

display(summary_1105a)

print("\n========== ALL ==========")
print("GT division parents:", df_1105a["division_parents"].sum())
print("GT edges:", df_1105a["total_edges"].sum())
print(
    "Division positive-edge fraction:",
    df_1105a["division_edges"].sum()
    / df_1105a["total_edges"].sum(),
)
print(
    "Extra second-daughter fraction:",
    df_1105a["division_parents"].sum()
    / df_1105a["total_edges"].sum(),
)

In [ ]:
# ============================================================
# PATCH — 11.05B Division-Positive Association Weight (FIXED)
# Purpose:
#   Add optional weighting ONLY to positive daughter edges of
#   matched division parents. Default=1.0 preserves baseline.
# Changes files: YES — training script + verified backup
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: NO — delete after verification
# ============================================================

from pathlib import Path
import hashlib
import shutil
import py_compile

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
PATH = PROJECT / "scripts/train_unet_transformer_mps.py"
BACKUP = PROJECT / "scripts/train_unet_transformer_mps.py.pre_divpos_backup"

EXPECTED = "c6cd92b3562d8a8ad42d40944a545122ca23b42e85eacd04690abae8ee00cbc1"

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Fail closed: previous failed patch must not have touched source.
assert sha(PATH) == EXPECTED, f"Unexpected source SHA: {sha(PATH)}"

if BACKUP.exists():
    assert sha(BACKUP) == EXPECTED, f"Bad backup SHA: {sha(BACKUP)}"
else:
    shutil.copy2(PATH, BACKUP)

text = PATH.read_text()

def replace_exact(old, new, expected_count=1):
    global text
    count = text.count(old)
    assert count == expected_count, (
        f"Expected {expected_count} matches, got {count}"
    )
    text = text.replace(old, new)

# 1. compute_loss API
replace_exact(
'''def compute_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:''',
'''def compute_loss(
    logits: torch.Tensor,
    target: torch.Tensor,
    division_positive_weight: float = 1.0,
) -> torch.Tensor:''',
)

# 2. Weight only positive entries belonging to a division-parent row
replace_exact(
'''    div_rows = target.sum(dim=1) > 1
    weight = torch.ones_like(loss)
    weight[div_rows] = 1.0

    return (loss * weight)[mask].mean()''',
'''    if division_positive_weight < 1.0:
        raise ValueError("division_positive_weight must be >= 1.0")

    div_rows = target.sum(dim=1) > 1
    div_positive = div_rows.unsqueeze(1) & (target > 0.5)

    weight = torch.ones_like(loss)
    weight[div_positive] = division_positive_weight

    return (loss * weight)[mask].mean()''',
)

# 3. compute_batch_loss API
replace_exact(
'''    mask_t1: torch.Tensor,
) -> torch.Tensor:''',
'''    mask_t1: torch.Tensor,
    division_positive_weight: float = 1.0,
) -> torch.Tensor:''',
)

# 4. Forward weight into compute_loss
replace_exact(
'''        losses.append(compute_loss(logits[b, :nt, :nt1], target[b, :nt, :nt1]))''',
'''        losses.append(compute_loss(
            logits[b, :nt, :nt1],
            target[b, :nt, :nt1],
            division_positive_weight=division_positive_weight,
        ))''',
)

# 5. BOTH train_epoch() and train() need the same new argument.
replace_exact(
'''    det_ignore_prefix: str | None = None,
    max_iters: int | None = None,''',
'''    det_ignore_prefix: str | None = None,
    division_positive_weight: float = 1.0,
    max_iters: int | None = None,''',
    expected_count=2,
)

# 6. train_epoch -> compute_batch_loss
replace_exact(
'''                frame_det[i][2], frame_det[i + 1][2],
            ))''',
'''                frame_det[i][2], frame_det[i + 1][2],
                division_positive_weight=division_positive_weight,
            ))''',
)

# 7. train -> train_epoch
replace_exact(
'''            det_ignore_prefix=det_ignore_prefix,
            max_iters=max_iters,''',
'''            det_ignore_prefix=det_ignore_prefix,
            division_positive_weight=division_positive_weight,
            max_iters=max_iters,''',
)

PATH.write_text(text)
py_compile.compile(str(PATH), doraise=True)

assert sha(BACKUP) == EXPECTED
assert sha(PATH) != EXPECTED

print("11.05B patch: PASS")
print("backup SHA :", sha(BACKUP))
print("new SHA    :", sha(PATH))
print("predictor  : UNTOUCHED")

In [ ]:
# ============================================================
# RUN — 11.05B Division-Positive Weight Regression Gate
# Purpose:
#   Verify baseline-exact behavior and localization of the new
#   division-positive association weighting.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO — synthetic association targets only
# Keep after running: NO — replace with RESULT after PASS
# ============================================================

from pathlib import Path
from importlib.machinery import SourceFileLoader
from importlib.util import spec_from_loader, module_from_spec
import hashlib
import inspect
import sys
import torch

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
SCRIPT = PROJECT / "scripts/train_unet_transformer_mps.py"
BACKUP = PROJECT / "scripts/train_unet_transformer_mps.py.pre_divpos_backup"
PREDICTOR = PROJECT / "scripts/predict_unet_transformer_mps.py"

sys.path.insert(0, str(PROJECT / "scripts"))
sys.path.insert(0, str(PROJECT / "external/official/scripts"))

def load_source(path, name):
    loader = SourceFileLoader(name, str(path))
    spec = spec_from_loader(name, loader)
    mod = module_from_spec(spec)
    sys.modules[name] = mod
    loader.exec_module(mod)
    return mod

old = load_source(BACKUP, "_stage11_old_train")
new = load_source(SCRIPT, "_stage11_new_train")

logits0 = torch.tensor([
    [ 2.0,  1.3, -0.4, -1.0],
    [ 0.4,  0.7,  1.6, -0.2],
    [-0.6, -0.2,  0.3,  1.5],
    [-1.2, -0.8, -0.5,  0.1],
], dtype=torch.float32)

division_target = torch.tensor([
    [1., 1., 0., 0.],   # division parent
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
    [0., 0., 0., 0.],
])

ordinary_target = torch.tensor([
    [1., 0., 0., 0.],
    [0., 1., 0., 0.],
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
])

def value_grad(fn, target, weight=None):
    x = logits0.clone().requires_grad_(True)
    loss = fn(x, target) if weight is None else fn(
        x, target, division_positive_weight=weight
    )
    loss.backward()
    return loss.detach(), x.grad.detach()

old_l, old_g = value_grad(old.compute_loss, division_target)
new1_l, new1_g = value_grad(new.compute_loss, division_target, 1.0)

assert torch.equal(old_l, new1_l)
assert torch.equal(old_g, new1_g)

ord1_l, ord1_g = value_grad(new.compute_loss, ordinary_target, 1.0)
ord16_l, ord16_g = value_grad(new.compute_loss, ordinary_target, 16.0)

assert torch.equal(ord1_l, ord16_l)
assert torch.equal(ord1_g, ord16_g)

div2_l, _ = value_grad(new.compute_loss, division_target, 2.0)
div16_l, div16_g = value_grad(new.compute_loss, division_target, 16.0)

# Weighting is linear in the selected loss entries.
assert torch.allclose(
    div16_l - new1_l,
    15 * (div2_l - new1_l),
    rtol=1e-6, atol=1e-7,
)

div_positive = (
    (division_target.sum(dim=1) > 1).unsqueeze(1)
    & (division_target > 0.5)
)
assert div_positive.nonzero().tolist() == [[0, 0], [0, 1]]
assert not torch.equal(new1_g, div16_g)

for name in ("compute_batch_loss", "train_epoch", "train"):
    assert "division_positive_weight" in inspect.signature(
        getattr(new, name)
    ).parameters

try:
    new.compute_loss(logits0, division_target, division_positive_weight=0.5)
    raise AssertionError("weight<1 should fail")
except ValueError:
    pass

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

print("11.05B regression gate: PASS")
print("baseline loss exact :", float(new1_l))
print("weight=16 loss      :", float(div16_l))
print("gradient Δ norm     :", float((div16_g - new1_g).norm()))
print("weighted entries    :", div_positive.nonzero().tolist())
print("patched train SHA   :", sha(SCRIPT))
print("backup train SHA    :", sha(BACKUP))
print("predictor SHA       :", sha(PREDICTOR))

In [ ]:
# ============================================================
# RUN — 11.05C DivPos16 Fold0 256-Iteration Pilot
# Purpose:
#   Causal ablation of division-positive association weighting.
#   Exact Baseline256 setup, changing ONLY:
#       division_positive_weight = 16.0
# Changes files: YES — checkpoint/config + training log
# Runs training: YES — 256 optimizer iterations
# Uses GT: YES — normal Fold0 train/validation supervision
# Keep after running: NO — delete after successful verification
# ============================================================

from pathlib import Path
import hashlib
import os
import subprocess
import sys
import time

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
PYTHON = PROJECT / ".venv/bin/python"
TRAIN_SCRIPT = PROJECT / "scripts/train_unet_transformer_mps.py"
TRAIN_ROOT = PROJECT / "data/raw/competition/biohub-cell-tracking-during-development/train"
SPLITS = PROJECT / "configs/dataset_splits.json"

WARM2 = (
    PROJECT / "models/official_baseline/"
    "official_baseline_fold0_warm2_lr1e5/"
    "split_0/edge_predictor_best.pth"
)

METHOD = "stage11_divpos16_seed1640_256"
OUT_DIR = PROJECT / "models/official_baseline" / METHOD / "split_0"
CKPT = OUT_DIR / "edge_predictor_best.pth"

LOG_DIR = PROJECT / "reports/stage11_divpos"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / f"{METHOD}.log"

EXPECTED_TRAIN_SHA = "3e9d0fb8f7f3e95e21d0af0c788a9e45d2bf67b771895d28efc4aa853ab506ac"
EXPECTED_WARM2_SHA = "31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

assert Path(sys.executable).resolve() == PYTHON.resolve()
assert sha256(TRAIN_SCRIPT) == EXPECTED_TRAIN_SHA
assert sha256(WARM2) == EXPECTED_WARM2_SHA
assert not OUT_DIR.exists(), f"Refusing overwrite: {OUT_DIR}"
assert not LOG.exists(), f"Refusing overwrite: {LOG}"

code = f"""
from pathlib import Path
from train_unet_transformer_mps import train

train(
    data_dir=Path({str(TRAIN_ROOT)!r}),
    fold=0,
    splits_file=Path({str(SPLITS)!r}),
    method={METHOD!r},
    n_epochs=1,
    lr=1e-5,
    batch_size=8,
    num_workers=4,
    init_weights=Path({str(WARM2)!r}),
    downsample=(1, 4, 4),
    det_loss_weight=1.0,
    det_neg_weight=0.01,
    det_ignore_prob=None,
    det_ignore_prefix=None,
    division_positive_weight=16.0,
    max_iters=256,
    seed=1640,
    window_size=2,
    pool_kernel_um=5.0,
    data_parallel=False,
)
"""

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join([
    str(PROJECT / "scripts"),
    str(PROJECT / "external/official/scripts"),
    env.get("PYTHONPATH", ""),
]).rstrip(os.pathsep)

print("11.05C training START")
print("method :", METHOD)
print("log    :", LOG)

started = time.time()

with LOG.open("w", encoding="utf-8") as f:
    proc = subprocess.run(
        [str(PYTHON), "-c", code],
        cwd=PROJECT,
        env=env,
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
    )

elapsed = time.time() - started

if proc.returncode != 0:
    tail = "\n".join(LOG.read_text(errors="replace").splitlines()[-80:])
    print(tail)
    raise RuntimeError(f"Training failed: returncode={proc.returncode}")

assert CKPT.exists()
assert sha256(TRAIN_SCRIPT) == EXPECTED_TRAIN_SHA

lines = LOG.read_text(errors="replace").splitlines()
epoch_lines = [x for x in lines if "Epoch " in x and "edge=" in x]

print("\n11.05C training: COMPLETE")
print(f"elapsed       : {elapsed / 60:.1f} min")
print("checkpoint SHA:", sha256(CKPT))
if epoch_lines:
    print("final metrics :", epoch_lines[-1].strip())

In [ ]:
# ============================================================
# RUN — 11.05D DivPos16 6bba-Only Fold0 Inference
# Purpose:
#   Generate only the 6bba predictions needed to test whether
#   DivPos16 improves the current best routed hybrid.
# Changes files: YES
#   - temporary 6bba-only split manifest under reports/
#   - two prediction directories
#   - two inference logs
# Runs training: NO
# Runs inference: YES — 25 datasets × 2 thresholds
# Uses GT: NO
# Keep after running: NO — retain RESULT/logs/predictions
# ============================================================

from pathlib import Path
import hashlib
import json
import os
import re
import subprocess
import sys
import time
import pandas as pd

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
PYTHON = PROJECT / ".venv/bin/python"
PREDICT = PROJECT / "scripts/predict_unet_transformer_mps.py"

DATA_ROOT = (
    PROJECT
    / "data/raw/competition/biohub-cell-tracking-during-development/train"
)

SPLITS = PROJECT / "configs/dataset_splits.json"
CV = PROJECT / "configs/cv_manifest.csv"

CKPT = (
    PROJECT
    / "models/official_baseline/stage11_divpos16_seed1640_256/"
      "split_0/edge_predictor_best.pth"
)

REPORT = PROJECT / "reports/stage11_divpos"
REPORT.mkdir(parents=True, exist_ok=True)

SUBSET_SPLIT = REPORT / "fold0_6bba_only.json"

EXPECTED_CKPT = (
    "32af8497962904508a6a7f1f06b45c7e334e3b1a037221cc9a11d87f6c2a3ac8"
)
EXPECTED_PREDICT = (
    "23b76f7c90ffe698e955efee2b4333277a07f5b58b7eff6e580f2bb2ff49bc55"
)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

assert Path(sys.executable).resolve() == PYTHON.resolve()
assert sha256(CKPT) == EXPECTED_CKPT
assert sha256(PREDICT) == EXPECTED_PREDICT

# Exact Fold0 validation 6bba subset.
cv = pd.read_csv(CV)
fold0_6bba = sorted(
    cv.loc[
        (cv["fold"] == 0) & (cv["prefix"] == "6bba"),
        "dataset",
    ].tolist()
)

assert len(fold0_6bba) == 25

splits = json.loads(SPLITS.read_text())
assert isinstance(splits, list)
assert len(splits[0]["test"]) == 40
assert set(fold0_6bba).issubset(set(splits[0]["test"]))

subset = [dict(splits[0])]
subset[0]["test"] = fold0_6bba
SUBSET_SPLIT.write_text(json.dumps(subset, indent=2))

env = os.environ.copy()
parts = [
    str(PROJECT / "scripts"),
    str(PROJECT / "external/official/scripts"),
]
if env.get("PYTHONPATH"):
    parts.append(env["PYTHONPATH"])
env["PYTHONPATH"] = os.pathsep.join(parts)

runs = [
    (
        "stage11_divpos16_1640_6bba_det995",
        0.995,
    ),
    (
        "stage11_divpos16_1640_6bba_det9975",
        0.9975,
    ),
]

save_re = re.compile(r"Saved\s+(\d+)\s+predictions\s+to\s+(.+)")

results = []

for method, det in runs:
    log_path = REPORT / f"{method}.log"
    assert not log_path.exists(), f"Refusing overwrite: {log_path}"

    cmd = [
        str(PYTHON),
        str(PREDICT),
        "--data-dir", str(DATA_ROOT),
        "--splits", str(SUBSET_SPLIT),
        "--split", "0",
        "--weights", str(CKPT),
        "--method", method,
        "--det-threshold", str(det),
        "--unet-batch-size", "4",
    ]

    print(f"START {method}  det={det}")
    started = time.time()

    with log_path.open("w", encoding="utf-8") as f:
        proc = subprocess.run(
            cmd,
            cwd=PROJECT,
            env=env,
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed = time.time() - started
    text = log_path.read_text(errors="replace")

    if proc.returncode != 0:
        print("\n".join(text.splitlines()[-80:]))
        raise RuntimeError(f"{method} failed")

    matches = list(save_re.finditer(text))
    assert matches

    n = int(matches[-1].group(1))
    out = Path(matches[-1].group(2).strip())
    if not out.is_absolute():
        out = PROJECT / out
    out = out.resolve()

    assert n == 25
    assert len(list(out.glob("*.geff"))) == 25

    results.append((method, det, elapsed, out))

    print(f"COMPLETE — {elapsed / 60:.1f} min")
    print("output   :", out)
    print()

print("11.05D inference: COMPLETE")
for method, det, elapsed, out in results:
    print(
        f"{method} | det={det} | "
        f"{elapsed / 60:.1f} min | {out}"
    )

In [ ]:
# ============================================================
# RUN — 11.05E DivPos16 Routed-Hybrid Frozen-V9 Evaluation
# Purpose:
#   Compare:
#     44b6 -> existing Sparse9995
#     6bba -> new DivPos16
#   against the current-best routed hybrid.
# Changes files: NO — temporary symlinks only
# Runs training/inference: NO
# Uses GT: YES — locked Fold0 metric evaluation
# Keep after running: YES
# ============================================================

from pathlib import Path
import tempfile
import pandas as pd

from biohub_cell_tracking.frozen_v9 import evaluate_frozen_v9

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
TRAIN_ROOT = (
    PROJECT
    / "data/raw/competition/biohub-cell-tracking-during-development/train"
)

OLD_44_995 = (
    PROJECT / "predictions/heqiuyan/"
    "stage10_infer_sparse9995_256_det995/split_0"
)
OLD_44_9975 = (
    PROJECT / "predictions/heqiuyan/"
    "stage10_infer_sparse9995_256_det9975/split_0"
)

NEW_6_995 = (
    PROJECT / "predictions/heqiuyan/"
    "stage11_divpos16_1640_6bba_det995/split_0"
)
NEW_6_9975 = (
    PROJECT / "predictions/heqiuyan/"
    "stage11_divpos16_1640_6bba_det9975/split_0"
)

cv = pd.read_csv(PROJECT / "configs/cv_manifest.csv")
fold0 = cv[cv["fold"] == 0].copy()

names = sorted(fold0["dataset"])
prefix_map = dict(zip(fold0["dataset"], fold0["prefix"], strict=True))
estimated_nodes = dict(
    zip(fold0["dataset"], fold0["estimated_nodes"], strict=True)
)

assert len(names) == 40
assert sum(fold0["prefix"] == "44b6") == 15
assert sum(fold0["prefix"] == "6bba") == 25

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    routed_995 = tmp / "det995"
    routed_9975 = tmp / "det9975"
    routed_995.mkdir()
    routed_9975.mkdir()

    for name in names:
        if prefix_map[name] == "44b6":
            src995 = OLD_44_995 / f"{name}.geff"
            src9975 = OLD_44_9975 / f"{name}.geff"
        else:
            src995 = NEW_6_995 / f"{name}.geff"
            src9975 = NEW_6_9975 / f"{name}.geff"

        assert src995.exists() and src9975.exists()

        (routed_995 / f"{name}.geff").symlink_to(
            src995, target_is_directory=src995.is_dir()
        )
        (routed_9975 / f"{name}.geff").symlink_to(
            src9975, target_is_directory=src9975.is_dir()
        )

    result_1105e = evaluate_frozen_v9(
        dataset_names=names,
        pred_995_dir=routed_995,
        pred_9975_dir=routed_9975,
        gt_dir=TRAIN_ROOT,
        estimated_nodes=estimated_nodes,
        prefix_map=prefix_map,
    )

print("========== DIVPOS16 HYBRID ==========")
for k, v in result_1105e.summary.items():
    print(f"{k:20s}: {v}")

print("\n========== PREFIX ==========")
display(
    result_1105e.dataset_df.groupby("prefix").agg(
        n=("dataset", "size"),
        edge=("edge_jaccard", "mean"),
        adj=("adj_edge_jaccard", "mean"),
        recall=("node_recall", "mean"),
    )
)

CURRENT_BEST = 0.777452947

print("\n========== GATE ==========")
print("current best :", CURRENT_BEST)
print("DivPos16     :", result_1105e.summary["score"])
print("delta        :", result_1105e.summary["score"] - CURRENT_BEST)

In [ ]:
# ============================================================
# RUN — 11.05F DivPos16 p2 Calibration Audit
# Purpose:
#   Test whether DivPos16 changed division-score calibration.
#   Frozen geometry; only p2 is varied on existing predictions.
# Changes files: NO — temporary symlinks only
# Runs training/inference: NO
# Uses GT: YES — Fold0 metric sensitivity audit
# Keep after running: YES
# ============================================================

from dataclasses import replace
from pathlib import Path
import tempfile
import pandas as pd

from biohub_cell_tracking.frozen_v9 import (
    evaluate_frozen_v9,
    FROZEN_V9_CONFIG,
)

CURRENT_BEST = 0.777452947
rows_1105f = []

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    p995, p9975 = tmp / "995", tmp / "9975"
    p995.mkdir()
    p9975.mkdir()

    for name in names:
        if prefix_map[name] == "44b6":
            src995, src9975 = (
                OLD_44_995 / f"{name}.geff",
                OLD_44_9975 / f"{name}.geff",
            )
        else:
            src995, src9975 = (
                NEW_6_995 / f"{name}.geff",
                NEW_6_9975 / f"{name}.geff",
            )

        (p995 / f"{name}.geff").symlink_to(src995, target_is_directory=True)
        (p9975 / f"{name}.geff").symlink_to(src9975, target_is_directory=True)

    for p2 in (0.80, 0.85, 0.90, 0.95):
        cfg = replace(FROZEN_V9_CONFIG, division_p2_min=p2)

        ev = evaluate_frozen_v9(
            names,
            p995,
            p9975,
            TRAIN_ROOT,
            estimated_nodes,
            prefix_map=prefix_map,
            config=cfg,
        )

        s = ev.summary
        rows_1105f.append({
            "p2": p2,
            "score": s["score"],
            "delta_vs_best": s["score"] - CURRENT_BEST,
            "adj_edge": s["adj_edge_jaccard"],
            "division_j": s["division_jaccard"],
            "TP": s["division_tp"],
            "FP": s["division_fp"],
            "FN": s["division_fn"],
        })

display(pd.DataFrame(rows_1105f))

In [ ]:
# ============================================================
# RUN — 11.06B Teacher-Forced Parent Representation Probe
# Purpose:
#   Test whether frozen UNet parent features already contain
#   linearly usable division-vs-continuation information.
# Changes files: NO
# Runs training: NO neural-network training
# Runs inference: YES — targeted frozen feature extraction only
# Uses GT: YES — GT parent coordinates / division labels
# Keep after running: YES
# ============================================================

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")
sys.path.insert(0, str(PROJECT / "scripts"))
sys.path.insert(0, str(PROJECT / "external/official/scripts"))
sys.path.insert(0, str(PROJECT / "src"))

from replay_stage11_association import (
    DatasetReplay,
    ROUTING,
    load_model,
    GT_DIR,
)
from biohub_cell_tracking.frozen_v9 import load_graph

GT_ROOT = PROJECT / GT_DIR
CV = pd.read_csv(PROJECT / "configs/cv_manifest.csv")

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

# ------------------------------------------------------------
# Load the two checkpoints used by the current-best hybrid.
# ------------------------------------------------------------

models = {}

for prefix, cfg in ROUTING.items():
    model, window_size, downsample = load_model(
        PROJECT / cfg["checkpoint"],
        device,
    )
    model.eval()
    models[prefix] = (model, window_size, tuple(downsample))

# ------------------------------------------------------------
# Build one row per annotated tracked parent.
# Keep only frames containing >=1 GT division.
# Negatives are ordinary continuation parents from SAME frame.
# ------------------------------------------------------------

examples = []

for meta in CV.itertuples(index=False):
    prefix = meta.prefix
    model, window_size, downsample = models[prefix]

    g = load_graph(GT_ROOT / f"{meta.dataset}.geff")
    nodes = g.node_attrs(unpack=True).to_pandas()
    edges = g.edge_attrs(
        attr_keys=["source_id", "target_id"],
        unpack=True,
    ).to_pandas()

    outdeg = edges.groupby("source_id").size()

    division_ids = set(
        int(x)
        for x in outdeg[outdeg == 2].index
    )

    if not division_ids:
        continue

    div_nodes = nodes[
        nodes["node_id"].isin(division_ids)
    ]

    division_times = sorted(
        set(int(t) for t in div_nodes["t"])
    )

    replay = DatasetReplay(
        meta.dataset,
        GT_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    for t in division_times:
        if t >= replay.T - 1:
            continue

        frame = nodes[nodes["t"] == t].copy()

        tracked_ids = set(
            int(x)
            for x in outdeg.index
        )

        frame = frame[
            frame["node_id"].isin(tracked_ids)
        ].copy()

        frame["label"] = frame["node_id"].isin(
            division_ids
        ).astype(int)

        # Same-frame negatives only.
        if frame["label"].sum() == 0:
            continue

        with torch.no_grad():
            unet_out, _ = replay._encode([t, t + 1])

        xyz = frame[["z", "y", "x"]].to_numpy(
            dtype=np.float32
        )

        ds = np.asarray(
            downsample,
            dtype=np.float32,
        )

        coords_ds = xyz / ds

        coords_t = torch.from_numpy(
            coords_ds
        ).unsqueeze(0).to(device)

        mask = torch.ones(
            1,
            len(frame),
            dtype=torch.bool,
            device=device,
        )

        feat = model._index_features(
            unet_out[:, 0],
            coords_t,
            mask,
        )[0].detach().cpu().numpy()

        for j, row in enumerate(
            frame.itertuples(index=False)
        ):
            examples.append({
                "dataset": meta.dataset,
                "prefix": prefix,
                "fold": int(meta.fold),
                "t": t,
                "node_id": int(row.node_id),
                "label": int(row.label),
                "feature": feat[j],
            })

probe_df = pd.DataFrame(examples)

print("Feature extraction complete")
print("examples :", len(probe_df))
print("divisions:", int(probe_df["label"].sum()))
print("feature dim:", len(probe_df.iloc[0]["feature"]))

# ------------------------------------------------------------
# Strict Fold0 split:
# fold==0 datasets = held-out validation.
# ------------------------------------------------------------

train_df = probe_df[probe_df["fold"] != 0]
val_df = probe_df[probe_df["fold"] == 0]

X_train = np.stack(train_df["feature"])
y_train = train_df["label"].to_numpy()

X_val = np.stack(val_df["feature"])
y_val = val_df["label"].to_numpy()

# Standardize using TRAIN only.
mu = X_train.mean(axis=0)
sd = X_train.std(axis=0)
sd[sd < 1e-6] = 1.0

X_train = (X_train - mu) / sd
X_val = (X_val - mu) / sd

probe = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    random_state=1640,
)

probe.fit(X_train, y_train)

p_train = probe.predict_proba(X_train)[:, 1]
p_val = probe.predict_proba(X_val)[:, 1]

def summarize(name, y, p):
    prevalence = y.mean()
    ap = average_precision_score(y, p)
    auc = roc_auc_score(y, p)

    return {
        "split": name,
        "n": len(y),
        "positive": int(y.sum()),
        "prevalence": prevalence,
        "PR_AUC": ap,
        "PR_lift": ap / prevalence,
        "ROC_AUC": auc,
    }

probe_summary_1106b = pd.DataFrame([
    summarize("train", y_train, p_train),
    summarize("val", y_val, p_val),
])

display(probe_summary_1106b)

In [ ]:
# ============================================================
# RESULT — 11.06C Division Probe Operating-Point Audit
# Purpose:
#   Measure whether the existing linear division signal reaches
#   useful precision/recall operating points and beat shuffle.
# Changes files: NO
# Runs training/inference: NO neural-network work
# Uses GT: NO new GT access — existing probe labels only
# Keep after running: YES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)

# ------------------------------------------------------------
# 1. Validation operating points
# ------------------------------------------------------------

precision, recall, threshold = precision_recall_curve(y_val, p_val)

rows = []
for target_recall in (0.10, 0.20, 0.30, 0.50, 0.70):
    valid = np.where(recall[:-1] >= target_recall)[0]

    if len(valid):
        j = valid[np.argmax(precision[:-1][valid])]
        rows.append({
            "target_recall": target_recall,
            "precision": precision[j],
            "recall": recall[j],
            "threshold": threshold[j],
        })

op_1106c = pd.DataFrame(rows)

# ------------------------------------------------------------
# 2. Top-k ranking quality
# ------------------------------------------------------------

order = np.argsort(-p_val)
top_rows = []

for k in (10, 20, 30, 50, 100):
    k = min(k, len(y_val))
    tp = int(y_val[order[:k]].sum())

    top_rows.append({
        "top_k": k,
        "TP": tp,
        "precision": tp / k,
        "recall": tp / y_val.sum(),
    })

topk_1106c = pd.DataFrame(top_rows)

# ------------------------------------------------------------
# 3. Train-label shuffle control
# ------------------------------------------------------------

rng = np.random.default_rng(1640)

shuffle_ap = []
shuffle_auc = []

for _ in range(100):
    shuffled = rng.permutation(y_train)

    null_probe = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=1640,
    )

    null_probe.fit(X_train, shuffled)
    p_null = null_probe.predict_proba(X_val)[:, 1]

    shuffle_ap.append(
        average_precision_score(y_val, p_null)
    )
    shuffle_auc.append(
        roc_auc_score(y_val, p_null)
    )

print("========== OPERATING POINTS ==========")
display(op_1106c)

print("\n========== TOP-K ==========")
display(topk_1106c)

print("\n========== SHUFFLE CONTROL ==========")
print(
    "Observed AP :", average_precision_score(y_val, p_val),
    "| null mean:", np.mean(shuffle_ap),
    "| null q95 :", np.quantile(shuffle_ap, 0.95),
)
print(
    "Observed AUC:", roc_auc_score(y_val, p_val),
    "| null mean:", np.mean(shuffle_auc),
    "| null q95 :", np.quantile(shuffle_auc, 0.95),
)

In [ ]:
# ============================================================
# RUN — 11.06D Judgeable-Fork Parent Probe Audit
# Purpose:
#   Score the 2 TP and 33 FP division parents using the frozen
#   parent-representation linear probe from 11.06B/C.
# Changes files: NO
# Runs training: NO
# Runs inference: YES — targeted feature extraction only
# Uses GT: YES — only to identify metric TP/FP cases
# Keep after running: YES
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

PROJECT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

PRED_DIV = (
    PROJECT
    / "reports/stage11_division_audit/pred_divisions.csv"
)

pred_div = pd.read_csv(PRED_DIV)

judgeable = pred_div[
    pred_div["counted_as_tp"] | pred_div["counted_as_fp"]
].copy()

assert len(judgeable) == 35
assert int(judgeable["counted_as_tp"].sum()) == 2
assert int(judgeable["counted_as_fp"].sum()) == 33

rows = []

for r in judgeable.itertuples(index=False):
    prefix = r.prefix
    model, window_size, downsample = models[prefix]

    replay = DatasetReplay(
        r.dataset,
        GT_ROOT,
        model,
        window_size,
        downsample,
        device,
    )

    # Existing routed prediction graph containing this parent.
    pred_dir = (
        PROJECT / ROUTING[prefix]["pred_995"]
    )
    g = load_graph(pred_dir / f"{r.dataset}.geff")

    nodes = (
        g.node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    parent = nodes.loc[int(r.parent_node)]

    t = int(parent["t"])
    xyz = parent[["z", "y", "x"]].to_numpy(
        dtype=np.float32
    )

    with torch.no_grad():
        unet_out, _ = replay._encode([t, t + 1])

        coords_ds = (
            xyz / np.asarray(downsample, dtype=np.float32)
        )

        coords_t = torch.from_numpy(
            coords_ds
        ).reshape(1, 1, 3).to(device)

        mask = torch.ones(
            1, 1,
            dtype=torch.bool,
            device=device,
        )

        feat = model._index_features(
            unet_out[:, 0],
            coords_t,
            mask,
        )[0, 0].cpu().numpy()

    feat_std = (feat - mu) / sd
    div_prob = float(
        probe.predict_proba(feat_std[None])[0, 1]
    )

    rows.append({
        "dataset": r.dataset,
        "prefix": prefix,
        "parent": int(r.parent_node),
        "t": t,
        "case": "TP" if r.counted_as_tp else "FP",
        "division_probe_prob": div_prob,
    })

fork_probe_1106d = (
    pd.DataFrame(rows)
    .sort_values(
        "division_probe_prob",
        ascending=False,
    )
    .reset_index(drop=True)
)

fork_probe_1106d["rank"] = (
    np.arange(len(fork_probe_1106d)) + 1
)

print("========== TP SCORES ==========")
display(
    fork_probe_1106d[
        fork_probe_1106d["case"] == "TP"
    ]
)

print("\n========== SUMMARY ==========")
display(
    fork_probe_1106d.groupby("case")[
        "division_probe_prob"
    ].agg(["count", "min", "median", "max"])
)

print("\n========== TOP 15 ==========")
display(fork_probe_1106d.head(15))

print("\n========== THRESHOLD AUDIT ==========")

threshold_rows = []

for th in (0.40, 0.50, 0.60, 0.70):
    kept = fork_probe_1106d[
        fork_probe_1106d["division_probe_prob"] >= th
    ]

    threshold_rows.append({
        "threshold": th,
        "TP_kept": int((kept["case"] == "TP").sum()),
        "FP_kept": int((kept["case"] == "FP").sum()),
        "forks_kept": len(kept),
    })

display(pd.DataFrame(threshold_rows))

In [ ]:
# ============================================================
# DEBUG — 11.06G-pre Fork Edge-Provenance Inspection
# Purpose:
#   Inspect how daughters and edge probabilities are encoded,
#   and determine whether a rejected fork has an unambiguous
#   "second/recovered daughter" edge that can be removed.
# Changes files: NO
# Runs training/inference: NO
# Uses GT: NO
# Keep after running: NO — delete after resolved
# ============================================================

import pandas as pd

LOW = scored_1106f[
    scored_1106f["division_probe_prob"] < 0.40
].copy()

cols = [
    "dataset",
    "prefix",
    "parent_node",
    "t",
    "daughters",
    "edge_probs",
    "p2",
    "origin_includes_recovered_edge",
    "division_probe_prob",
]

print("========== TYPES ==========")
for c in cols:
    print(c, ":", LOW[c].dtype)

print("\n========== REJECTED FORK BREAKDOWN ==========")
display(
    LOW.groupby(
        ["prefix", "origin_includes_recovered_edge"],
        dropna=False,
    ).size().rename("n").reset_index()
)

print("\n========== FIRST 12 REJECTED ==========")
display(LOW[cols].head(12))

print("\n========== RECOVERED-EDGE EXAMPLES ==========")
display(
    LOW[
        LOW["origin_includes_recovered_edge"].astype(bool)
    ][cols].head(12)
)

print("\n========== NATIVE-FORK EXAMPLES ==========")
display(
    LOW[
        ~LOW["origin_includes_recovered_edge"].astype(bool)
    ][cols].head(12)
)

In [ ]:
# ============================================================
# RUN — 11.06G True-Continuation Branch Audit
# Purpose:
#   For the 33 judgeable FP forks, use the official full-graph
#   node matching to determine whether the higher-probability
#   or lower-probability daughter is the GT continuation.
# Changes files: NO — temporary routed symlinks only
# Runs training/inference: NO
# Uses GT: YES — official Fold0 node matching
# Keep after running: YES
# ============================================================

from pathlib import Path
import tempfile
import pandas as pd

from biohub_cell_tracking.frozen_v9 import (
    FROZEN_V9_CONFIG,
    apply_frozen_v9,
    build_frozen_v9_policies,
    load_graph,
)
from biohub_cell_tracking.division_audit import (
    _match_full,
    _pred_to_gt_map,
)

fp_rows = scored_1106f[
    scored_1106f["counted_as_fp"].astype(bool)
].copy()

assert len(fp_rows) == 33

audit_rows = []

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    routed995 = tmp / "995"
    routed9975 = tmp / "9975"
    routed995.mkdir()
    routed9975.mkdir()

    # Recreate the exact current-best routed hybrid.
    for name in names:
        if prefix_map[name] == "44b6":
            src995 = OLD_44_995 / f"{name}.geff"
            src9975 = OLD_44_9975 / f"{name}.geff"
        else:
            src995 = (
                PROJECT
                / "predictions/heqiuyan/"
                  "stage10_infer_baseline_256_det995/split_0"
                / f"{name}.geff"
            )
            src9975 = (
                PROJECT
                / "predictions/heqiuyan/"
                  "stage10_infer_baseline_256_det9975/split_0"
                / f"{name}.geff"
            )

        (routed995 / f"{name}.geff").symlink_to(
            src995, target_is_directory=True
        )
        (routed9975 / f"{name}.geff").symlink_to(
            src9975, target_is_directory=True
        )

    policies = build_frozen_v9_policies(
        names,
        routed995,
        routed9975,
        TRAIN_ROOT,
        prefix_map=prefix_map,
        progress=False,
    )

    for dataset, group in fp_rows.groupby("dataset"):
        graph, scale, _ = apply_frozen_v9(
            dataset,
            routed995,
            routed9975,
            policies,
            TRAIN_ROOT,
        )

        gt_graph = load_graph(
            TRAIN_ROOT / f"{dataset}.geff"
        )

        pred_to_gt = _pred_to_gt_map(
            _match_full(
                graph,
                gt_graph,
                scale,
                FROZEN_V9_CONFIG.metric_max_distance,
            )
        )

        edge_df = graph.edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        ).to_pandas()

        for r in group.itertuples(index=False):
            parent = int(r.parent_node)

            parent_edges = (
                edge_df[
                    edge_df["source_id"].astype(int) == parent
                ]
                .sort_values(
                    ["edge_prob", "target_id"],
                    ascending=[False, True],
                )
            )

            daughters = (
                parent_edges["target_id"]
                .astype(int)
                .tolist()
            )

            probs = (
                parent_edges["edge_prob"]
                .astype(float)
                .tolist()
            )

            gt_parent = pred_to_gt.get(parent)

            if gt_parent is None:
                outcome = "PARENT_UNMATCHED"
                gt_child = None

            elif int(gt_graph.out_degree(gt_parent)) != 1:
                outcome = "GT_PARENT_NOT_DEG1"
                gt_child = None

            elif len(daughters) != 2:
                outcome = "NOT_TWO_DAUGHTERS"
                gt_child = None

            else:
                gt_child = int(
                    next(iter(gt_graph.successors(gt_parent)))
                )

                high_gt = pred_to_gt.get(daughters[0])
                low_gt = pred_to_gt.get(daughters[1])

                if high_gt == gt_child:
                    outcome = "HIGHER_PROB_CORRECT"
                elif low_gt == gt_child:
                    outcome = "LOWER_PROB_CORRECT"
                else:
                    outcome = "NEITHER_CORRECT"

            audit_rows.append({
                "dataset": dataset,
                "prefix": r.prefix,
                "parent": parent,
                "probe_prob": float(r.division_probe_prob),
                "higher_daughter": daughters[0] if daughters else None,
                "higher_prob": probs[0] if probs else None,
                "lower_daughter": daughters[1] if len(daughters) > 1 else None,
                "lower_prob": probs[1] if len(probs) > 1 else None,
                "gt_parent": gt_parent,
                "gt_child": gt_child,
                "outcome": outcome,
            })

branch_1106g = pd.DataFrame(audit_rows)

print("========== OUTCOME ==========")
display(
    branch_1106g["outcome"]
    .value_counts()
    .rename("n")
    .reset_index()
)

eligible = branch_1106g[
    branch_1106g["outcome"].isin([
        "HIGHER_PROB_CORRECT",
        "LOWER_PROB_CORRECT",
        "NEITHER_CORRECT",
    ])
]

high_correct = int(
    (eligible["outcome"] == "HIGHER_PROB_CORRECT").sum()
)

print("\n========== SUMMARY ==========")
print("judgeable FP       :", len(branch_1106g))
print("eligible deg1 forks:", len(eligible))
print("higher-prob correct:", high_correct)
print(
    "higher-prob rate   :",
    high_correct / len(eligible) if len(eligible) else float("nan"),
)

print("\n========== PROBE < 0.40 SUBSET ==========")

low_probe = eligible[
    eligible["probe_prob"] < 0.40
]

display(
    low_probe["outcome"]
    .value_counts()
    .rename("n")
    .reset_index()
)

print("low-probe eligible :", len(low_probe))
print(
    "higher correct    :",
    int(
        (low_probe["outcome"] == "HIGHER_PROB_CORRECT").sum()
    ),
)

print("\n========== DETAILS ==========")
display(
    branch_1106g.sort_values(
        ["probe_prob", "dataset", "parent"]
    )
)

Stage 11.06 — Parent Division Representation Audit

Frozen UNet parent representation contains real but moderate
division information:

Val PR-AUC      = 0.237856
Val prevalence  = 0.102740
PR lift         = 2.315x
Val ROC-AUC     = 0.679898

Both metrics exceed 95th-percentile label-shuffle controls.

However, the signal does not solve the actual predicted-fork problem:

Current metric TP parent probe ranks: 18 / 24
Top 15 scored judgeable forks: all FP

At probe < 0.40:
TP retained = 2/2
Judgeable FP rejected = 11/33

But after fork rejection, higher association probability identifies
the true continuation branch in only 7/11 = 63.6% of those cases.

Decision:
PARENT_DIVISION_SIGNAL_PRESENT_BUT_NOT_ACTIONABLE_AS_SIMPLE_GATE

Do not promote:
- parent-probe threshold post-processing
- lower-probability daughter deletion
- simple standalone parent division head

Next:
audit context-aware transformer representation.